### BABILong: evaluation example

Download BABILong repository.


In [2]:
#pip install datasets transformers flash_attention accelerate

In [3]:
#!git clone https://github.com/booydar/babilong source

In [4]:
import os
import sys
sys.path.append('..')
os.environ['CUDA_VISIBLE_DEVICES'] = "0"

In [5]:
import torch
import datasets
from transformers import AutoTokenizer, AutoModelForCausalLM

from tqdm.auto import tqdm
import pandas as pd
import json

from pathlib import Path

#from babilong.babilong.prompts import DEFAULT_PROMPTS, DEFAULT_TEMPLATE, get_formatted_input
#from babilong.babilong.babilong_utils import compare_answers

In [6]:
t11 = "Where is?"
t12 = "sdnsk. wdjoi."
t13 = "fafaf."
t13.split(".")[-1]

''

#### Some functions from babilong repo; copied to remove imports

In [7]:
TASK_LABELS = {'qa1': ['bathroom', 'bedroom', 'garden', 'hallway', 'kitchen', 'office'], 
 'qa2': ['bathroom', 'bedroom', 'garden', 'hallway', 'kitchen', 'office'], 
 'qa3': ['bathroom', 'bedroom', 'garden', 'hallway', 'kitchen', 'office'], 
 'qa4': ['bathroom', 'bedroom', 'garden', 'hallway', 'kitchen', 'office'], 
 'qa5': ['Bill', 'Fred', 'Jeff', 'Mary', 'apple', 'football', 'milk'], 
 'qa6': ['no', 'yes'], 
 'qa7': ['none', 'one', 'three', 'two'], 
 'qa8': ['apple', 'football', 'milk', 'nothing'], 
 'qa9': ['no', 'yes'], 
 'qa10': ['maybe', 'no', 'yes'],
 'qa11': ['bathroom', 'bedroom', 'garden', 'hallway', 'kitchen', 'office'], 
 'qa12': ['bathroom', 'bedroom', 'garden', 'hallway', 'kitchen', 'office'], 
 'qa13': ['bathroom', 'bedroom', 'garden', 'hallway', 'kitchen', 'office'], 
 'qa14': ['bedroom', 'cinema', 'kitchen', 'office', 'park', 'school'], 
 'qa15': ['cat', 'mouse', 'sheep', 'wolf'], 
 'qa16': ['gray', 'green', 'white', 'yellow'], 
 'qa17': ['no', 'yes'], 
 'qa18': ['no', 'yes'], 
 'qa19': ['e,e', 'e,n', 'e,s', 'n,e', 'n,n', 'n,w', 's,e', 's,s', 's,w', 'w,n', 'w,s', 'w,w'], 
 'qa20': ['bedroom', 'bored', 'garden', 'hungry', 'kitchen', 'thirsty', 'tired']
}


def preprocess_output(output):
    output = output.lower()
    # take only the first sentence from output
    output = output.split('.')[0]
    # filter responses when model tries to generate examples
    output = output.split('<context>')[0]
    output = output.split('<example>')[0]
    output = output.split('Question')[0]
    return output


def preprocess_output_cot(output):
    output = output.lower()
    # take only the first sentence from output
    splitted_output = output.split('.')
    if len(splitted_output) == 1:
        output = splitted_output[-1]
    else:
        output = splitted_output[-2]
    # filter responses when model tries to generate examples
    output = output.split('<context>')[0]
    output = output.split('<example>')[0]
    output = output.split('Question')[0]
    return output


def compare_answers(target, output, question, task_labels, cot_answer=False):
    if cot_answer:
        output = preprocess_output_cot(output)
    else:
        output = preprocess_output(output)
    target = str(target).lower()
    task_labels = {str(label).lower() for label in task_labels}

    # extract labels that were mentioned in the question
    labels_in_question = {label for label in task_labels if label in str(question).lower()}
    # extract labels that were mentioned in the model output
    labels_in_output = {label for label in task_labels if label in output}
    # filter labels in the output to exclude mentioned in the question
    # mentions in questions are never targets
    labels_in_output = labels_in_output - labels_in_question

    # check if the target is the only prediction
    if ',' in target and len(target) > 3: 
        # if target contains multiple subtargets in qa8
        subtargets = target.split(',')
        num_subtargets = len(subtargets)
        if all([t in labels_in_output for t in subtargets]) and len(labels_in_output) == num_subtargets:
            return True
    else:
        if target in labels_in_output and len(labels_in_output) == 1:
            return True

    return False

In [8]:
SYSTEM_TEMPLATE = '{instruction}\n\n{examples}\n\n{post_prompt}'
USER_TEMPLATE = '<context>\n{context}\n</context>\n\nQuestion: {question}'
DEFAULT_TEMPLATE = f'{SYSTEM_TEMPLATE}\n\n{USER_TEMPLATE}'

CUSTOM_SYSTEM_PROMPTS = {
    # https://github.com/dvlab-research/LongLoRA/blob/2345c6d030f61ac3a031906386a103a5b05e0e6f/inference.py#L18
    'LONGLORA_LLAMA2':
        'You are a helpful, respectful and honest assistant. Always answer as helpfully as possible, while being safe. '
        'Your answers should not include any harmful, unethical, racist, sexist, toxic, dangerous, or illegal content. '
        'Please ensure that your responses are socially unbiased and positive in nature.\n\n'
        'If a question does not make any sense, or is not factually coherent, explain why instead of answering '
        'something not correct. If you don\'t know the answer to a question, please don\'t share false information.'
}


def get_formatted_input(context, question, examples, instruction, post_prompt, template=DEFAULT_TEMPLATE):
    # pre_prompt - general instruction
    # examples - in-context examples
    # post_prompt - any additional instructions after examples
    # context - text to use for qa
    # question - question to answer based on context
    formatted_input = template.format(instruction=instruction, examples=examples, post_prompt=post_prompt,
                                      context=context.strip(), question=question)
    return formatted_input.strip()


DEFAULT_PROMPTS = {
    'qa1': {
        'instruction':
            'I will give you context with the facts about positions of different persons hidden in some random text '
            'and a question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location to answer the question.',
        'examples':
            '<example>\n'
            'Charlie went to the hallway. Judith come back to the kitchen. Charlie travelled to balcony. '
            'Where is Charlie?\n'
            'Answer: The most recent location of Charlie is balcony.\n'
            '</example>\n\n'
            '<example>\n'
            'Alan moved to the garage. Charlie went to the beach. Alan went to the shop. Rouse '
            'travelled to balcony. Where is Alan?\n'
            'Answer: The most recent location of Alan is shop.\n'
            '</example>',
        'post_prompt':
            'Always return your answer in the following format: '
            'The most recent location of ’person’ is ’location’. Do not write anything else after that.'
    },
    'qa2': {
        'instruction':
            'I give you context with the facts about locations and actions of different persons '
            'hidden in some random text and a question.'
            'You need to answer the question based only on the information from the facts.\n'
            'If a person got an item in the first location and travelled to the second location '
            'the item is also in the second location. '
            'If a person dropped an item in the first location and moved to the second location '
            'the item remains in the first location.',
        'examples':
            '<example>\n'
            'Charlie went to the kitchen. Charlie got a bottle. Charlie moved to the balcony. '
            'Where is the bottle?\n'
            'Answer: The bottle is in the balcony.\n'
            '</example>\n'
            '<example>\n'
            'Alan moved to the garage. Alan got a screw driver. Alan moved to the kitchen. Where '
            'is the screw driver?\n'
            'Answer: The screw driver is in the kitchen.\n'
            '</example>',
        'post_prompt':
            'Always return your answer in the following format: The ’item’ is in ’location’. '
            'Do not write anything else after that.'
    },
    'qa3': {
        'instruction':
            'I give you context with the facts about locations and actions of different persons '
            'hidden in some random text and a question. '
            'You need to answer the question based only on the information from the facts.\n'
            'If a person got an item in the first location and travelled to the second location '
            'the item is also in the second location. '
            'If a person dropped an item in the first location and moved to the second location '
            'the item remains in the first location.',
        'examples':
            '<example>\n'
            'John journeyed to the bedroom. Mary grabbed the apple. Mary went back to the bathroom. '
            'Daniel journeyed to the bedroom. Daniel moved to the garden. Mary travelled to the kitchen. '
            'Where was the apple before the kitchen?\n'
            'Answer: Before the kitchen the apple was in the bathroom.\n'
            '</example>\n'
            '<example>\n'
            'John went back to the bedroom. John went back to the garden. John went back to the kitchen. '
            'Sandra took the football. Sandra travelled to the garden. Sandra journeyed to the bedroom. '
            'Where was the football before the bedroom?\n'
            'Answer: Before the bedroom the football was in the garden.\n'
            '</example>',
        'post_prompt':
            'Always return your answer in the following format: '
            'Before the $location_1$ the $item$ was in the $location_2$. Do not write anything else after that.'
    },
    'qa4': {
        'instruction':
            'I will give you context with the facts about different people, their location and actions, hidden in '
            'some random text and a question. '
            'You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'The hallway is south of the kitchen. The bedroom is north of the kitchen. '
            'What is the kitchen south of?\n'
            'Answer: bedroom\n'
            '</example>\n'
            '<example>\n'
            'The garden is west of the bedroom. The bedroom is west of the kitchen. What is west of the bedroom?\n'
            'Answer: garden\n'
            '</example>',
        'post_prompt':
            'Your answer should contain only one word - location. Do not write anything else after that.'
    },
    'qa5': {
        'instruction':
            'I will give you context with the facts about locations and their relations hidden in some random text '
            'and a question. You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'Mary picked up the apple there. Mary gave the apple to Fred. Mary moved to the bedroom. '
            'Bill took the milk there. Who did Mary give the apple to?\n'
            'Answer: Fred\n'
            '</example>\n'
            '<example>\n'
            'Jeff took the football there. Jeff passed the football to Fred. Jeff got the milk there. '
            'Bill travelled to the bedroom. Who gave the football?\n'
            'Answer: Jeff\n'
            '</example>\n'
            '<example>\n'
            'Fred picked up the apple there. Fred handed the apple to Bill. Bill journeyed to the bedroom. '
            'Jeff went back to the garden. What did Fred give to Bill?\n'
            'Answer: apple\n'
            '</example>',
        'post_prompt':
            'Your answer should contain only one word. Do not write anything else after that. '
            'Do not explain your answer.'
    },
    'qa6': {
        'instruction':
            'I will give you context with the facts about people and their locations hidden in some random text and a '
            'question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location the person was in to answer the question.',
        'examples':
            '<example>\n'
            'John travelled to the hallway. John travelled to the garden. Is John in the garden?\n'
            'Answer: yes\n'
            '</example>\n'
            '<example>\n'
            'Mary went to the office. Daniel journeyed to the hallway. Mary went to the bedroom. '
            'Sandra went to the garden. Is Mary in the office?\n'
            'Answer: no\n'
            '</example>\n',
        'post_prompt':
            'Your answer should contain only one word - $yes$ or $no$. Do not write anything else after that. '
            'Do not explain your answer.'
    },
    'qa7': {
        'instruction':
            'I will give you context with the facts about people and objects they carry, hidden in some random text '
            'and a question. You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'Daniel went to the bedroom. Daniel got the apple there. How many objects is Daniel carrying?\n'
            'Answer: one\n'
            '</example>\n'
            '<example>\n'
            'Mary grabbed the apple there. Mary gave the apple to John. How many objects is Mary carrying?\n'
            'Answer: none\n'
            '</example>\n'
            '<example>\n'
            'Sandra travelled to the hallway. Sandra picked up the milk there. Sandra took the apple there. '
            'Mary travelled to the garden. How many objects is Sandra carrying?\n'
            'Answer: two\n'
            '</example>\n',
        'post_prompt':
            'Your answer should contain only one word - $none$ or $number_of_objects$. '
            'Do not write anything else after that. Do not explain your answer.',
    },
    'qa8': {
        'instruction':
            'I will give you context with the facts about people and objects they carry, hidden in some random text '
            'and a question. You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'Sandra travelled to the garden. Mary grabbed the milk there. What is Mary carrying?\n'
            'Answer: milk\n'
            '</example>\n'
            '<example>\n'
            'Mary travelled to the kitchen. Sandra travelled to the office. John travelled to the office. '
            'Sandra discarded the milk there. What is Sandra carrying?\n'
            'Answer: nothing\n'
            '</example>\n'
            '<example>\n'
            'Daniel grabbed the apple there. Mary went to the office. Daniel moved to the garden. '
            'Daniel grabbed the milk there. Mary went to the kitchen. What is Daniel carrying?\n'
            "Answer: apple,milk\n"
            "</example>\n",
        'post_prompt':
            'Your answer should contain only one or two words: $nothing$ or $object$ or $object_1$, $object_2$. '
            'Do not write anything else. Do not explain your answer.'
    },
    'qa9': {
        'instruction':
            'I will give you context with the facts about people and their locations hidden in some random text and '
            'a question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location the person was in to answer the question.',
        'examples':
            '<example>\n'
            'John is not in the bathroom. Sandra is not in the bedroom. Is John in the bathroom?\n'
            'Answer: no\n'
            '</example>\n'
            '<example>\n'
            'Mary journeyed to the kitchen. John is in the bedroom. Sandra is not in the garden. '
            'Is Mary in the kitchen?\n'
            'Answer: yes\n'
            '</example>\n',
        'post_prompt':
            'Your answer should contain only one word - $yes$ or $no$. Do not write anything else. '
            'Do not explain your answer.'
    },
    'qa10': {
        'instruction':
            'I will give you context with the facts about people and their locations hidden in some random text and a '
            'question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location the person was in to answer the question.',
        'examples':
            '<example>\n'
            'Bill is in the kitchen. Julie is either in the school or the cinema. Is Bill in the bedroom?\n'
            'Answer: no\n'
            '</example>\n'
            '<example>\n'
            'Fred is in the bedroom. Mary is either in the school or the cinema. Is Mary in the school?\n'
            'Answer: maybe\n'
            '</example>\n'
            '<example>\n'
            'Fred is either in the kitchen or the park. Bill moved to the cinema. Is Bill in the cinema?\n'
            'Answer: yes\n'
            '</example>\n'
            '<context>\n',
        'post_prompt':
            'Your answer should contain only one word - $yes$ or $no$ or $maybe$. Do not write anything else. '
            'Do not explain your answer.'
    },
    'qa11': {
        'instruction':
            'I will give you context with the facts about people and their locations hidden in some random text and a '
            'question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location the person was in to answer the question.',
        'examples':
            '<example>\n'
            'Daniel journeyed to the hallway. After that he journeyed to the garden. Where is Daniel?\n'
            'Answer: garden\n'
            '</example>\n'
            '<example>\n'
            'Mary moved to the office. Afterwards she journeyed to the kitchen. Daniel went to the hallway. '
            'Then he journeyed to the garden. Where is Mary?\n'
            'Answer: kitchen\n'
            '</example>\n'
            '<example>\n'
            'Sandra moved to the kitchen. After that she went back to the hallway. Sandra moved to the bedroom. '
            'Then she went to the hallway. Mary moved to the bedroom. Afterwards she travelled to the bathroom. '
            'Where is Sandra?\n'
            'Answer: hallway\n'
            '</example>\n'
            '<context>\n',
        'post_prompt':
            'Your answer should contain only one word - location. Do not write anything else after that. '
            'Do not explain your answer.'
    },
    'qa12': {
        'instruction':
            'I will give you context with the facts about people and their locations hidden in some random text and a '
            'question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location the person was in to answer the question.',
        'examples':
            '<example>\n'
            'Mary and Daniel travelled to the bathroom. John and Daniel travelled to the office. Where is Daniel?\n'
            'Answer: office\n'
            '</example>\n'
            '<example>\n'
            'Sandra and Mary went back to the office. Daniel and Sandra went to the bedroom. Sandra and Mary travelled to the hallway. '
            'John and Mary went to the kitchen. Where is Mary?\n'
            'Answer: kitchen\n'
            '</example>\n'
            '<example>\n'
            'Daniel and Sandra went back to the hallway. Daniel and John moved to the office. Daniel and John moved to the garden. '
            'Daniel and Mary went back to the bathroom. Daniel and John went back to the kitchen. Daniel and Sandra went to the bathroom. '
            'Where is John?\n'
            'Answer: kitchen\n'
            '</example>\n'
            '<context>\n',
        'post_prompt':
            'Your answer should contain only one word - location. Do not write anything else after that. '
            'Do not explain your answer.'
    },
    'qa13': {
        'instruction':
            'I will give you context with the facts about people and their locations hidden in some random text and a '
            'question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location the person was in to answer the question.',
        'examples':
            '<example>\n'
            'Mary and Daniel travelled to the bathroom. Then they journeyed to the hallway. Where is Daniel?\n'
            'Answer: hallway\n'
            '</example>\n'
            '<example>\n'
            'Daniel and Sandra travelled to the kitchen. After that they journeyed to the hallway. Mary and Daniel travelled to the bedroom. '
            'After that they travelled to the hallway. Where is Sandra?\n'
            'Answer: hallway\n'
            '</example>\n'
            '<example>\n'
            'John and Mary moved to the bathroom. Then they travelled to the office. John and Mary went to the kitchen. '
            'Afterwards they went to the bedroom. John and Sandra moved to the bathroom. Following that they went back to the kitchen. '
            'Where is Mary?\n'
            'Answer: bedroom\n'
            '</example>\n'
            '<context>\n',
        'post_prompt':
            'Your answer should contain only one word - location. Do not write anything else after that. '
            'Do not explain your answer.'
    },
    'qa14': {
        'instruction':
            'I will give you context with the facts about people and their locations hidden in some random text and a '
            'question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location the person was in to answer the question.',
        'examples':
            '<example>\n'
            'Bill went back to the cinema yesterday. Julie went to the school this morning. Fred went to the park yesterday. '
            'Yesterday Julie went to the office. Where was Julie before the school?\n'
            'Answer: office\n'
            '</example>\n'
            '<example>\n'
            'This morning Fred went to the kitchen. Fred journeyed to the bedroom yesterday. Mary travelled to the bedroom this morning. '
            'Yesterday Mary went to the cinema. Where was Mary before the bedroom?\n'
            'Answer: cinema\n'
            '</example>\n'
            '<example>\n'
            'Yesterday Julie went back to the park. Julie went to the bedroom this morning. Bill journeyed to the cinema yesterday. '
            'This morning Bill went back to the park. This evening Julie went to the school. This afternoon Julie went back to the park. '
            'Where was Julie before the bedroom?\n'
            'Answer: park\n'
            '</example>\n'
            '<context>\n',
        'post_prompt':
            'Your answer should contain only one word - location. Do not write anything else after that. '
            'Do not explain your answer.'
    },
    'qa15': {
        'instruction':
            'I will give you context with the facts about animals, their names and relations. The facts and a question '
            'are hidden in some random text. You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'Mice are afraid of wolves. Gertrude is a mouse. Cats are afraid of sheep. '
            'Winona is a mouse. Sheep are afraid of wolves. Emily is a mouse. Jessica is a wolf. '
            'What is gertrude afraid of?\n'
            'Answer: wolf\n'
            '</example>\n'
            '<example>\n'
            'Mice are afraid of wolves. Gertrude is a mouse. Cats are afraid of sheep. '
            'Winona is a mouse. Sheep are afraid of wolves. Emily is a mouse. Jessica is a wolf. '
            'What is jessica afraid of?\n'
            'Answer: cat\n'
            '</example>\n'
            '<example>\n'
            'Mice are afraid of cats. Wolves are afraid of sheep. Emily is a wolf. '
            'Cats are afraid of sheep. Gertrude is a wolf. Sheep are afraid of cats. Winona is a wolf. '
            'What is emily afraid of?\n'
            'Answer: sheep\n'
            '</example>\n'
            '<context>\n',
        'post_prompt':
            'Your answer should contain only one word - an animal species. Do not write anything else after that. '
            'Do not explain your answer.'
    },
    'qa16': {
        'instruction':
            'I will give you context with the facts about animals, their names and colors. The facts and a question '
            'are hidden in some random text. You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'Lily is a frog. Bernhard is a frog. Bernhard is green. Brian is a lion. Brian is white. '
            'Julius is a swan. Julius is green. Lily is green. Greg is a swan. What color is Greg?\n'
            'Answer: green\n'
            '</example>\n'
            '<example>\n'
            'Julius is a lion. Lily is a rhino. Bernhard is a swan. Lily is white. Bernhard is green. '
            'Greg is a rhino. Greg is gray. Julius is white. Brian is a lion. What color is Brian?\n'
            'Answer: white\n'
            '</example>\n'
            '<example>\n'
            'Brian is a rhino. Julius is a lion. Bernhard is a lion. Greg is a swan. Brian is gray. '
            'Greg is white. Lily is a rhino. Bernhard is yellow. Lily is gray. What color is Julius?\n'
            'Answer: yellow\n'
            '</example>\n'
            '<context>\n',
        'post_prompt':
            'Your answer should contain only one word - a color. Do not write anything else after that. '
            'Do not explain your answer.'
    },
    'qa17': {
        'instruction':
            'I will give you context with the facts about different figures, their location and colors, hidden in '
            'some random text and a question. '
            'You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'The triangle is above the pink rectangle. The blue square is to the left of the triangle. '
            'Is the pink rectangle to the right of the blue square?\n'
            'Answer: yes\n'
            '</example>\n'
            '<example>\n'
            'The red sphere is to the left of the yellow square. The red sphere is below the pink rectangle. '
            'Is the pink rectangle to the left of the yellow square?\n'
            'Answer: yes\n'
            '</example>'
            '<example>\n'
            'The red sphere is above the pink rectangle. The red sphere is to the right of the red square. '
            'Is the pink rectangle above the red square?\n'
            'Answer: no\n'
            '</example>',
        'post_prompt':
            'Your answer should contain only one word - $yes$ or $no$. Do not write anything else. '
            'Do not explain your answer.'
    },
    'qa18': {
        'instruction':
            'I will give you context with the facts about different objects and their sizes, hidden in '
            'some random text and a question. '
            'You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'The box of chocolates fits inside the chest. The box is bigger than the chest. The box is bigger than the suitcase. '
            'The suitcase fits inside the box. The container is bigger than the box of chocolates. Does the box fit in the box of chocolates?\n'
            'Answer: no\n'
            '</example>\n'
            '<example>\n'
            'The suitcase is bigger than the container. The container fits inside the box. The chest is bigger than the chocolate.'
            'The suitcase fits inside the box. The chest fits inside the box. Does the chocolate fit in the box?\n'
            'Answer: yes\n'
            '</example>'
            '<example>\n'
            'The chocolate fits inside the box of chocolates. The suitcase fits inside the box. The chocolate fits inside the box. '
            'The box is bigger than the box of chocolates. The suitcase is bigger than the box of chocolates. Is the chocolate bigger than the box?\n'
            'Answer: no\n'
            '</example>',
        'post_prompt':
            'Your answer should contain only one word - $yes$ or $no$. Do not write anything else. '
            'Do not explain your answer.'
    },
    'qa19': {
        'instruction':
            'I will give you context with the facts about different places and their locations, hidden in '
            'some random text and a question. '
            'You need to answer the question based only on the information from the facts.',
        'examples':
            '<example>\n'
            'The office is east of the hallway. The kitchen is north of the office. The garden is west of the bedroom. '
            'The office is west of the garden. The bathroom is north of the garden. How do you go from the kitchen to the garden?\n'
            'Answer: s,e\n'
            '</example>\n'
            '<example>\n'
            'The bedroom is west of the hallway. The office is east of the garden. The garden is north of the kitchen. '
            'The kitchen is north of the bathroom. The hallway is west of the garden. How do you go from the kitchen to the hallway?\n'
            'Answer: n,w\n'
            '</example>\n'
            '<example>\n'
            'The bedroom is south of the hallway. The bathroom is east of the office. The kitchen is west of the garden. '
            'The garden is south of the office. The office is south of the bedroom. How do you go from the garden to the bedroom?\n'
            'Answer: n,n\n'
            '</example>\n',
        'post_prompt':
            'Your answer should contain only two letters, separated by a comma - ordinal directions. You can choose the letters from '
             '$n$, $s$, $e$ and $w$. Do not write anything else after that.'
    },
    'qa20': {
        'instruction':
            'I will give you context with the facts about people, their locations and condition hidden in some random text and a '
            'question. You need to answer the question based only on the information from the facts. '
            'If a person was in different locations, use the latest location the person was in to answer the question.',
        'examples':
            '<example>\n'
            'Sumit is tired. Where will sumit go?\n'
            'Answer: bedroom\n'
            '</example>\n'
            '<example>\n'
            'Yann is hungry. Yann journeyed to the kitchen. Why did yann go to the kitchen?\n'
            'Answer: hungry\n'
            '</example>\n'
            '<example>\n'
            'Antoine is thirsty. Yann is tired. Yann went back to the bedroom. Yann picked up the pajamas there.'
            'Jason is thirsty. Antoine went back to the kitchen. Why did antoine go to the kitchen?\n'
            'Answer: thirsty\n'
            '</example>\n'
            '<context>\n',
        'post_prompt':
            'Your answer should contain only one word - a person condition or a place. Do not write anything else after that. '
            'Do not explain your answer.'
    }
    

}

In [9]:
#import seaborn as sns
import matplotlib
import matplotlib.pylab as plt
from matplotlib.colors import LinearSegmentedColormap
import os

import pandas as pd
import numpy as np

#from source.babilong.metrics import compare_answers, TASK_LABELS

## New results for 3 datasets with bigger models

In [9]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'
prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'

mod_name = "Qwen/Qwen2.5-0.5B-Instruct"

dataset_pathes = [
    "databricks-dolly-15k_ift_1024_seed_42",
    "alpaca-cleaned_ift_1024_seed_42",
    "unnatural-instructions-full_ift_1024_seed_42"
]

all_dfs = []
for dat_idx, ift_dataset_path in enumerate(dataset_pathes):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    dname = ift_dataset_path.split("-")[0]
    if "instruct" in mod_name.lower():
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-500',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-800',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-1000',]
        ]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-500',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-800',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-1000',]
        ]
    names = all_names[dat_idx]
    for model_name in names:
        overall_results[model_name] = {}
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k']#, '2k', '4k', '8k', '16k', '32k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
        
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                overall_results[model_name][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {dname + " " + key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

,databricks Qwen2.5-0.5B-Instruct,databricks checkpoint-100,databricks checkpoint-500,alpaca checkpoint-800,unnatural checkpoint-100,unnatural checkpoint-1000
qa1_0k,63.90,71.30,75.70,64.70,67.30,68.30
qa1_1k,64.20,58.60,55.70,59.60,62.00,61.40
qa2_0k,36.54,38.64,36.54,38.44,32.93,32.63
qa2_1k,31.93,28.63,27.03,29.33,27.63,27.13
qa4_0k,27.93,42.84,26.03,20.02,28.03,36.44
qa4_1k,21.92,37.64,23.92,34.23,22.62,39.14
qa5_0k,56.16,58.66,55.76,54.85,56.96,51.65
qa5_1k,56.77,53.56,48.75,59.98,59.98,56.87


In [10]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_name = "Qwen/Qwen2.5-1.5B-Instruct"

dataset_pathes = [
    "databricks-dolly-15k_ift_1024_seed_42",
    "alpaca-cleaned_ift_1024_seed_42",
    "unnatural-instructions-full_ift_1024_seed_42"
]

all_dfs = []
for dat_idx, ift_dataset_path in enumerate(dataset_pathes):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    dname = ift_dataset_path.split("-")[0]
    if "instruct" in mod_name.lower():
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-400',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-1000',]
        ]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-400',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100',f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-500',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-800',]
        ]
    names = all_names[dat_idx]
    for model_name in names:
        overall_results[model_name] = {}
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k']#, '2k', '4k', '8k', '16k', '32k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
        
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                overall_results[model_name][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {dname + " " + key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

,databricks Qwen2.5-1.5B-Instruct,databricks checkpoint-100,databricks checkpoint-400,alpaca checkpoint-100,unnatural checkpoint-100,unnatural checkpoint-1000
qa1_0k,73.90,71.90,67.10,68.70,72.60,74.00
qa1_1k,67.50,69.00,69.60,70.10,70.10,73.40
qa2_0k,48.75,46.45,41.24,46.75,45.15,44.14
qa2_1k,42.34,40.24,31.03,41.74,38.04,33.93
qa4_0k,44.14,31.33,32.13,30.03,28.63,37.24
qa4_1k,50.85,45.75,32.13,43.84,33.23,49.05
qa5_0k,72.27,64.56,60.76,67.47,55.26,59.36
qa5_1k,70.81,68.71,62.29,68.20,60.48,61.89


In [164]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_name = "Qwen/Qwen2.5-3B-Instruct"

dataset_pathes = [
    "databricks-dolly-15k_ift_1024_seed_42",
    "alpaca-cleaned_ift_1024_seed_42",
    "unnatural-instructions-full_ift_1024_seed_42"
]

all_dfs = []
for dat_idx, ift_dataset_path in enumerate(dataset_pathes):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    dname = ift_dataset_path.split("-")[0]
    if "instruct" in mod_name.lower():
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-300',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-1000',]
        ]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-200',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100',f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-500',],\
            [f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-100', f'./qwen_exps/es_ablation/{mod_name}/{ift_dataset_path}/checkpoint-800',]
        ]
    names = all_names[dat_idx]
    for model_name in names:
        overall_results[model_name] = {}
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k']#, '2k', '4k', '8k', '16k', '32k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
        
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                overall_results[model_name][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {dname + " " + key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

No such file: ./../../babilong/babilong_evals/./qwen_exps/es_ablation/Qwen/Qwen2.5-3B-Instruct/alpaca-cleaned_ift_1024_seed_42/checkpoint-100/qa5_0k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../babilong/babilong_evals/./qwen_exps/es_ablation/Qwen/Qwen2.5-3B-Instruct/alpaca-cleaned_ift_1024_seed_42/checkpoint-100/qa5_1k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv


,databricks Qwen2.5-3B-Instruct,databricks checkpoint-100,databricks checkpoint-300,alpaca checkpoint-100,unnatural checkpoint-100,unnatural checkpoint-1000
qa1_0k,91.60,84.80,84.50,88.50,81.50,82.10
qa1_1k,78.20,81.20,81.20,76.60,77.90,75.80
qa2_0k,49.45,48.55,50.05,49.75,47.55,42.64
qa2_1k,41.54,44.44,43.04,45.75,41.94,36.64
qa4_0k,45.35,49.95,47.95,46.75,46.35,48.65
qa4_1k,61.06,61.16,56.76,60.50,58.26,50.55
qa5_0k,77.48,80.28,82.28,NaN,78.28,78.28
qa5_1k,78.94,78.03,75.63,NaN,70.71,70.81


### Compare base Qwen2.5 models with RMT-tuned on Flan

In [52]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    #"Qwen/Qwen2.5-0.5B",
    #"Qwen/Qwen2.5-1.5B",
    #"Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin",
    "Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_8172-128-2x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_1-2/run_2/model_best/pytorch_model.bin",
    "Qwen/Qwen2.5-0.5B-Instruct",
    "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen/Qwen2.5-3B-Instruct",
]
disp_names = [
    "RMT-4K-Qwen2.5-0.5B-Instruct",
    "RMT-4K-2Segm-Qwen2.5-0.5B-Instruct",
    "Qwen2.5-0.5B-Instruct",
    "Qwen2.5-1.5B-Instruct",
    "Qwen2.5-3B-Instruct",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if "instruct" in mod_name.lower():
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        #tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

1000
998
999
998
1000
999
999
999
1000
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/qa1_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/qa2_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/qa4_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/qa5_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
1000
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-1.5B-Instruct/qa1_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-1.5B-Instruct/qa2_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../../babilong/babil

,RMT-4K-Qwen2.5-0.5B-Instruct,RMT-4K-2Segm-Qwen2.5-0.5B-Instruct,Qwen2.5-0.5B-Instruct,Qwen2.5-1.5B-Instruct,Qwen2.5-3B-Instruct
qa1_0k,30.20,63.60,63.90,73.90,91.60
qa1_1k,0.80,10.70,64.20,67.50,78.20
qa1_2k,0.20,1.20,60.80,65.60,76.20
qa1_4k,0.10,0.50,55.40,62.90,77.60
qa1_8k,0.00,0.60,50.10,63.60,74.30
qa1_16k,0.00,0.10,41.10,64.40,76.60
qa1_32k,0.00,0.00,38.10,59.00,76.20
qa1_64k,0.00,0.10,NaN,NaN,NaN
qa2_0k,32.43,35.34,36.54,48.75,49.45
qa2_1k,28.63,29.33,31.93,42.34,41.54


### Compare base Qwen2.5 models with RMT-tuned on LongDolly

In [51]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder_rmt = "../babilong_evals"
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    #"Qwen/Qwen2.5-0.5B",
    #"Qwen/Qwen2.5-1.5B",
    #"Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/longdolly/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_1004-128-1x1024_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin",
    "Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/longdolly/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_2008-128-2x1024_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_1-2/run_2/model_best/pytorch_model.bin",
    "Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/longdolly/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4016-128-4x1024_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_2-4/run_2/model_best/pytorch_model.bin",
    "Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/longdolly/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_8032-128-8x1024_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_4-8/run_2/model_best/pytorch_model.bin",
    "Qwen/Qwen2.5-0.5B-Instruct",
    #"Qwen/Qwen2.5-1.5B-Instruct",
    #"Qwen/Qwen2.5-3B-Instruct",
]
disp_names = [
    "RMT-1K-LD-Qwen2.5-0.5B-Instruct",
    "RMT-1K-LD-2Segm-Qwen2.5-0.5B-Instruct",
    "RMT-1K-LD-4Segm-Qwen2.5-0.5B-Instruct",
    "RMT-1K-LD-8Segm-Qwen2.5-0.5B-Instruct",
    "Qwen2.5-0.5B-Instruct",
    #"Qwen2.5-1.5B-Instruct",
    #"Qwen2.5-3B-Instruct",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if "instruct" in mod_name.lower():
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        #tasks = ['qa1', 'qa2', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

999
No such file: ./../babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/longdolly/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_1004-128-1x1024_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin/qa2_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/longdolly/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_2008-128-2x1024_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_1-2/run_2/model_best/pytorch_model.bin/qa2_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/longdolly/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4016-128-4x1024_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_2-4/run_2/model_best/pytorch_model.bin/qa2_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../babilong_evals/./Qwen/Qwen2.5

,RMT-1K-LD-Qwen2.5-0.5B-Instruct,RMT-1K-LD-2Segm-Qwen2.5-0.5B-Instruct,RMT-1K-LD-4Segm-Qwen2.5-0.5B-Instruct,RMT-1K-LD-8Segm-Qwen2.5-0.5B-Instruct,Qwen2.5-0.5B-Instruct
qa2_0k,0.0,0.0,0.0,0.0,36.54
qa2_1k,0.1,0.1,0.0,0.0,31.93
qa2_2k,0.0,3.5,2.2,3.7,31.23
qa2_4k,0.0,1.9,0.3,1.2,26.23
qa2_8k,0.0,0.0,0.2,0.4,20.32
qa2_16k,0.0,0.0,0.0,0.0,15.52
qa2_32k,0.0,0.0,0.0,0.0,13.51


### Compare base Qwen2.5 models with RMT-tuned on LAMix

In [56]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder_rmt = "../babilong_evals"
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    #"Qwen/Qwen2.5-0.5B",
    #"Qwen/Qwen2.5-1.5B",
    #"Qwen/Qwen2.5-3B",
    "Qwen/Qwen2.5-0.5B-Instruct/runs/lm_no_st/LAMix/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_6124-128-1x6144_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin",
    "Qwen/Qwen2.5-0.5B-Instruct",
    #"Qwen/Qwen2.5-1.5B-Instruct",
    #"Qwen/Qwen2.5-3B-Instruct",
]
disp_names = [
    "RMT-6K-LAMix-Qwen2.5-0.5B-Instruct",
    "Qwen2.5-0.5B-Instruct",
    #"Qwen2.5-1.5B-Instruct",
    #"Qwen2.5-3B-Instruct",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if "instruct" in mod_name.lower():
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

1000
999
999
999
1000
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/qa1_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/qa2_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/qa4_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
999
No such file: ./../../babilong/babilong_evals/./Qwen/Qwen2.5-0.5B-Instruct/qa5_64k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv


,RMT-6K-LAMix-Qwen2.5-0.5B-Instruct,Qwen2.5-0.5B-Instruct
qa1_0k,53.90,63.90
qa1_1k,36.90,64.20
qa1_2k,19.10,60.80
qa1_4k,9.20,55.40
qa1_8k,0.00,50.10
qa1_16k,0.00,41.10
qa1_32k,0.00,38.10
qa1_64k,0.00,NaN
qa2_0k,16.92,36.54
qa2_1k,9.61,31.93


### Compare base Qwen2.5 models with IFT ones on long datasets

In [14]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
new_results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    #"Qwen/Qwen2.5-0.5B",
    #"Qwen/Qwen2.5-1.5B",
    #"Qwen/Qwen2.5-3B",
    "qwen_exps/es_ablation/Qwen/Qwen2.5-0.5B-Instruct/dolly_15k_1024_ift_16384_seed_42/checkpoint-950",
    "qwen_exps/es_ablation/Qwen/Qwen2.5-0.5B-Instruct/LongAlign-10k_ift_16384_seed_42/checkpoint-700",
    "mergekit/test_LA_merge",
    "qwen_exps/es_ablation/Qwen/Qwen2.5-0.5B-Instruct/LongAlign_sg_16k_mix_ift_16384_seed_42/checkpoint-4400",
    "qwen_exps/es_ablation/Qwen/Qwen2.5-0.5B-Instruct/LongAlign_sg_16k_mix_ift_16384_seed_42/checkpoint-6800",
    "qwen_exps/es_ablation/Qwen/Qwen2.5-0.5B-Instruct/LongAlign_sg_16k_mix_ift_16384_seed_42/checkpoint-12000",
    "Qwen/Qwen2.5-0.5B-Instruct",
    #"Qwen/Qwen2.5-1.5B-Instruct",
    #"Qwen/Qwen2.5-3B-Instruct",
]
disp_names = [
    "LD-Qwen2.5-0.5B-Instruct",
    "LA-Qwen2.5-0.5B-Instruct",
    "LA-Merge-Qwen2.5-0.5B-Instruct",
    "LAMix-Qwen2.5-0.5B-Instruct-4400",
    "LAMix-Qwen2.5-0.5B-Instruct-6800",
    "LAMix-Qwen2.5-0.5B-Instruct-12000",
    "Qwen2.5-0.5B-Instruct",
    #"Qwen2.5-1.5B-Instruct",
    #"Qwen2.5-3B-Instruct",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if "instruct" in mod_name.lower() or "mergekit" in mod_name.lower():
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999


,LD-Qwen2.5-0.5B-Instruct,LA-Qwen2.5-0.5B-Instruct,LA-Merge-Qwen2.5-0.5B-Instruct,LAMix-Qwen2.5-0.5B-Instruct-4400,LAMix-Qwen2.5-0.5B-Instruct-6800,LAMix-Qwen2.5-0.5B-Instruct-12000,Qwen2.5-0.5B-Instruct
qa1_0k,44.70,62.60,63.30,65.50,61.50,61.70,63.90
qa1_1k,48.60,59.90,61.20,61.00,60.40,62.70,64.20
qa1_2k,43.40,53.20,59.40,47.80,45.50,48.80,60.80
qa1_4k,30.90,47.80,52.40,45.90,43.60,44.90,55.40
qa1_8k,25.20,32.20,39.40,40.90,38.60,38.10,50.10
qa1_16k,27.70,27.80,24.40,34.90,32.80,32.00,41.10
qa1_32k,22.80,23.90,23.00,27.10,25.60,25.80,38.10
qa2_0k,23.12,35.84,36.84,33.53,34.13,35.14,36.54
qa2_1k,26.73,20.42,31.73,26.33,25.13,24.92,31.93
qa2_2k,27.43,21.02,28.83,18.12,16.52,16.22,31.23


### Compare results for LLama1b models after IFT

In [34]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    "meta-llama/Llama-3.2-1B",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/alpaca-cleaned_ift_1024_seed_42/checkpoint-7000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/unnatural-instructions-full_ift_1024_seed_42/checkpoint-8000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/hf_dataset_sharegpt_ift_1024_seed_42/checkpoint-30000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/ultrachat_200k_flattened_ift_1024_seed_42/checkpoint-30000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/flan_collection_ift_1024_seed_42/checkpoint-30000",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Alpaca-Llama-3.2-1B",
    "Unnat-Llama-3.2-1B",

    "SGpt-Llama-3.2-1B",
    "UC-Llama-3.2-1B",
    "FLAN-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    #if ("Llama-3.2-1B-Instruct".lower() in mod_name.lower()):
    #    prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    #    all_names = [
    #        [f'./{mod_name}']]
    #else:
    prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999
1000
999
999
999


,Llama-3.2-1B,Dolly-Llama-3.2-1B,Alpaca-Llama-3.2-1B,Unnat-Llama-3.2-1B,SGpt-Llama-3.2-1B,UC-Llama-3.2-1B,FLAN-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,44.40,66.50,43.40,61.30,17.70,30.20,0.00
qa1_1k,2.90,5.50,62.10,10.70,60.20,14.60,21.10,2.20
qa1_2k,20.90,18.70,53.30,9.10,54.70,14.10,16.70,1.20
qa1_4k,41.20,50.80,48.30,4.70,33.40,28.40,15.30,0.90
qa1_8k,48.90,40.80,37.50,5.30,28.90,35.70,14.50,2.40
qa1_16k,38.90,36.50,26.00,5.30,23.20,33.50,12.40,1.70
qa1_32k,29.80,29.80,23.00,3.20,17.20,36.00,5.00,2.00
qa2_0k,0.00,0.00,35.64,23.02,37.54,15.32,24.52,0.00
qa2_1k,6.31,1.80,25.83,7.31,27.53,30.13,1.60,1.60
qa2_2k,20.82,10.61,24.82,12.11,25.33,24.82,5.71,4.70


In [37]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    #"meta-llama/Llama-3.2-1B",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/alpaca-cleaned_ift_1024_seed_42/checkpoint-7000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/unnatural-instructions-full_ift_1024_seed_42/checkpoint-8000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/hf_dataset_sharegpt_ift_1024_seed_42/checkpoint-30000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/ultrachat_200k_flattened_ift_1024_seed_42/checkpoint-30000",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/flan_collection_ift_1024_seed_42/checkpoint-30000",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_base/flan/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_1004-128-1x1024_mem10_bs16_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_base/flan_mix/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_4076-128-1x4096_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_base/flan_mix/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_8152-128-2x4096_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_1-2/run_2/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_base_long/flan_mix/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_4076-128-1x4096_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_base_long/flan_mix/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_8152-128-2x4096_mem10_bs32_regular_bptt-2_lora_freeze_from_cpt_1-2/run_2/model_best/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]

disp_names = [
    #"Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Alpaca-Llama-3.2-1B",
    "Unnat-Llama-3.2-1B",

    "SGpt-Llama-3.2-1B",
    "UC-Llama-3.2-1B",
    "FLAN-Llama-3.2-1B",
    "1xRMT-FLAN-1k-Llama-3.2-1B",
    "RMT-FLANMix-Llama-3.2-1B",
    "2xRMT-FLANMix-Llama-3.2-1B",
    "LT-RMT-FLANMix-Llama-3.2-1B",
    "LT-2xRMT-FLANMix-Llama-3.2-1B",
    
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    #if ("Llama-3.2-1B-Instruct".lower() in mod_name.lower()):
    prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    #else:
    #    prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
    #    all_names = [
    #        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

No such file: ./../../rmt_ift/babilong_evals/./meta-llama/Llama-3.2-1B/runs/rmt_ift_from_base_long/flan_mix/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_4076-128-1x4096_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin/qa1_32k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./meta-llama/Llama-3.2-1B/runs/rmt_ift_from_base_long/flan_mix/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_4076-128-1x4096_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin/qa2_32k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./meta-llama/Llama-3.2-1B/runs/rmt_ift_from_base_long/flan_mix/meta-llama/Llama-3.2-1B/linear_adamw_wd1e-03_4076-128-1x4096_mem10_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/pytorch_model.bin/qa4_32k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such fi

,Dolly-Llama-3.2-1B,Alpaca-Llama-3.2-1B,Unnat-Llama-3.2-1B,SGpt-Llama-3.2-1B,UC-Llama-3.2-1B,FLAN-Llama-3.2-1B,1xRMT-FLAN-1k-Llama-3.2-1B,RMT-FLANMix-Llama-3.2-1B,2xRMT-FLANMix-Llama-3.2-1B,LT-RMT-FLANMix-Llama-3.2-1B,LT-2xRMT-FLANMix-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,66.90,67.10,64.30,63.40,63.70,54.50,48.70,56.30,11.30,16.00,0.00,70.70
qa1_1k,65.30,27.00,68.80,65.90,58.40,48.10,47.50,16.30,2.90,2.10,0.00,66.50
qa1_2k,59.20,14.50,67.80,61.90,52.10,45.50,0.00,20.70,4.20,1.70,0.00,59.80
qa1_4k,53.00,0.30,60.30,44.30,32.70,42.20,0.00,3.60,6.30,7.90,0.00,57.50
qa1_8k,33.50,0.00,41.30,29.20,34.70,38.30,0.00,0.00,0.40,0.00,1.90,55.20
qa1_16k,31.90,0.10,36.80,25.90,32.30,33.70,0.40,0.10,3.20,0.20,0.80,45.40
qa1_32k,22.20,0.00,20.30,8.90,26.20,25.60,0.00,0.20,0.70,NaN,NaN,28.50
qa2_0k,44.94,43.14,39.84,37.54,43.54,36.54,43.54,40.24,1.80,12.71,0.00,45.35
qa2_1k,34.93,17.22,30.63,31.63,32.03,16.42,24.22,11.81,1.00,0.60,0.00,35.34
qa2_2k,34.83,15.72,30.73,31.13,30.53,20.02,0.00,11.21,4.10,0.10,0.00,26.93


### Eval LM-pretrained RMT and its instruction tuned versions

In [53]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model.safetensors",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/dolly/meta-llama/Llama-3.2-1B/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/hf_dataset_sharegpt_ift_1024_seed_42/checkpoint-30000",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/sharegpt/meta-llama/Llama-3.2-1B/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/ultrachat/meta-llama/Llama-3.2-1B/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_3968/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/sharegpt/meta-llama/Llama-3.2-1B/SEGM_4x1024_3968/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]

disp_names = [
    "Llama-3.2-1B-WO-chat",
    "2s-LM-RMT-1k-Llama-3.2-1B-WO-chat",
    "Dolly-2s-LM-RMT-1k-Llama-3.2-1B",
    "SGPT-1k-Llama-3.2-1B",
    "SGPT-2s-LM-RMT-1k-Llama-3.2-1B",
    "UC-2s-LM-RMT-1k-Llama-3.2-1B",

    "Dolly-4s-2s-LM-RMT-1k-Llama-3.2-1B",
    "SGPT-4s-2s-LM-RMT-1k-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    #if ("Llama-3.2-1B-Instruct".lower() in mod_name.lower()):
    if "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model.safetensors" in mod_name or mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    #else:
    #    prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
    #    all_names = [
    #        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

,Llama-3.2-1B-WO-chat,2s-LM-RMT-1k-Llama-3.2-1B-WO-chat,Dolly-2s-LM-RMT-1k-Llama-3.2-1B,SGPT-1k-Llama-3.2-1B,SGPT-2s-LM-RMT-1k-Llama-3.2-1B,UC-2s-LM-RMT-1k-Llama-3.2-1B,Dolly-4s-2s-LM-RMT-1k-Llama-3.2-1B,SGPT-4s-2s-LM-RMT-1k-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,1.30,0.10,63.40,24.90,6.20,0.00,0.0,70.70
qa1_1k,2.90,8.80,1.80,65.90,13.00,6.80,0.10,0.0,66.50
qa1_2k,20.90,5.30,5.00,61.90,4.50,4.30,7.20,0.6,59.80
qa1_4k,41.20,5.10,5.10,44.30,8.50,5.40,1.80,1.6,57.50
qa1_8k,48.90,2.40,2.30,29.20,1.60,0.70,2.50,0.8,55.20
qa1_16k,38.90,0.20,0.40,25.90,0.30,0.30,1.00,0.2,45.40
qa2_0k,0.00,2.60,0.20,37.54,2.50,1.10,0.00,0.0,45.35
qa2_1k,6.31,6.71,1.40,31.63,4.00,4.00,0.20,0.0,35.34
qa2_2k,20.82,4.40,3.50,31.13,3.70,2.30,6.41,0.5,26.93
qa2_4k,24.92,5.41,4.60,27.13,5.41,3.20,0.90,1.1,20.02


### SGPT model variations

In [115]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model.safetensors",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-4x1024-mem16-pg19-31k_it/model.safetensors",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/hf_dataset_sharegpt_ift_1024_seed_42/checkpoint-30000",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/sharegpt/meta-llama/Llama-3.2-1B/real_RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/sharegpt/meta-llama/Llama-3.2-1B/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_it_mem_mean/sharegpt/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_it_mem_mean/sharegpt/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_3968/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_it_mem_mean/sharegpt/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_7936/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_it_mem_only/sharegpt/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984/RMT-Llama-3.2-1B-2x1024-mem16_from_it/model_best/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_it/sharegpt/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984/RMT-Llama-3.2-1B-2x1024-mem16_from_it/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_it_mem_mean_mem_only/sharegpt/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_it_mem_mean_mem_only/sharegpt/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_3968/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_it_mem_mean_mem_only/sharegpt/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_7936/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/sharegpt/meta-llama/Llama-3.2-1B/SEGM_4x1024_3968/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]

disp_names = [
    "Llama-3.2-1B",
    "2s-LM-RMT-1k-Llama-3.2-1B",
    "4s-LM-RMT-1k-Llama-3.2-1B",
    "SGPT-1k-Llama-3.2-1B",
    "SGPT-2s-LM-RMT-1k-Llama-3.2-1B",
    "SGPT-4s-LM-RMT-1k-Llama-3.2-1B",
    
    "SGPT-2s-RMT-MemMean-1k-Llama-3.2-1B-IT",
    "SGPT-4s-RMT-MemMean-1k-Llama-3.2-1B-IT",
    "SGPT-8s-RMT-MemMean-1k-Llama-3.2-1B-IT",
    "SGPT-2s-RMT-MemOnly-1k-Llama-3.2-1B-IT",

    "SGPT-2s-RMT-1k-Llama-3.2-1B-IT",
    "SGPT-2s-RMT-MemOnlyMean-1k-Llama-3.2-1B-IT",
    "SGPT-4s-RMT-MemOnlyMean-1k-Llama-3.2-1B-IT",
    "SGPT-8s-RMT-MemOnlyMean-1k-Llama-3.2-1B-IT",

    #"SGPT-4s-2s-LM-RMT-1k-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    #if ("Llama-3.2-1B-Instruct".lower() in mod_name.lower()):
    if "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model.safetensors" in mod_name or mod_name == "meta-llama/Llama-3.2-1B" or "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-4x1024-mem16-pg19-31k_it/model.safetensors" in mod_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    #else:
    #    prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
    #    all_names = [
    #        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

,Llama-3.2-1B,2s-LM-RMT-1k-Llama-3.2-1B,4s-LM-RMT-1k-Llama-3.2-1B,SGPT-1k-Llama-3.2-1B,SGPT-2s-LM-RMT-1k-Llama-3.2-1B,SGPT-4s-LM-RMT-1k-Llama-3.2-1B,SGPT-2s-RMT-MemMean-1k-Llama-3.2-1B-IT,SGPT-4s-RMT-MemMean-1k-Llama-3.2-1B-IT,SGPT-8s-RMT-MemMean-1k-Llama-3.2-1B-IT,SGPT-2s-RMT-MemOnly-1k-Llama-3.2-1B-IT,SGPT-2s-RMT-1k-Llama-3.2-1B-IT,SGPT-2s-RMT-MemOnlyMean-1k-Llama-3.2-1B-IT,SGPT-4s-RMT-MemOnlyMean-1k-Llama-3.2-1B-IT,SGPT-8s-RMT-MemOnlyMean-1k-Llama-3.2-1B-IT,Llama-3.2-1B-IT
qa1_0k,0.00,1.30,3.20,63.40,24.90,0.00,0.00,0.00,0.00,33.50,0.00,51.30,55.60,56.30,70.70
qa1_1k,2.90,8.80,6.10,65.90,13.00,0.00,0.30,0.10,0.00,33.50,0.00,60.70,63.50,60.80,66.50
qa1_2k,20.90,5.30,16.20,61.90,4.50,3.20,24.20,17.20,12.60,7.50,2.80,0.10,0.00,0.00,59.80
qa1_4k,41.20,5.10,13.70,44.30,8.50,2.70,11.30,6.80,5.20,0.40,1.20,0.20,0.10,0.00,57.50
qa1_8k,48.90,2.40,8.50,29.20,1.60,4.70,4.80,6.60,0.90,0.70,2.50,0.00,0.00,0.10,55.20
qa1_16k,38.90,0.20,3.70,25.90,0.30,1.30,2.50,1.20,1.90,0.00,0.30,0.00,0.00,0.20,45.40
qa2_0k,0.00,2.60,3.70,37.54,2.50,0.00,5.01,0.20,0.00,7.81,0.00,33.43,35.54,32.23,45.35
qa2_1k,6.31,6.71,2.80,31.63,4.00,0.20,0.40,0.00,0.00,13.41,0.00,28.53,31.83,22.42,35.34
qa2_2k,20.82,4.40,10.81,31.13,3.70,1.10,7.21,9.31,8.01,3.90,1.20,0.10,0.00,0.00,26.93
qa2_4k,24.92,5.41,6.41,27.13,5.41,1.50,5.11,3.10,1.00,0.00,0.50,0.10,0.20,0.00,20.02


### Compare results for LLama1b models after IFT and RMT on Dolly

In [197]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-700/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    "DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "RMT-1k-16mem-Dolly-Llama-3.2-1B",
    "RMT-1k-1mem-Dolly-Llama-3.2-1B",
    "RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    "RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-Dolly-Llama-3.2-1B",
    #"ARMT-1k-2s-16mem-64-Dolly-Llama-3.2-1B",
    #"ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

,Llama-3.2-1B,DollyLoraOld-Llama-3.2-1B,Dolly-Llama-3.2-1B,RMT-1k-16mem-Dolly-Llama-3.2-1B,RMT-1k-1mem-Dolly-Llama-3.2-1B,RMT-1k-1mem-mean-Dolly-Llama-3.2-1B,RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,66.90,66.10,41.70,41.00,8.10,3.10,70.70
qa1_1k,2.90,65.30,65.00,28.80,35.90,42.80,2.30,66.50
qa1_2k,20.90,59.20,65.10,0.00,0.40,1.50,6.90,59.80
qa1_4k,41.20,53.00,57.60,0.00,1.10,0.90,5.10,57.50
qa1_8k,48.90,33.50,44.30,0.10,0.70,0.90,1.10,55.20
qa1_16k,38.90,31.90,37.90,0.10,0.80,0.50,0.20,45.40
qa2_0k,0.00,44.94,44.14,18.72,41.94,12.01,0.50,45.35
qa2_1k,6.31,34.93,33.13,6.21,26.33,22.22,1.70,35.34
qa2_2k,20.82,34.83,30.83,0.00,0.30,0.90,4.80,26.93
qa2_4k,24.92,30.53,27.13,0.00,0.50,0.70,3.10,20.02


### Compare results for LLama1b models after IFT and RMT on SGPT

In [172]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/hf_dataset_sharegpt_ift_1024_seed_42/checkpoint-30000",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-28500",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/full_rmt_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-30000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/full_rmt_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-25500/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    "SGPTLoraOld-Llama-3.2-1B",
    "SGPT-Llama-3.2-1B",
    "RMT-1k-16mem-SGPT-Llama-3.2-1B",
    "RMT-1k-1mem-SGPT-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

,Llama-3.2-1B,SGPTLoraOld-Llama-3.2-1B,SGPT-Llama-3.2-1B,RMT-1k-16mem-SGPT-Llama-3.2-1B,RMT-1k-1mem-SGPT-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,63.40,63.00,52.90,61.00,70.70
qa1_1k,2.90,65.90,61.40,40.10,54.20,66.50
qa1_2k,20.90,61.90,49.30,0.00,0.10,59.80
qa1_4k,41.20,44.30,27.20,0.00,0.20,57.50
qa1_8k,48.90,29.20,26.00,0.10,0.30,55.20
qa1_16k,38.90,25.90,25.40,0.10,0.70,45.40
qa2_0k,0.00,37.54,35.24,33.23,37.04,45.35
qa2_1k,6.31,31.63,29.73,0.80,22.72,35.34
qa2_2k,20.82,31.13,29.53,0.00,0.00,26.93
qa2_4k,24.92,27.13,25.13,0.00,0.50,20.02


### Compare results for LLama1b models after IFT and RMT on Flan

In [226]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/flan_collection_ift_1024_seed_42/checkpoint-30000",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/full_rmt_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-98500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/full_rmt_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-98000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    "FlanLoraOld-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",
    "RMT-1k-16mem-Flan-Llama-3.2-1B",
    "RMT-1k-1mem-Flan-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        tasks = ['qa1', 'qa2', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

,Llama-3.2-1B,FlanLoraOld-Llama-3.2-1B,Flan-Llama-3.2-1B,RMT-1k-16mem-Flan-Llama-3.2-1B,RMT-1k-1mem-Flan-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,54.50,57.00,12.20,6.40,70.70
qa1_1k,2.90,48.10,47.40,25.20,12.90,66.50
qa1_2k,20.90,45.50,45.70,0.00,0.10,59.80
qa1_4k,41.20,42.20,43.30,0.00,1.00,57.50
qa1_8k,48.90,38.30,39.70,0.10,1.00,55.20
qa1_16k,38.90,33.70,34.80,0.20,1.70,45.40
qa2_0k,0.00,36.54,35.94,9.91,22.42,45.35
qa2_1k,6.31,16.42,14.01,8.71,5.31,35.34
qa2_2k,20.82,20.02,18.72,0.00,0.00,26.93
qa2_4k,24.92,19.22,14.61,0.00,0.10,20.02


### Compare results for LLama1b models after IFT and ARMT on Dolly

In [409]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-700/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_128/checkpoint-300/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_256/checkpoint-300/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    "DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-Dolly-Llama-3.2-1B",
    "ARMT-1k-1mem-64-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-16mem-64-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-all-s-16mem-128-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-all-s-16mem-256-Dolly-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=0, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,Llama-3.2-1B,DollyLoraOld-Llama-3.2-1B,Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-Dolly-Llama-3.2-1B,ARMT-1k-1mem-64-Dolly-Llama-3.2-1B,ARMT-1k-2s-16mem-64-Dolly-Llama-3.2-1B,ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B,ARMT-1k-2s-all-s-16mem-128-Dolly-Llama-3.2-1B,ARMT-1k-2s-all-s-16mem-256-Dolly-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,66.90,66.10,64.80,65.40,37.20,62.60,64.60,64.80,70.70
qa1_1k,2.90,65.30,65.00,64.70,65.20,63.80,65.00,64.50,63.80,66.50
qa1_2k,20.90,59.20,65.10,22.10,20.70,30.60,34.60,29.20,26.60,59.80
qa1_4k,41.20,53.00,57.60,13.60,12.70,18.20,23.70,17.00,14.20,57.50
qa1_8k,48.90,33.50,44.30,9.40,7.60,10.70,14.40,9.40,8.50,55.20
qa1_16k,38.90,31.90,37.90,5.70,5.70,3.70,6.30,5.00,3.70,45.40
qa2_0k,0.00,44.94,44.14,44.44,45.85,21.92,41.84,43.44,43.04,45.35
qa2_1k,6.31,34.93,33.13,32.63,33.73,28.93,32.43,33.23,33.43,35.34
qa2_2k,20.82,34.83,30.83,19.92,21.52,23.12,25.63,22.52,22.62,26.93
qa2_4k,24.92,30.53,27.13,10.21,11.21,11.51,14.41,13.41,12.91,20.02


### Compare results for LLama1b models after IFT and ARMT on CL

In [328]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    "ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B,Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B,ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B,ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,66.10,65.30,63.80,61.20,62.10,65.20,68.10,62.80,59.80,70.70
qa1_1k,2.90,65.00,64.80,65.90,63.30,69.60,62.90,66.40,67.00,50.80,66.50
qa1_2k,20.90,65.10,28.00,23.90,22.50,24.70,28.20,4.00,2.80,48.40,59.80
qa1_4k,41.20,57.60,16.50,11.70,9.80,15.30,15.60,1.60,2.10,30.20,57.50
qa1_8k,48.90,44.30,10.00,7.10,5.50,6.70,9.30,1.00,1.20,9.20,55.20
qa1_16k,38.90,37.90,5.30,3.40,2.10,3.20,5.10,0.70,0.80,0.50,45.40
qa2_0k,0.00,44.14,43.54,42.74,42.34,45.15,44.44,40.44,38.54,38.44,45.35
qa2_1k,6.31,33.13,33.33,32.03,31.53,32.43,31.83,32.43,32.23,18.72,35.34
qa2_2k,20.82,30.83,23.12,19.92,19.12,23.62,19.62,3.70,6.91,17.22,26.93
qa2_4k,24.92,27.13,11.41,10.01,9.51,12.51,9.81,0.60,2.30,8.11,20.02


### Compare results for LLama1b models after IFT and ARMT on CL, SGPT

In [327]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    "ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    "ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B,Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B,ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B,ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B,ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,66.10,62.10,61.90,63.30,62.20,59.80,70.70
qa1_1k,2.90,65.00,62.70,63.00,62.40,62.20,50.80,66.50
qa1_2k,20.90,65.10,7.20,6.50,7.10,7.40,48.40,59.80
qa1_4k,41.20,57.60,2.10,2.30,1.40,1.90,30.20,57.50
qa1_8k,48.90,44.30,0.10,2.30,0.70,1.10,9.20,55.20
qa1_16k,38.90,37.90,0.20,0.10,0.40,0.70,0.50,45.40
qa2_0k,0.00,44.14,34.63,34.73,34.93,35.24,38.44,45.35
qa2_1k,6.31,33.13,29.23,28.63,27.83,28.13,18.72,35.34
qa2_2k,20.82,30.83,9.01,6.81,6.91,8.11,17.22,26.93
qa2_4k,24.92,27.13,1.10,1.30,1.20,1.60,8.11,20.02


### Various exps with ARMT and LLama1b models, mostly on Dolly

In [408]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    "ARMT-1k-2s-16mem-64-Dolly-CompLoss-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B,Dolly-Llama-3.2-1B,ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B,ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT,ARMT-1k-2s-16mem-64-Dolly-CompLoss-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,66.10,62.60,8.00,73.60,60.00,70.70
qa1_1k,2.90,65.00,65.00,26.90,58.20,67.10,66.50
qa1_2k,20.90,65.10,34.60,4.90,35.20,34.90,59.80
qa1_4k,41.20,57.60,23.70,2.90,22.30,18.00,57.50
qa1_8k,48.90,44.30,14.40,2.40,12.00,9.40,55.20
qa1_16k,38.90,37.90,6.30,1.30,5.70,2.60,45.40
qa2_0k,0.00,44.14,41.84,39.74,45.25,41.64,45.35
qa2_1k,6.31,33.13,32.43,31.93,31.03,30.33,35.34
qa2_2k,20.82,30.83,25.63,7.31,13.71,23.52,26.93
qa2_4k,24.92,27.13,14.41,3.00,7.01,10.81,20.02


### Eval long mixes on Dolly

In [365]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-250/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_1_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_5_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-8s-16mem-64-Dolly-No-Mix-Llama-3.2-1B",
    "ARMT-1k-8s-16mem-64-Dolly-0.5%-Mix-Llama-3.2-1B",

    "ARMT-1k-8s-16mem-64-Dolly-1%-Mix-Llama-3.2-1B",
    "ARMT-1k-8s-16mem-64-Dolly-5%-Mix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B,Dolly-Llama-3.2-1B,ARMT-1k-8s-16mem-64-Dolly-No-Mix-Llama-3.2-1B,ARMT-1k-8s-16mem-64-Dolly-0.5%-Mix-Llama-3.2-1B,ARMT-1k-8s-16mem-64-Dolly-1%-Mix-Llama-3.2-1B,ARMT-1k-8s-16mem-64-Dolly-5%-Mix-Llama-3.2-1B,ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT,Llama-3.2-1B-IT
qa1_0k,0.00,66.10,62.10,65.50,65.30,66.20,76.30,70.70
qa1_1k,2.90,65.00,69.60,65.70,64.30,66.30,61.40,66.50
qa1_2k,20.90,65.10,24.70,30.50,26.40,20.20,41.90,59.80
qa1_4k,41.20,57.60,15.30,18.30,13.30,9.10,27.30,57.50
qa1_8k,48.90,44.30,6.70,11.10,7.60,5.50,16.20,55.20
qa1_16k,38.90,37.90,3.20,5.10,3.00,2.60,5.90,45.40
qa2_0k,0.00,44.14,45.15,45.15,43.14,41.64,44.34,45.35
qa2_1k,6.31,33.13,32.43,34.23,32.73,32.13,32.43,35.34
qa2_2k,20.82,30.83,23.62,22.82,17.02,13.41,15.12,26.93
qa2_4k,24.92,27.13,12.51,13.41,7.51,6.91,7.71,20.02


### Eval short 128tok segms on Dolly

In [316]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_short/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x128_224_64/checkpoint-1000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_short/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x128_448_64/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_short/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x128_896_64/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_short/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_16x128_1792_64/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x128_896_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-8s-16mem-64-Dolly-NoMix-Llama-3.2-1B",
    "ARMT-1k-2s-128-16mem-64-Dolly-CL-Llama-3.2-1B",
    "ARMT-1k-4s-128-16mem-64-Dolly-CL-Llama-3.2-1B",
    "ARMT-1k-8s-128-16mem-64-Dolly-CL-Llama-3.2-1B",
    "ARMT-1k-16s-128-16mem-64-Dolly-CL-Llama-3.2-1B",

    "ARMT-1k-8s-128-16mem-64-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_short/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_16x128_1792_64/checkpoint-100/pytorch_model.bin/qa4_0k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_short/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_16x128_1792_64/checkpoint-100/pytorch_model.bin/qa4_1k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_short/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_16x128_1792_64/checkpoint-100/pytorch_model.bin/qa4_2k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_short/armt_all_s_ift_from_base_hf_train

,Llama-3.2-1B,Dolly-Llama-3.2-1B,ARMT-1k-8s-16mem-64-Dolly-NoMix-Llama-3.2-1B,ARMT-1k-2s-128-16mem-64-Dolly-CL-Llama-3.2-1B,ARMT-1k-4s-128-16mem-64-Dolly-CL-Llama-3.2-1B,ARMT-1k-8s-128-16mem-64-Dolly-CL-Llama-3.2-1B,ARMT-1k-16s-128-16mem-64-Dolly-CL-Llama-3.2-1B,ARMT-1k-8s-128-16mem-64-Dolly-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,66.10,62.10,57.30,53.40,52.70,51.00,nan,70.70
qa1_1k,2.90,65.00,69.60,12.80,15.00,15.40,11.50,nan,66.50
qa1_2k,20.90,65.10,24.70,5.20,7.40,9.00,6.70,nan,59.80
qa1_4k,41.20,57.60,15.30,2.90,4.10,4.40,3.40,nan,57.50
qa1_8k,48.90,44.30,6.70,1.80,3.20,3.10,3.20,nan,55.20
qa1_16k,38.90,37.90,3.20,1.70,2.90,2.40,2.10,nan,45.40
qa2_0k,0.00,44.14,45.15,27.23,25.33,21.92,22.22,nan,45.35
qa2_1k,6.31,33.13,32.43,9.11,10.91,11.71,8.11,nan,35.34
qa2_2k,20.82,30.83,23.62,2.40,4.30,4.00,3.20,nan,26.93
qa2_4k,24.92,27.13,12.51,1.60,3.30,3.60,2.70,nan,20.02


### Eval short 256tok segms on Dolly

In [338]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x256_480_64/checkpoint-300/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_3x256_720_64/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_5x256_1200_64/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x256_1920_64/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_16x256_3840_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-8s-16mem-64-Dolly-NoMix-Llama-3.2-1B",
    "ARMT-1k-2s-256-16mem-64-Dolly-CL-Llama-3.2-1B",
    "ARMT-1k-3s-256-16mem-64-Dolly-CL-Llama-3.2-1B",
    "ARMT-1k-5s-256-16mem-64-Dolly-CL-Llama-3.2-1B",
    "ARMT-1k-8s-256-16mem-64-Dolly-CL-Llama-3.2-1B",
    "ARMT-1k-16s-256-16mem-64-Dolly-CL-Llama-3.2-1B",

    #"ARMT-1k-8s-128-16mem-64-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_16x256_3840_64/checkpoint-100/pytorch_model.bin/qa5_0k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_16x256_3840_64/checkpoint-100/pytorch_model.bin/qa5_1k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_16x256_3840_64/checkpoint-100/pytorch_model.bin/qa5_2k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_256/armt_all_s_ift_from_base_hf_trainer/dolly

,Llama-3.2-1B,Dolly-Llama-3.2-1B,ARMT-1k-8s-16mem-64-Dolly-NoMix-Llama-3.2-1B,ARMT-1k-2s-256-16mem-64-Dolly-CL-Llama-3.2-1B,ARMT-1k-3s-256-16mem-64-Dolly-CL-Llama-3.2-1B,ARMT-1k-5s-256-16mem-64-Dolly-CL-Llama-3.2-1B,ARMT-1k-8s-256-16mem-64-Dolly-CL-Llama-3.2-1B,ARMT-1k-16s-256-16mem-64-Dolly-CL-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,66.10,62.10,38.20,38.40,37.20,37.90,36.80,70.70
qa1_1k,2.90,65.00,69.60,15.10,17.00,15.60,14.60,10.80,66.50
qa1_2k,20.90,65.10,24.70,14.00,15.70,14.80,13.30,10.20,59.80
qa1_4k,41.20,57.60,15.30,8.10,9.20,9.20,8.10,5.00,57.50
qa1_8k,48.90,44.30,6.70,7.10,6.80,6.70,5.50,3.00,55.20
qa1_16k,38.90,37.90,3.20,5.00,5.10,4.20,3.70,1.50,45.40
qa2_0k,0.00,44.14,45.15,33.23,33.13,33.83,33.83,32.43,45.35
qa2_1k,6.31,33.13,32.43,12.91,12.61,13.41,14.31,11.71,35.34
qa2_2k,20.82,30.83,23.62,10.91,10.41,11.51,11.21,7.71,26.93
qa2_4k,24.92,27.13,12.51,6.71,5.51,6.21,6.51,5.21,20.02


### Eval Llama-base-1b, compare evaluation

In [335]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",

    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
    #"unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-base-IEP",
    "Llama-3.2-1B-base-IP",
    "Llama-3.2-1B-base-Zero",
    "Llama-3.2-1B-base-E",
    "Llama-3.2-1B-base-I",
    "Llama-3.2-1B-base-P",

    "Llama-3.2-1B-IT-IEPC",
    "Llama-3.2-1B-IT-IPC",
    "Llama-3.2-1B-IT-Zero-noC",
    "Llama-3.2-1B-IT-E-noC",
    "Llama-3.2-1B-IT-EC",
    "Llama-3.2-1B-IT-C",

    #"Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    table_name_mod = disp_names[dat_idx]
    if "IEP" in table_name_mod and "IT" not in table_name_mod:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "IP" in table_name_mod and "IT" not in table_name_mod:
        prompt_name = 'instruction_yes_examples_no_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "I" in table_name_mod and "IT" not in table_name_mod:
        prompt_name = 'instruction_yes_examples_no_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "P" in table_name_mod and "IT" not in table_name_mod:
        prompt_name = 'instruction_no_examples_no_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "E" in table_name_mod and "IT" not in table_name_mod:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Zero" in table_name_mod and "IT" not in table_name_mod:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "IEPC" in table_name_mod and "IT" in table_name_mod:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "IPC" in table_name_mod and "IT" in table_name_mod:
        prompt_name = 'instruction_yes_examples_no_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "E" in table_name_mod and "IT" in table_name_mod and "noC" in table_name_mod:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "EC" in table_name_mod and "IT" in table_name_mod:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Zero" in table_name_mod and "IT" in table_name_mod:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "IC" in table_name_mod and "IT" in table_name_mod and table_name_mod == "Llama-3.2-1B-IT-IC":
        prompt_name = 'instruction_yes_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "I" in table_name_mod and "IT" in table_name_mod and table_name_mod == "Llama-3.2-1B-IT-C":
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-base-IEP,Llama-3.2-1B-base-IP,Llama-3.2-1B-base-Zero,Llama-3.2-1B-base-E,Llama-3.2-1B-base-I,Llama-3.2-1B-base-P,Llama-3.2-1B-IT-IEPC,Llama-3.2-1B-IT-IPC,Llama-3.2-1B-IT-Zero-noC,Llama-3.2-1B-IT-E-noC,Llama-3.2-1B-IT-EC,Llama-3.2-1B-IT-C
qa1_0k,0.00,4.80,44.90,58.20,3.40,3.20,70.70,79.20,44.60,0.20,0.00,50.20
qa1_1k,2.90,3.50,31.10,71.90,8.50,28.80,66.50,63.60,8.00,43.50,54.20,14.80
qa1_2k,20.90,11.00,32.10,70.10,11.60,32.80,59.80,55.90,7.30,25.90,38.50,9.60
qa1_4k,41.20,22.00,32.20,62.40,23.90,37.50,57.50,48.10,10.50,10.50,16.00,7.30
qa1_8k,48.90,28.50,32.30,52.00,27.00,35.90,55.20,43.80,10.70,12.20,14.70,6.80
qa1_16k,38.90,25.50,27.80,47.20,22.80,34.20,45.40,34.10,7.60,7.20,9.20,4.00
qa2_0k,0.00,0.10,17.72,33.83,0.10,0.00,45.35,40.54,37.94,0.10,0.30,17.22
qa2_1k,6.31,2.40,22.32,32.23,2.20,13.21,35.34,30.23,3.70,13.61,23.32,11.21
qa2_2k,20.82,12.61,20.12,29.13,9.11,18.12,26.93,19.52,4.50,9.41,22.32,12.51
qa2_4k,24.92,19.52,20.42,26.23,19.22,19.72,20.02,12.41,5.81,6.11,17.02,11.61


### Compare results for Flan with LLama1b models after IFT and ARMT on CL

In [430]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-98000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    "ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",
    "ARMT-1k-16mem-64-5s-CL-FLAN-48k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-5s-CL-FLAN-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B,ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B,ARMT-1k-16mem-64-5s-CL-FLAN-48k-Llama-3.2-1B,ARMT-1k-16mem-64-5s-CL-FLAN-Llama-3.2-1B,ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT,Llama-3.2-1B-IT
qa1_0k,0.00,66.10,57.00,65.30,62.10,59.80,62.00,62.00,61.50,76.30,70.70
qa1_1k,2.90,65.00,47.40,64.80,62.70,50.80,50.80,52.50,52.40,61.40,66.50
qa1_2k,20.90,65.10,45.70,28.00,7.20,48.40,46.90,47.40,46.60,41.90,59.80
qa1_4k,41.20,57.60,43.30,16.50,2.10,30.20,31.30,25.80,25.90,27.30,57.50
qa1_8k,48.90,44.30,39.70,10.00,0.10,9.20,18.20,14.50,13.50,16.20,55.20
qa1_16k,38.90,37.90,34.80,5.30,0.20,0.50,6.40,7.00,5.60,5.90,45.40
qa2_0k,0.00,44.14,35.94,43.54,34.63,38.44,38.24,38.64,39.14,44.34,45.35
qa2_1k,6.31,33.13,14.01,33.33,29.23,18.72,21.82,23.02,23.52,32.43,35.34
qa2_2k,20.82,30.83,18.72,23.12,9.01,17.22,17.72,18.52,18.92,15.12,26.93
qa2_4k,24.92,27.13,14.61,11.41,1.10,8.11,8.31,9.11,8.11,7.71,20.02


### 2 segm baselines for DPO comparison

In [351]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model.safetensors",
    "meta-llama/Llama-3.2-1B/runs/rmt_ift_from_pretrain/sharegpt/meta-llama/Llama-3.2-1B/real_RMT-Llama-3.2-1B-2x1024-mem16-pg19/model_best/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "2s-LM-RMT-1k-Llama-3.2-1B-WO-chat",
    "SGPT-2s-LM-RMT-1k-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B-IT",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-2x1024-mem16-pg19/model.safetensors" in mod_name or mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k']#, '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_2k', 'qa2_2k', 'qa3_2k', 'qa4_2k', 'qa5_2k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B,Dolly-Llama-3.2-1B,2s-LM-RMT-1k-Llama-3.2-1B-WO-chat,SGPT-2s-LM-RMT-1k-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B-IT,ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,0.00,66.10,1.30,24.90,73.60,59.80,70.70
qa1_1k,2.90,65.00,8.80,13.00,58.20,50.80,66.50
qa1_2k,20.90,65.10,5.30,4.50,35.20,48.40,59.80
qa2_0k,0.00,44.14,2.60,2.50,45.25,38.44,45.35
qa2_1k,6.31,33.13,6.71,4.00,31.03,18.72,35.34
qa2_2k,20.82,30.83,4.40,3.70,13.71,17.22,26.93
qa3_0k,0.00,22.22,1.90,2.20,33.43,30.53,34.73
qa3_1k,0.81,16.24,13.34,7.89,13.69,11.02,36.08
qa3_2k,5.31,15.03,6.51,3.81,9.32,15.23,31.06
qa4_0k,0.00,26.03,34.73,0.00,15.92,15.52,9.51


### Only LM-ARMT results

In [377]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B",
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    "ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E",
    "ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E",
    "ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E",

    #"ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-DollyMix05-8s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or "data/pretrained_models" in mod_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B,ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E,Llama-3.2-1B-IT
qa1_0k,0.00,58.20,66.10,65.30,37.30,25.90,19.50,48.90,44.60,29.80,18.70,52.40,70.70
qa1_1k,2.90,71.90,65.00,64.80,35.70,29.90,20.80,56.50,40.60,35.40,32.70,58.00,66.50
qa1_2k,20.90,70.10,65.10,28.00,42.90,37.10,36.70,34.00,39.30,37.10,36.40,32.90,59.80
qa1_4k,41.20,62.40,57.60,16.50,27.50,26.10,19.00,1.00,25.90,25.20,21.10,1.70,57.50
qa1_8k,48.90,52.00,44.30,10.00,0.00,0.00,0.00,0.10,0.00,0.00,0.00,0.00,55.20
qa1_16k,38.90,47.20,37.90,5.30,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,45.40
qa2_0k,0.00,33.83,44.14,43.54,36.84,35.94,34.43,33.03,34.43,32.23,26.73,30.93,45.35
qa2_1k,6.31,32.23,33.13,33.33,20.82,24.82,21.12,26.93,25.43,24.32,23.72,27.03,35.34
qa2_2k,20.82,29.13,30.83,23.12,17.32,16.62,16.52,15.92,15.22,15.12,15.42,13.91,26.93
qa2_4k,24.92,26.23,27.13,11.41,12.91,13.31,13.01,1.90,11.41,12.61,11.11,1.70,20.02


### LM-ARMT results

In [425]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #"meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_4s_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-300/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #"Llama-3.2-1B",
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    "ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E",
    "ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E",
    "ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E",

    "ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-31.5k-Dolly-2s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-49.5k-Dolly-2s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-4s-LM-19.5k-Dolly-4s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-DollyMix05-8s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or "data/pretrained_models" in mod_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-31.5k-Dolly-2s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-49.5k-Dolly-2s-Llama-3.2-1B,ARMT-1k-16mem-64-4s-LM-19.5k-Dolly-4s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-DollyMix05-8s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,57.00,65.30,44.60,29.80,18.70,52.40,67.80,67.70,67.30,55.40,62.40,57.40,70.70
qa1_1k,71.90,65.00,47.40,64.80,40.60,35.40,32.70,58.00,62.80,63.70,62.80,61.00,61.10,56.40,66.50
qa1_2k,70.10,65.10,45.70,28.00,39.30,37.10,36.40,32.90,31.40,35.30,30.50,24.00,33.00,38.70,59.80
qa1_4k,62.40,57.60,43.30,16.50,25.90,25.20,21.10,1.70,11.70,14.10,11.60,13.20,14.30,20.50,57.50
qa1_8k,52.00,44.30,39.70,10.00,0.00,0.00,0.00,0.00,4.30,4.60,3.70,8.70,5.30,17.90,55.20
qa1_16k,47.20,37.90,34.80,5.30,0.00,0.00,0.00,0.00,0.70,0.30,1.10,3.40,1.60,6.20,45.40
qa2_0k,33.83,44.14,35.94,43.54,34.43,32.23,26.73,30.93,43.14,42.84,43.04,36.84,41.24,33.43,45.35
qa2_1k,32.23,33.13,14.01,33.33,25.43,24.32,23.72,27.03,33.23,32.83,33.13,30.03,32.93,17.42,35.34
qa2_2k,29.13,30.83,18.72,23.12,15.22,15.12,15.42,13.91,16.62,16.12,15.92,15.62,20.12,12.71,26.93
qa2_4k,26.23,27.13,14.61,11.41,11.41,12.61,11.11,1.70,6.91,5.71,7.21,9.71,9.71,7.11,20.02


### LM-ARMT results, small

In [171]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #"meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx2048_step19500_prolong.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx2048_step49500_prolong.bin",
    'meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_sliding_llama3.2-1B_ctx512x4_step24000_pg19.bin',
    'meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_sliding_llama3.2-1B_ctx512x4_step24000_pg19.bin',
    'meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_sliding_llama3.2-1B_ctx512x4_step38500_pg19.bin',
    'meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_sliding_llama3.2-1B_ctx512x4_step38500_pg19.bin',
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_4s_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #"Llama-3.2-1B",
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E",
    "ARMT-1k-32mem-64-2s-LM-19.5k-Prolong-Llama-3.2-1B-E",
    "ARMT-1k-32mem-64-2s-LM-49.5k-Prolong-Llama-3.2-1B-E",
    "ARMT-1k-32mem-4-2s-sliding-LM-24k-PG19-Llama-3.2-1B-E",
    "ARMT-1k-32mem-4-2s-sliding-LM-24k-PG19-Llama-3.2-1B-E-Ans",
    "ARMT-1k-32mem-4-2s-sliding-LM-38.5k-PG19-Llama-3.2-1B-E",
    "ARMT-1k-32mem-4-2s-sliding-LM-38.5k-PG19-Llama-3.2-1B-E-Ans",

    #"ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Dolly-4s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-DollyMix05-8s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E-Ans" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or "data/pretrained_models" in mod_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E,ARMT-1k-32mem-64-2s-LM-19.5k-Prolong-Llama-3.2-1B-E,ARMT-1k-32mem-64-2s-LM-49.5k-Prolong-Llama-3.2-1B-E,ARMT-1k-32mem-4-2s-sliding-LM-24k-PG19-Llama-3.2-1B-E,ARMT-1k-32mem-4-2s-sliding-LM-24k-PG19-Llama-3.2-1B-E-Ans,ARMT-1k-32mem-4-2s-sliding-LM-38.5k-PG19-Llama-3.2-1B-E,ARMT-1k-32mem-4-2s-sliding-LM-38.5k-PG19-Llama-3.2-1B-E-Ans,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,57.00,65.30,44.60,35.30,23.20,70.90,71.70,64.70,68.70,72.90
qa1_1k,71.90,65.00,47.40,64.80,40.60,34.30,31.30,46.20,52.70,41.40,48.20,66.80
qa1_2k,70.10,65.10,45.70,28.00,39.30,37.50,39.80,28.00,32.10,23.50,29.00,60.90
qa1_4k,62.40,57.60,43.30,16.50,25.90,0.20,0.20,11.50,15.20,6.70,12.80,58.30
qa1_8k,52.00,44.30,39.70,10.00,0.00,0.00,0.00,5.70,11.40,5.30,8.70,55.60
qa1_16k,47.20,37.90,34.80,5.30,0.00,0.00,0.00,5.30,6.80,0.30,0.80,45.60
qa2_0k,33.83,44.14,35.94,43.54,34.43,25.43,26.13,36.94,37.34,33.23,35.34,45.15
qa2_1k,32.23,33.13,14.01,33.33,25.43,18.02,14.41,24.72,23.82,22.22,23.22,35.34
qa2_2k,29.13,30.83,18.72,23.12,15.22,10.91,12.41,14.41,15.92,11.61,14.21,27.33
qa2_4k,26.23,27.13,14.61,11.41,11.41,0.10,0.00,4.90,5.61,4.60,5.71,20.42


### Poster results

In [395]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #"meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #"Llama-3.2-1B",
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E",

    #"ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Dolly-2s-Llama-3.2-1B",
    "ARMT-16-64-3s-FLAN-Llama-3.2-1B",
    "ARMT-16-64-8s-DollyMix-5%-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or "data/pretrained_models" in mod_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
cmap = "BuGn"
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
s.set_table_styles([
    {"selector": "tr", "props": "line-height: 15px;"},
    {"selector": "td,th", "props": "line-height: inherit; padding: 5px;"}
])
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,ARMT-16-64-3s-FLAN-Llama-3.2-1B,ARMT-16-64-8s-DollyMix-5%-Llama-3.2-1B-IT,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,57.00,62.00,76.30,70.70
qa1_1k,71.90,65.00,47.40,50.80,61.40,66.50
qa1_2k,70.10,65.10,45.70,46.90,41.90,59.80
qa1_4k,62.40,57.60,43.30,31.30,27.30,57.50
qa1_8k,52.00,44.30,39.70,18.20,16.20,55.20
qa1_16k,47.20,37.90,34.80,6.40,5.90,45.40
qa2_0k,33.83,44.14,35.94,38.24,44.34,45.35
qa2_1k,32.23,33.13,14.01,21.82,32.43,35.34
qa2_2k,29.13,30.83,18.72,17.72,15.12,26.93
qa2_4k,26.23,27.13,14.61,8.31,7.71,20.02


### Question first results

In [437]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_quest_first'
results_folder_regular = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"meta-llama/Llama-3.2-1B",
    #"meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",

    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-Regular-E",
    #"Llama-3.2-1B",
    #"Llama-3.2-1B-E",
    "Llama-3.2-1B-Zero",
    "Llama-3.2-1B-Zero-Q2",
    #"DollyLoraOld-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    #"Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E",

    "Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-DollyMix05-8s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT",
    "ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B-IT-Q2",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT-Q2",
    "ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT-Q2",
    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
    "Llama-3.2-1B-IT-Q2",
    "Llama-3.2-1B-IT-Regular",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "Regular-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Regular" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-Zero-Q2" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_quest_first_2_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-IT-Q2" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_quest_first_2_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-Zero" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or "data/pretrained_models" in mod_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "Regular" in disp_name:
                    fname = f'./{results_folder_regular}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                #elif "mix" in model_name or "mergekit" in model_name:
                #    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-Regular-E,Llama-3.2-1B-Zero,Llama-3.2-1B-Zero-Q2,Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B,ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT,ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT,ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B-IT-Q2,ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT-Q2,ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT-Q2,Llama-3.2-1B-IT,Llama-3.2-1B-IT-Q2,Llama-3.2-1B-IT-Regular
qa1_0k,58.20,0.00,21.10,59.20,57.20,68.20,53.00,51.60,50.00,54.90,54.10,54.60,60.70,60.80,70.70
qa1_1k,71.90,0.00,7.50,14.90,16.10,51.80,33.30,38.90,39.00,51.70,43.40,45.50,49.90,59.60,66.50
qa1_2k,70.10,0.10,3.70,1.20,3.80,0.60,4.10,0.90,1.10,39.60,47.10,46.50,40.30,51.70,59.80
qa1_4k,62.40,0.00,7.70,0.30,1.10,0.80,1.30,0.10,0.80,20.70,30.50,31.60,41.20,46.10,57.50
qa1_8k,52.00,0.00,4.50,0.00,0.30,0.40,0.50,0.20,0.70,18.00,9.00,18.10,31.80,40.20,55.20
qa1_16k,47.20,0.00,3.40,0.00,0.10,0.30,0.60,0.10,0.40,8.10,0.60,5.50,20.80,31.10,45.40
qa2_0k,33.83,0.00,9.81,33.23,22.22,37.04,34.43,37.04,36.54,27.93,31.43,32.33,38.44,41.94,45.35
qa2_1k,32.23,0.00,6.21,24.02,21.82,26.93,17.22,23.42,21.42,22.22,19.82,21.32,31.13,33.63,35.34
qa2_2k,29.13,0.00,3.90,4.30,3.40,0.60,4.90,1.70,2.20,13.21,16.42,17.02,20.12,29.73,26.93
qa2_4k,26.23,0.00,5.01,0.60,1.50,0.30,2.80,0.10,1.40,7.11,7.41,8.21,14.11,21.52,20.02


### ARMT Layer size ablation

In [411]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-700/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_128/checkpoint-300/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_256/checkpoint-300/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    "DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-Dolly-Llama-3.2-1B",
    "ARMT-1k-1mem-64-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-16mem-64-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-all-s-16mem-128-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-all-s-16mem-256-Dolly-Llama-3.2-1B",
    
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=0, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)

,Llama-3.2-1B-E,DollyLoraOld-Llama-3.2-1B,Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-Dolly-Llama-3.2-1B,ARMT-1k-1mem-64-Dolly-Llama-3.2-1B,ARMT-1k-2s-16mem-64-Dolly-Llama-3.2-1B,ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B,ARMT-1k-2s-all-s-16mem-128-Dolly-Llama-3.2-1B,ARMT-1k-2s-all-s-16mem-256-Dolly-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,58.20,66.90,66.10,64.80,65.40,37.20,62.60,64.60,64.80,70.70
qa1_1k,71.90,65.30,65.00,64.70,65.20,63.80,65.00,64.50,63.80,66.50
qa1_2k,70.10,59.20,65.10,22.10,20.70,30.60,34.60,29.20,26.60,59.80
qa1_4k,62.40,53.00,57.60,13.60,12.70,18.20,23.70,17.00,14.20,57.50
qa1_8k,52.00,33.50,44.30,9.40,7.60,10.70,14.40,9.40,8.50,55.20
qa1_16k,47.20,31.90,37.90,5.70,5.70,3.70,6.30,5.00,3.70,45.40
qa2_0k,33.83,44.94,44.14,44.44,45.85,21.92,41.84,43.44,43.04,45.35
qa2_1k,32.23,34.93,33.13,32.63,33.73,28.93,32.43,33.23,33.43,35.34
qa2_2k,29.13,34.83,30.83,19.92,21.52,23.12,25.63,22.52,22.62,26.93
qa2_4k,26.23,30.53,27.13,10.21,11.21,11.51,14.41,13.41,12.91,20.02


### Loss masking ablation

In [439]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/TEST_LOSS_MASK_armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    "ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    "ARMT-1k-2s-16mem-64-Dolly-CompLoss-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-CompLoss-Llama-3.2-1B",
    
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,ARMT-1k-2s-all-s-16mem-64-Dolly-Llama-3.2-1B,ARMT-1k-2s-16mem-64-Dolly-CompLoss-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-CompLoss-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,62.60,60.00,44.60,67.80,65.70,70.70
qa1_1k,71.90,65.00,65.00,67.10,40.60,62.80,64.60,66.50
qa1_2k,70.10,65.10,34.60,34.90,39.30,31.40,39.70,59.80
qa1_4k,62.40,57.60,23.70,18.00,25.90,11.70,25.50,57.50
qa1_8k,52.00,44.30,14.40,9.40,0.00,4.30,3.30,55.20
qa1_16k,47.20,37.90,6.30,2.60,0.00,0.70,0.00,45.40
qa2_0k,33.83,44.14,41.84,41.64,34.43,43.14,43.64,45.35
qa2_1k,32.23,33.13,32.43,30.33,25.43,33.23,31.33,35.34
qa2_2k,29.13,30.83,25.63,23.52,15.22,16.62,17.92,26.93
qa2_4k,26.23,27.13,14.41,10.81,11.41,6.91,7.71,20.02


### Flan updates

In [43]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-48000-inter/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-98000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64-inter/checkpoint-4000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    "ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",
    "ARMT-1k-16mem-64-5s-CL-FLAN-48k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-5s-CL-FLAN-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-CL-FLAN-4k-Llama-3.2-1B",
    "ARMT-1k-16mem-64-8s-CL-FLAN-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Flan-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Flan-Llama-3.2-1B,ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B,ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B,ARMT-1k-16mem-64-5s-CL-FLAN-48k-Llama-3.2-1B,ARMT-1k-16mem-64-5s-CL-FLAN-Llama-3.2-1B,ARMT-1k-16mem-64-8s-CL-FLAN-4k-Llama-3.2-1B,ARMT-1k-16mem-64-8s-CL-FLAN-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E,ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Flan-2s-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,58.20,57.00,59.80,62.00,62.00,61.50,61.90,61.50,44.60,57.40,57.60,72.90
qa1_1k,71.90,47.40,50.80,50.80,52.50,52.40,52.40,53.90,40.60,56.40,55.90,66.80
qa1_2k,70.10,45.70,48.40,46.90,47.40,46.60,47.10,47.50,39.30,38.70,34.50,60.90
qa1_4k,62.40,43.30,30.20,31.30,25.80,25.90,27.50,24.50,25.90,20.50,16.20,58.30
qa1_8k,52.00,39.70,9.20,18.20,14.50,13.50,15.20,13.50,0.00,17.90,14.60,55.60
qa1_16k,47.20,34.80,0.50,6.40,7.00,5.60,6.80,5.80,0.00,6.20,5.20,45.60
qa2_0k,33.83,35.94,38.44,38.24,38.64,39.14,39.04,39.14,34.43,33.43,34.83,45.15
qa2_1k,32.23,14.01,18.72,21.82,23.02,23.52,23.62,24.72,25.43,17.42,17.72,35.34
qa2_2k,29.13,18.72,17.22,17.72,18.52,18.92,18.42,19.32,15.22,12.71,12.41,27.33
qa2_4k,26.23,14.61,8.11,8.31,9.11,8.11,8.01,7.31,11.41,7.11,6.61,20.42


### Question first results - for Dolly, cause Dolly has questions first

In [441]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_quest_first'
results_folder_regular = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"meta-llama/Llama-3.2-1B",
    #"meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",

    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-Regular-E",
    #"Llama-3.2-1B",
    #"Llama-3.2-1B-E",
    "Llama-3.2-1B-Zero",
    "Llama-3.2-1B-Zero-Q2",
    #"DollyLoraOld-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    #"Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E",

    "Dolly-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B-Regular",
    "ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-DollyMix05-8s-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT",
    "ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    "ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B-IT-Q2",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    "ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT-Q2",
    "ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT-Q2",
    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
    "Llama-3.2-1B-IT-Q2",
    "Llama-3.2-1B-IT-Regular",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "Regular-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Regular" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-Zero-Q2" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_quest_first_2_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-IT-Q2" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_quest_first_2_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-Zero" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or "data/pretrained_models" in mod_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "Regular" in disp_name:
                    fname = f'./{results_folder_regular}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                #elif "mix" in model_name or "mergekit" in model_name:
                #    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-Regular-E,Llama-3.2-1B-Zero,Llama-3.2-1B-Zero-Q2,Dolly-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B-Regular,ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B,ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B,ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT,ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT,ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B-IT-Q2,ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT-Q2,ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT-Q2,Llama-3.2-1B-IT,Llama-3.2-1B-IT-Q2,Llama-3.2-1B-IT-Regular
qa1_0k,58.20,0.00,21.10,59.20,57.20,67.80,68.20,53.00,51.60,50.00,54.90,54.10,54.60,60.70,60.80,70.70
qa1_1k,71.90,0.00,7.50,14.90,16.10,62.80,51.80,33.30,38.90,39.00,51.70,43.40,45.50,49.90,59.60,66.50
qa1_2k,70.10,0.10,3.70,1.20,3.80,31.40,0.60,4.10,0.90,1.10,39.60,47.10,46.50,40.30,51.70,59.80
qa1_4k,62.40,0.00,7.70,0.30,1.10,11.70,0.80,1.30,0.10,0.80,20.70,30.50,31.60,41.20,46.10,57.50
qa1_8k,52.00,0.00,4.50,0.00,0.30,4.30,0.40,0.50,0.20,0.70,18.00,9.00,18.10,31.80,40.20,55.20
qa1_16k,47.20,0.00,3.40,0.00,0.10,0.70,0.30,0.60,0.10,0.40,8.10,0.60,5.50,20.80,31.10,45.40
qa2_0k,33.83,0.00,9.81,33.23,22.22,43.14,37.04,34.43,37.04,36.54,27.93,31.43,32.33,38.44,41.94,45.35
qa2_1k,32.23,0.00,6.21,24.02,21.82,33.23,26.93,17.22,23.42,21.42,22.22,19.82,21.32,31.13,33.63,35.34
qa2_2k,29.13,0.00,3.90,4.30,3.40,16.62,0.60,4.90,1.70,2.20,13.21,16.42,17.02,20.12,29.73,26.93
qa2_4k,26.23,0.00,5.01,0.60,1.50,6.91,0.30,2.80,0.10,1.40,7.11,7.41,8.21,14.11,21.52,20.02


### Lora ablations

In [67]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-9000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",

    'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",


    
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    "ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    "ARMT-16-64-8s-DollyMix05-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",
    "ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    "ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"Smoltalk-Llama-3.2-1B",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin/qa4_8k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin/qa4_16k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin/qa5_0k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/bab

,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B,ARMT-16-64-8s-DollyMix05-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B,ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,57.00,62.60,76.30,70.60,63.20,73.10,72.30,72.90,72.90,73.00,72.60,73.80,74.40,0.00,73.70,14.10,13.90,8.40,37.90,72.90
qa1_1k,71.90,65.00,47.40,65.00,61.40,60.00,65.10,62.90,64.40,64.40,63.50,63.10,63.40,63.50,62.90,0.00,66.20,6.60,17.30,11.50,2.60,66.80
qa1_2k,70.10,65.10,45.70,34.60,41.90,40.00,28.00,47.40,48.10,50.90,49.80,39.30,48.80,39.20,32.10,0.40,17.00,3.50,17.20,17.60,23.20,60.90
qa1_4k,62.40,57.60,43.30,23.70,27.30,21.00,13.80,34.00,26.70,35.20,35.50,13.70,27.50,26.60,28.70,0.30,8.10,1.90,4.40,4.80,4.00,58.30
qa1_8k,52.00,44.30,39.70,14.40,16.20,12.00,2.10,18.10,13.90,19.60,19.40,1.80,15.90,0.40,13.80,2.30,4.10,0.70,0.20,1.10,1.90,55.60
qa1_16k,47.20,37.90,34.80,6.30,5.90,5.00,0.30,8.90,7.90,10.40,10.40,0.20,8.20,0.00,7.00,16.40,1.90,0.90,0.10,0.20,1.10,45.60
qa2_0k,33.83,44.14,35.94,41.84,44.34,44.74,41.74,47.95,47.25,47.45,46.95,47.65,47.45,42.94,42.94,0.10,44.64,1.20,7.61,1.70,21.62,45.15
qa2_1k,32.23,33.13,14.01,32.43,32.43,29.53,30.13,35.84,35.94,36.04,35.94,34.03,35.64,32.33,32.63,0.50,34.53,2.50,5.61,4.80,6.01,35.34
qa2_2k,29.13,30.83,18.72,25.63,15.12,15.92,22.12,17.62,16.12,19.02,18.72,18.92,16.92,23.12,26.83,0.20,18.12,4.40,4.20,3.30,16.42,27.33
qa2_4k,26.23,27.13,14.61,14.41,7.71,9.51,11.81,9.81,11.11,12.71,12.41,9.41,10.01,12.71,16.52,0.60,9.51,0.70,0.70,0.70,5.41,20.42


### Loss & lora ablations

In [23]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-9000/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",

    'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",


    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-DollyMix05-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"Smoltalk-Llama-3.2-1B",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    

    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1",


    "Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    "Llama-3.2-1B-IT-Q1-only",
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin/qa5_8k_instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin/qa5_16k_instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes.csv


,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B,ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1,Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only,Llama-3.2-1B-IT-Q1-only,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,72.60,73.80,73.70,14.10,13.90,8.40,13.40,12.80,37.90,40.70,23.80,9.00,23.40,45.90,9.80,72.90
qa1_1k,71.90,65.00,63.40,63.50,66.20,6.60,17.30,11.50,10.60,35.30,2.60,16.00,41.40,4.10,16.10,51.60,0.00,66.80
qa1_2k,70.10,65.10,48.80,39.20,17.00,3.50,17.20,17.60,29.50,33.30,23.20,34.60,38.40,13.70,15.10,53.20,0.60,60.90
qa1_4k,62.40,57.60,27.50,26.60,8.10,1.90,4.40,4.80,16.90,7.90,4.00,9.90,18.00,9.60,6.80,38.40,0.60,58.30
qa1_8k,52.00,44.30,15.90,0.40,4.10,0.70,0.20,1.10,5.60,0.90,1.90,3.70,6.00,5.50,1.70,18.70,0.30,55.60
qa1_16k,47.20,37.90,8.20,0.00,1.90,0.90,0.10,0.20,0.90,0.00,1.10,1.90,2.60,4.10,0.80,31.70,0.40,45.60
qa2_0k,33.83,44.14,47.45,42.94,44.64,1.20,7.61,1.70,3.00,5.61,21.62,27.33,37.04,5.01,1.40,30.53,28.83,45.15
qa2_1k,32.23,33.13,35.64,32.33,34.53,2.50,5.61,4.80,0.80,7.41,6.01,9.81,16.12,4.40,2.70,10.81,9.51,35.34
qa2_2k,29.13,30.83,16.92,23.12,18.12,4.40,4.20,3.30,1.80,11.81,16.42,16.12,3.70,4.50,2.80,8.31,5.21,27.33
qa2_4k,26.23,27.13,10.01,12.71,9.51,0.70,0.70,0.70,2.40,4.00,5.41,4.40,2.70,2.10,1.50,10.21,0.30,20.42


### Loss ablations, bigger dataset

In [392]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-9000/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",

    #'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-10000/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora_ext/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64_1e-05_100000/checkpoint-88000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_ext/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_100000/checkpoint-85000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",


    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-DollyMix05-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"Smoltalk-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-no-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-no-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",


    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-1B-IT-from-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-no-Lora-CompLoss-Llama-3.2-1B-IT-from-IT-Q1-only",
    
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-no-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-no-Lora-CompLoss-Llama-IT-3.2-1B-Q1-only",
    

    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1",


    "Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    "Llama-3.2-1B-IT-Q1-only",
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-1B-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-no-Lora-CompLoss-Llama-3.2-1B-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-1B-IT-from-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-no-Lora-CompLoss-Llama-3.2-1B-IT-from-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-no-Lora-CompLoss-Llama-3.2-1B-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-no-Lora-CompLoss-Llama-IT-3.2-1B-Q1-only,Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only,Llama-3.2-1B-IT-Q1-only,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,13.40,12.80,0.20,32.10,23.80,24.70,45.70,24.40,49.50,45.90,9.80,72.90
qa1_1k,71.90,65.00,10.60,35.30,2.20,47.10,41.40,42.30,56.20,42.70,50.60,51.60,0.00,66.80
qa1_2k,70.10,65.10,29.50,33.30,39.90,35.20,38.40,44.00,42.00,30.00,36.80,53.20,0.60,60.90
qa1_4k,62.40,57.60,16.90,7.90,14.80,11.40,18.00,18.70,12.40,7.10,12.10,38.40,0.60,58.30
qa1_8k,52.00,44.30,5.60,0.90,0.80,2.60,6.00,6.70,2.60,1.90,1.70,18.70,0.30,55.60
qa1_16k,47.20,37.90,0.90,0.00,0.10,0.70,2.60,3.10,1.30,1.00,0.40,31.70,0.40,45.60
qa2_0k,33.83,44.14,3.00,5.61,0.00,4.30,37.04,24.42,40.24,12.51,32.63,30.53,28.83,45.15
qa2_1k,32.23,33.13,0.80,7.41,0.30,19.92,16.12,20.12,40.04,7.81,30.43,10.81,9.51,35.34
qa2_2k,29.13,30.83,1.80,11.81,7.81,8.61,3.70,8.81,6.91,8.21,15.92,8.31,5.21,27.33
qa2_4k,26.23,27.13,2.40,4.00,3.70,3.90,2.70,3.90,2.80,1.90,4.00,10.21,0.30,20.42


### Double lora ablations

In [16]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-9000/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",

    #'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",

    "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    "meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_4s_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-10000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_loss_ablation/smol_lora_comp_la_2s_synth_1k_2k_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64/checkpoint-10000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_loss_ablation/smol_lora_comp_la_2s_synth_1k_2k_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64/checkpoint-10000/pytorch_model.bin",


    
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    #"Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Dolly-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-DollyMix05-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"Smoltalk-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",

    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1",

    "ARMT-32-64-4s-BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-32-64-4s-BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-32-64-4s-Smol_from_BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    "ARMT-32-64-2s-Smol_from_DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-32-64-2s-Smol_from_DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_loss_ablation/smol_lora_comp_la_2s_synth_1k_2k_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64/checkpoint-10000/pytorch_model.bin/qa4_16k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_loss_ablation/smol_lora_comp_la_2s_synth_1k_2k_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64/checkpoint-10000/pytorch_model.bin/qa5_0k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_loss_ablation/smol_lora_comp_la_2s_synth_1k_2k_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64/checkpoint-10000/pytorch_model.bin/qa5_1k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_ye

,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-32-64-4s-BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-32-64-4s-BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT-CT,ARMT-32-64-4s-Smol_from_BL_qa1-5-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1,ARMT-32-64-2s-Smol_from_DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-32-64-2s-Smol_from_DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,72.60,73.80,94.90,81.50,86.20,37.90,40.70,73.80,70.80,72.90
qa1_1k,71.90,65.00,63.40,63.50,96.80,94.20,91.50,2.60,16.00,62.20,58.00,66.80
qa1_2k,70.10,65.10,48.80,39.20,87.60,86.00,66.70,23.20,34.60,37.00,4.10,60.90
qa1_4k,62.40,57.60,27.50,26.60,37.60,34.70,40.60,4.00,9.90,5.90,0.30,58.30
qa1_8k,52.00,44.30,15.90,0.40,8.70,8.50,20.80,1.90,3.70,1.60,0.10,55.60
qa1_16k,47.20,37.90,8.20,0.00,2.00,1.50,1.40,1.10,1.90,0.50,0.00,45.60
qa2_0k,33.83,44.14,47.45,42.94,42.74,43.14,73.07,21.62,27.33,49.65,41.74,45.15
qa2_1k,32.23,33.13,35.64,32.33,56.36,58.76,59.26,6.01,9.81,34.33,36.84,35.34
qa2_2k,29.13,30.83,16.92,23.12,63.26,68.07,39.04,16.42,16.12,17.82,0.70,27.33
qa2_4k,26.23,27.13,10.01,12.71,32.13,37.44,25.63,5.41,4.40,4.00,0.20,20.42


### Evaluation debug

In [300]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #  "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #  "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"DEBUG_eval_versions/orig_eval/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16_no_autocast/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16_no_autocast_ainst/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #  "DEBUG_eval_versions/orig_eval_bf16_no_autocast_ainst_right/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    
    #"DEBUG_eval_versions/orig_eval_latest/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",

    #"DEBUG_eval_versions/orig_eval_base/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",

    #"DEBUG_eval_versions/our_eval_1k/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_30_gen/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_no_corr/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_corr/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_1k_seg/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_it_base/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_our_segm/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_it_base_our_segm_bf16/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_our_segm_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_orig_segm_30_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",

    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_after_dev_ev/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_our_load/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_orig_load_cpu/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_our_layers/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-500/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",

    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-400/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-800/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no_mem/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",

    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",
    
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    
    
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_correction/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    "DEBUG_eval_versions/our_eval_100_it_base_orig_segm_30_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5_ct-v3.1/model.safetensors",

    #"DEBUG_eval_versions/our_eval_100_it_base_it_1b_tok/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",


    #"DEBUG_eval_versions/armt_it_no_lora_base/meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_it/unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_base_bf16/meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_it_bf16/unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #  "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    #  "Dolly-Llama-3.2-1B",

    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast_ainst-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast_ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_latest-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_base-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_1k-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen_no_corr-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen_corr-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_1k_seg-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-bf16-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-ainst_right-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",

    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_dev_ev-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_our_load-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_orig_load_cpu-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_our_layers-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT-CT",

    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-200-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-400-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-800-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-1000-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4-200-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4_no_mem-200-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-2kSynth_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_corr-CT",
    "ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",
    #"ARMT-Smol-2s-Llama-3.2-1B-IT_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_it_1b_tok-CT",

    #"ARMT-IT-QF-no-Lora-base-Q1-only",
    #"ARMT-IT-QF-no-Lora-IT-Q1-only",
    #"ARMT-IT-QF-no-Lora-base-Q1-only_bf16",
    #"ARMT-IT-QF-no-Lora-IT-Q1-only_bf16",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
#for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
for l0 in ['qa1_32k', 'qa2_32k', 'qa3_32k', 'qa4_32k', 'qa5_32k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no_mem/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/qa4_32k_instruction_no_examples_no_post_prompt_no_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no_mem/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/qa5_0k_instruction_no_examples_no_post_prompt_no_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no

,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-200-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-400-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-800-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-1000-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4-200-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4_no_mem-200-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-2kSynth_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT,ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_our_eval_100_IT-base_orig_segm_30-ainst_right-CT,Llama-3.2-1B-IT
qa1_0k,100.00,99.00,100.00,100.00,100.00,100.00,100.00,66.00,100.00,100.00,100.00,72.90
qa1_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,53.00,100.00,100.00,100.00,66.80
qa1_2k,96.00,90.00,100.00,100.00,100.00,100.00,97.00,27.00,99.00,100.00,100.00,60.90
qa1_4k,87.00,74.00,100.00,100.00,100.00,100.00,99.00,8.00,100.00,100.00,100.00,58.30
qa1_8k,86.00,69.00,100.00,100.00,100.00,100.00,97.00,13.00,97.00,100.00,100.00,55.60
qa1_16k,84.00,78.00,99.00,99.00,99.00,99.00,80.00,12.00,76.00,98.00,100.00,45.60
qa1_32k,41.00,49.00,99.00,73.00,67.00,68.00,34.00,11.00,18.00,48.00,100.00,28.50
qa2_0k,99.00,98.00,100.00,100.00,100.00,100.00,99.00,48.00,99.00,100.00,100.00,45.15
qa2_1k,99.00,98.00,100.00,100.00,100.00,100.00,100.00,59.00,100.00,100.00,100.00,35.34
qa2_2k,95.00,90.00,100.00,100.00,100.00,100.00,99.00,41.00,100.00,100.00,100.00,27.33


### Evaluation debug, tune from BL model

In [444]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #  "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #  "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"DEBUG_eval_versions/orig_eval/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16_no_autocast/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16_no_autocast_ainst/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #  "DEBUG_eval_versions/orig_eval_bf16_no_autocast_ainst_right/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    
    #"DEBUG_eval_versions/orig_eval_latest/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",

    #"DEBUG_eval_versions/orig_eval_base/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",

    #"DEBUG_eval_versions/our_eval_1k/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_30_gen/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_no_corr/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_corr/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_1k_seg/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_it_base/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_our_segm/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_it_base_our_segm_bf16/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_our_segm_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_orig_segm_30_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",

    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_after_dev_ev/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_our_load/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_orig_load_cpu/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_our_layers/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-500/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",

    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-400/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-800/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",
    # "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no_mem/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",

    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",
    
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",
    #"DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1s_local/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_1x1024_1008_64/checkpoint-2500/pytorch_model.bin",
    #"DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_cont/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2000/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_new/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",
    #"DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_new_on_merge/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",

    
    
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_correction/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    "DEBUG_eval_versions/our_eval_100_it_base_orig_segm_30_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5_ct-v3.1/model.safetensors",

    #"DEBUG_eval_versions/our_eval_100_it_base_it_1b_tok/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",


    #"DEBUG_eval_versions/armt_it_no_lora_base/meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_it/unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_base_bf16/meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_it_bf16/unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #  "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    #  "Dolly-Llama-3.2-1B",

    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast_ainst-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast_ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_latest-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_base-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_1k-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen_no_corr-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen_corr-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_1k_seg-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-bf16-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-ainst_right-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",

    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_dev_ev-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_our_load-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_orig_load_cpu-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_our_layers-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT-CT",

    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-200-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-400-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-800-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-1000-CompLoss-Llama-3.2-1B-IT-CT",
    # "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4-200-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4_no_mem-200-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-2kSynth_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    #"ARMT-16-64-1s-Smol_from-BL-QA1-5_v3.1-unmergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    #"ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-contLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-unmergedLora-ArmtLora-CompLoss-Llama-3.2-1B-IT-CT",
    #"ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-ArmtLora-CompLoss-Llama-3.2-1B-IT-CT",

    
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_corr-CT",
    "ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",
    #"ARMT-Smol-2s-Llama-3.2-1B-IT_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_it_1b_tok-CT",

    #"ARMT-IT-QF-no-Lora-base-Q1-only",
    #"ARMT-IT-QF-no-Lora-IT-Q1-only",
    #"ARMT-IT-QF-no-Lora-base-Q1-only_bf16",
    #"ARMT-IT-QF-no-Lora-IT-Q1-only_bf16",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
#for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
for l0 in ['qa1_32k', 'qa2_32k', 'qa3_32k', 'qa4_32k', 'qa5_32k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no_mem/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/qa4_32k_instruction_no_examples_no_post_prompt_no_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no_mem/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin/qa5_0k_instruction_no_examples_no_post_prompt_no_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no

,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-200-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-400-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-800-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-1000-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4_no_mem-200-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-2kSynth_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-unmergedLora-ArmtLora-CompLoss-Llama-3.2-1B-IT-CT,ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_our_eval_100_IT-base_orig_segm_30-ainst_right-CT,Llama-3.2-1B-IT
qa1_0k,100.00,99.00,100.00,100.00,100.00,100.00,66.00,100.00,100.00,66.00,100.00,72.90
qa1_1k,100.00,100.00,100.00,100.00,100.00,100.00,53.00,100.00,100.00,53.00,100.00,66.80
qa1_2k,96.00,90.00,100.00,100.00,100.00,100.00,27.00,99.00,100.00,57.00,100.00,60.90
qa1_4k,87.00,74.00,100.00,100.00,100.00,100.00,8.00,100.00,100.00,44.00,100.00,58.30
qa1_8k,86.00,69.00,100.00,100.00,100.00,100.00,13.00,97.00,100.00,33.00,100.00,55.60
qa1_16k,84.00,78.00,99.00,99.00,99.00,99.00,12.00,76.00,98.00,6.00,100.00,45.60
qa1_32k,41.00,49.00,99.00,73.00,67.00,68.00,11.00,18.00,48.00,3.00,100.00,28.50
qa2_0k,99.00,98.00,100.00,100.00,100.00,100.00,48.00,99.00,100.00,30.00,100.00,45.15
qa2_1k,99.00,98.00,100.00,100.00,100.00,100.00,59.00,100.00,100.00,4.00,100.00,35.34
qa2_2k,95.00,90.00,100.00,100.00,100.00,100.00,41.00,100.00,100.00,20.00,100.00,27.33


### Evaluation debug, tune from BL model, qa6-10

In [446]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #  "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #  "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"DEBUG_eval_versions/orig_eval/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16_no_autocast/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_bf16_no_autocast_ainst/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #  "DEBUG_eval_versions/orig_eval_bf16_no_autocast_ainst_right/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    
    #"DEBUG_eval_versions/orig_eval_latest/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",

    #"DEBUG_eval_versions/orig_eval_base/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",

    #"DEBUG_eval_versions/our_eval_1k/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_30_gen/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_no_corr/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_corr/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_1k_seg/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_it_base/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_our_segm/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #"DEBUG_eval_versions/our_eval_100_it_base_our_segm_bf16/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_our_segm_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",
    #  "DEBUG_eval_versions/our_eval_100_it_base_orig_segm_30_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",

    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_after_dev_ev/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_our_load/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_orig_load_cpu/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_our_layers/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-500/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",

    #  "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    #  "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-400/pytorch_model.bin",
    #  "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-800/pytorch_model.bin",
    #  "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",
    #  # "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    #  "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_1e4_no_mem/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",

    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",
    
    #  "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/dolly_mix_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_cont/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2000/pytorch_model.bin",
    "DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_new/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",
    #"DEBUG_eval_versions/our_eval_fixed_ift_from_8s_bl/unsloth/Llama-3.2-1B-Instruct/runs/armt_ift_from_babilong/smol_lora_comp_bl_8s_v31_qa1_5_no_pack_with_lora_new_on_merge/smoltalk/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-2500/pytorch_model.bin",

    
    #  #"DEBUG_eval_versions/orig_eval_8s_best_bf16_no_autocast_ainst_right_correction/rmt-llama3.2-1b-/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5-ct-v3-align_left",
    "DEBUG_eval_versions/our_eval_100_it_base_orig_segm_30_ainst_right/unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-8x1024-mem16-lora-babilong-qa1-5_ct-v3.1/model.safetensors",

    #"DEBUG_eval_versions/our_eval_100_it_base_it_1b_tok/meta-llama/Llama-3.2-1B/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5_ct-v3/model.safetensors",


    #"DEBUG_eval_versions/armt_it_no_lora_base/meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_it/unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_base_bf16/meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",
    #"DEBUG_eval_versions/armt_it_no_lora_it_bf16/unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9600/pytorch_model.bin",

    
    #  "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #  "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    #  "Dolly-Llama-3.2-1B",

    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast_ainst-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_bf16_no_autocast_ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_latest-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_orig_eval_base-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_1k-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen_no_corr-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_30_gen_corr-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_1k_seg-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-bf16-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_our_segm-ainst_right-CT",
    #  "ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",

    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_dev_ev-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_our_load-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_orig_load_cpu-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_our_layers-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT-CT",

    #  "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-200-CompLoss-Llama-3.2-1B-IT-CT",
    #  "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-400-CompLoss-Llama-3.2-1B-IT-CT",
    #  "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-800-CompLoss-Llama-3.2-1B-IT-CT",
    #  "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora-1000-CompLoss-Llama-3.2-1B-IT-CT",
    #  # "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4-200-CompLoss-Llama-3.2-1B-IT-CT",
    #  "ARMT-16-64-8s-DollyMix05_from-BL-QA1-5_v3.1-mergedLora-Lora_1e4_no_mem-200-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    #  "ARMT-16-64-8s-2kSynth_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    #"ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-contLora-Lora-CompLoss-Llama-3.2-1B-IT-CT",
    "ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-unmergedLora-ArmtLora-CompLoss-Llama-3.2-1B-IT-CT",
    #"ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-ArmtLora-CompLoss-Llama-3.2-1B-IT-CT",


    
    #"ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_orig_eval_bf16_no_autocast_ainst_right_corr-CT",
    "ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",
    #"ARMT-Smol-2s-Llama-3.2-1B-IT_our_eval_100_IT-base_orig_segm_30-ainst_right-CT",
    #"ARMT-BL-Llama-3.2-1B-IT_v3_our_eval_100_IT-base_it_1b_tok-CT",

    #"ARMT-IT-QF-no-Lora-base-Q1-only",
    #"ARMT-IT-QF-no-Lora-IT-Q1-only",
    #"ARMT-IT-QF-no-Lora-base-Q1-only_bf16",
    #"ARMT-IT-QF-no-Lora-IT-Q1-only_bf16",

    #  "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa6_16k', 'qa7_16k', 'qa8_16k', 'qa9_16k', 'qa10_16k']:
    #for l0 in ['qa1_32k', 'qa2_32k', 'qa3_32k', 'qa4_32k', 'qa5_32k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-500-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-noLora-2500-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-mergedLora-Lora-CompLoss-Llama-3.2-1B-IT-CT,ARMT-16-64-8s-Smol_from-BL-QA1-5_v3.1-unmergedLora-ArmtLora-CompLoss-Llama-3.2-1B-IT-CT,ARMT-BL-Llama-3.2-1B-IT_8s_v3.1_our_eval_100_IT-base_orig_segm_30-ainst_right-CT
qa6_0k,77.00,84.00,2.00,68.00,0.00
qa6_1k,60.00,82.00,3.00,65.00,0.00
qa6_2k,54.00,72.00,1.00,60.00,0.00
qa6_4k,33.00,53.00,0.00,60.00,0.00
qa6_8k,30.00,46.00,0.00,36.00,0.00
qa6_16k,10.00,49.00,0.00,0.00,0.00
qa7_0k,4.00,2.00,0.00,30.00,0.00
qa7_1k,0.00,0.00,0.00,45.00,0.00
qa7_2k,0.00,0.00,0.00,20.00,0.00
qa7_4k,0.00,1.00,0.00,10.00,0.00


### Loss ablations, LLaMA 8b

In [21]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64/checkpoint-2000/pytorch_model.bin",

    
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-5000/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9200/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora_ext/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_100000/checkpoint-20000/pytorch_model.bin",


    # cot res
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",


    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",


    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-2400/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-2400/pytorch_model.bin",


    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1600/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/from_scratch_model_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1600/pytorch_model.bin",

    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1100/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1100/pytorch_model.bin",


    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_llama8b_mix_pajama_all_mix_1k_1920/meta-llama/Llama-3.1-8B-Instruct/from_scratch_model_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-3200/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_llama8b_mix_pajama_all_mix_1k_1920/meta-llama/Llama-3.1-8B-Instruct/from_scratch_model_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-3200/pytorch_model.bin",


    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot_sw/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/sink_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-2200/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot_sw/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/sink_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-2200/pytorch_model.bin",



    
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/unsloth/Llama-3.2-1B-Instruct/distill_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-5700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/unsloth/Llama-3.2-1B-Instruct/distill_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-5700/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/unsloth/Llama-3.2-1B-Instruct/distill_08_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-5100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/unsloth/Llama-3.2-1B-Instruct/distill_08_SEGM_2x1024_2016_64_1e-05_10000/checkpoint-5100/pytorch_model.bin",

    
    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-8100",
    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-8100",

    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",

    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-SmolMix05-Lora-CompLoss-Llama-3.1-8B-IT",
    
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-prelim-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",



    #cot res
    "ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    "ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT",


    "ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    "ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT",
    "ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-100",

    "ARMT-16-64-2s-CL-CoTSynth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-CoTSynth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1",


    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1",

    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1",

    "ARMT-16-64-2s-CL-v4-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    "ARMT-16-64-2s-CL-v4-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",



    "ARMT-16-64-2s-SW-1024-sink-v2-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    "ARMT-16-64-2s-SW-1024-sink-v2-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",


    "ARMT-16-64-2s-CL-v2-CompLoss-Llama-3.2-1B-dist1.0-IT-Q1",
    "ARMT-16-64-2s-CL-v2-CompLoss-Llama-3.2-1B-dist1.0-IT-Q1-only",


    "ARMT-16-64-2s-CL-v2-CompLoss-Llama-3.2-1B-dist0.8-IT-Q1",
    "ARMT-16-64-2s-CL-v2-CompLoss-Llama-3.2-1B-dist0.8-IT-Q1-only",


    
    
    "Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT",
    "Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",
    
    "Llama-3.2-8B-IT-Q1-only",
    "Llama-3.2-8B-IT",
    "Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    "Llama-3.2-1B-IT-Q1-only",
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "IT-Q1-100" in disp_name:
        prompt_name = '100_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task],# cot_answer="CoT" in disp_name
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin/qa4_16k_100_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin/qa5_0k_100_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin/qa5_1k_100_instruction_yes_examples_yes_post_prompt_

,Llama-3.2-1B-E,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-SmolMix05-Lora-CompLoss-Llama-3.1-8B-IT,ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1,ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-prelim-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1,ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT,ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1,ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT,ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-100,ARMT-16-64-2s-CL-CoTSynth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-CoTSynth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1,ARMT-16-64-2s-CL-v4-Lora-CompLoss-Llama-3.2-8B-IT-Q1,ARMT-16-64-2s-CL-v4-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-SW-1024-sink-v2-Lora-CompLoss-Llama-3.2-8B-IT-Q1,ARMT-16-64-2s-SW-1024-sink-v2-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-v2-CompLoss-Llama-3.2-1B-dist1.0-IT-Q1,ARMT-16-64-2s-CL-v2-CompLoss-Llama-3.2-1B-dist1.0-IT-Q1-only,ARMT-16-64-2s-CL-v2-CompLoss-Llama-3.2-1B-dist0.8-IT-Q1,ARMT-16-64-2s-CL-v2-CompLoss-Llama-3.2-1B-dist0.8-IT-Q1-only,Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT,Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only,Llama-3.2-8B-IT-Q1-only,Llama-3.2-8B-IT,Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only,Llama-3.2-1B-IT-Q1-only,Llama-3.2-1B-IT
qa1_0k,58.20,23.80,99.00,91.00,96.00,96.00,75.00,73.00,74.00,73.00,97.00,100.00,62.00,89.00,90.00,89.00,74.00,98.00,86.00,98.00,64.00,88.00,96.00,74.00,98.00,74.00,61.00,41.00,53.00,30.00,98.00,79.00,65.00,99.00,45.90,9.80,72.90
qa1_1k,71.90,41.40,96.00,60.00,88.00,92.00,51.00,54.00,48.00,45.00,94.00,95.00,48.00,83.00,91.00,83.00,51.00,94.00,60.00,94.00,52.00,81.00,95.00,57.00,95.00,31.00,50.00,0.00,58.00,0.00,94.00,44.00,24.00,94.00,51.60,0.00,66.80
qa1_2k,70.10,38.40,59.00,63.00,41.00,46.00,46.00,46.00,46.00,5.00,3.00,20.00,5.00,1.00,15.00,1.00,0.00,5.00,10.00,4.00,47.00,7.00,37.00,43.00,3.00,35.00,5.00,2.00,7.00,6.00,88.00,53.00,19.00,92.00,53.20,0.60,60.90
qa1_4k,62.40,18.00,38.00,38.00,16.00,24.00,30.00,33.00,31.00,2.00,1.00,2.00,1.00,0.00,2.00,0.00,1.00,6.00,18.00,7.00,19.00,4.00,16.00,20.00,1.00,0.00,0.00,0.00,0.00,0.00,89.00,51.00,12.00,91.00,38.40,0.60,58.30
qa1_8k,52.00,6.00,20.00,22.00,5.00,16.00,22.00,23.00,21.00,0.00,0.00,0.00,1.00,1.00,2.00,2.00,1.00,7.00,2.00,3.00,12.00,1.00,8.00,9.00,0.00,0.00,0.00,0.00,0.00,0.00,89.00,42.00,5.00,93.00,18.70,0.30,55.60
qa1_16k,47.20,2.60,6.00,11.00,2.00,4.00,9.00,11.00,12.00,0.00,0.00,1.00,2.00,0.00,0.00,0.00,0.00,1.00,0.00,1.00,6.00,0.00,3.00,4.00,0.00,0.00,0.00,0.00,1.00,3.00,77.00,26.00,3.00,82.00,31.70,0.40,45.60
qa2_0k,33.83,37.04,50.00,50.00,44.00,47.00,32.00,29.00,46.00,19.00,54.00,48.00,27.00,30.00,34.00,30.00,37.00,55.00,46.00,49.00,35.00,29.00,50.00,44.00,51.00,22.00,40.00,34.00,40.00,33.00,51.00,36.00,11.00,50.00,30.53,28.83,45.15
qa2_1k,32.23,16.12,49.00,17.00,44.00,46.00,13.00,13.00,29.00,38.00,43.00,50.00,18.00,17.00,32.00,18.00,38.00,48.00,37.00,49.00,18.00,26.00,54.00,32.00,47.00,39.00,23.00,14.00,36.00,11.00,51.00,26.0

### Loss ablations, LLaMA 8b, reduced res

In [24]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #"meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64/checkpoint-2000/pytorch_model.bin",

    
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-5000/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9200/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora_ext/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_100000/checkpoint-20000/pytorch_model.bin",


    # cot res
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",


    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",

    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1600/pytorch_model.bin",

    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1100/pytorch_model.bin",



    
    #"runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-8100",
    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-8100",

    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",
    "princeton-nlp/Llama-3-8B-ProLong-512k-Instruct",
    "nvidia/Llama-3.1-8B-UltraLong-1M-Instruct",

    #"runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    #"unsloth/Llama-3.2-1B-Instruct",
    #"unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #"Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-SmolMix05-Lora-CompLoss-Llama-3.1-8B-IT",
    
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-prelim-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",

    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT",


    "ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT",


    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",


    #"Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT",
    "Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",
    
    "Llama-3.1-8B-IT-Q1-only",
    "Llama-3.1-8B-IT",
    "Llama-3-8B-ProLong-512k-IT",
    "Llama-3-8B-UltraLong-1M-IT",
    #"Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    #"Llama-3.2-1B-IT-Q1-only",
    #"Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-16-64-2s-CL-SmolMix05-Lora-CompLoss-Llama-3.1-8B-IT,ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only,Llama-3.1-8B-IT-Q1-only,Llama-3.1-8B-IT,Llama-3-8B-ProLong-512k-IT,Llama-3-8B-UltraLong-1M-IT
qa1_0k,99.00,91.00,73.00,74.00,62.00,86.00,64.00,79.00,65.00,99.00,96.00,99.00
qa1_1k,96.00,60.00,54.00,48.00,48.00,60.00,52.00,44.00,24.00,94.00,91.00,96.00
qa1_2k,59.00,63.00,46.00,46.00,5.00,10.00,47.00,53.00,19.00,92.00,84.00,87.00
qa1_4k,38.00,38.00,33.00,31.00,1.00,18.00,19.00,51.00,12.00,91.00,83.00,87.00
qa1_8k,20.00,22.00,23.00,21.00,1.00,2.00,12.00,42.00,5.00,93.00,72.00,82.00
qa1_16k,6.00,11.00,11.00,12.00,2.00,0.00,6.00,26.00,3.00,82.00,53.00,74.00
qa2_0k,50.00,50.00,29.00,46.00,27.00,46.00,35.00,36.00,11.00,50.00,45.00,49.00
qa2_1k,49.00,17.00,13.00,29.00,18.00,37.00,18.00,26.00,22.00,49.00,43.00,55.00
qa2_2k,28.00,3.00,12.00,12.00,0.00,2.00,16.00,23.00,13.00,46.00,50.00,51.00
qa2_4k,17.00,3.00,8.00,9.00,0.00,1.00,3.00,15.00,10.00,50.00,37.00,42.00


### Loss ablations, LLaMA 8b, reduced res for pres

In [520]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #"meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64/checkpoint-2000/pytorch_model.bin",

    
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-5000/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9200/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora_ext/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_100000/checkpoint-20000/pytorch_model.bin",


    # cot res
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",


    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",

    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1600/pytorch_model.bin",

    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1100/pytorch_model.bin",

    
    #"runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-8100",
    #"runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-8100",

    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",

    #"runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    #"unsloth/Llama-3.2-1B-Instruct",
    #"unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #"Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-SmolMix05-Lora-CompLoss-Llama-3.1-8B-IT",
    
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-prelim-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",

    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT",


    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT",


    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",

    #"Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT",
    #"Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",
    
    "Llama-3.2-8B-IT-Q1-only",
    "Llama-3.2-8B-IT",
    #"Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    #"Llama-3.2-1B-IT-Q1-only",
    #"Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,Llama-3.2-8B-IT-Q1-only,Llama-3.2-8B-IT
qa1_0k,86.00,64.00,65.00,99.00
qa1_1k,60.00,52.00,24.00,94.00
qa1_2k,10.00,47.00,19.00,92.00
qa1_4k,18.00,19.00,12.00,91.00
qa1_8k,2.00,12.00,5.00,93.00
qa1_16k,0.00,6.00,3.00,82.00
qa2_0k,46.00,35.00,11.00,50.00
qa2_1k,37.00,18.00,22.00,49.00
qa2_2k,2.00,16.00,13.00,46.00
qa2_4k,1.00,3.00,10.00,50.00


### Loss ablations, LLaMA 8b, reduced res, 4s

In [516]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #"meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64/checkpoint-2000/pytorch_model.bin",

    
    #  "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-5000/pytorch_model.bin",
    #  "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-9200/pytorch_model.bin",
    #  "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_lora_ext/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_100000/checkpoint-20000/pytorch_model.bin",


    # cot res
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_2k/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_2000/checkpoint-600/pytorch_model.bin",


    #  "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",
    #"meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_cot_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x992_1952_64_1e-05_10000/checkpoint-1400/pytorch_model.bin",

    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1600/pytorch_model.bin",

    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-1100/pytorch_model.bin",


    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_2k_4k_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_4x1024_4032_64_1e-05_10000/checkpoint-1800/pytorch_model.bin",
    "meta-llama/Llama-3.1-8B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_4k_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_4x1024_4032_64_1e-05_10000/checkpoint-2400/pytorch_model.bin",

    #"runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-8100",
    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.1-8B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-8100",

    "meta-llama/Llama-3.1-8B-Instruct",
    "meta-llama/Llama-3.1-8B-Instruct",

    #"runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    #"unsloth/Llama-3.2-1B-Instruct",
    #"unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #"Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-SmolMix05-Lora-CompLoss-Llama-3.1-8B-IT",
    
    #  "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-8B-IT",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-prelim-Q1-only",
    #  "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #  "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-100k-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",

    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-small-Lora-CompLoss-Llama-3.2-8B-IT",


    #  "ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1",
    #"ARMT-16-64-2s-CL-CoTSynth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT",


    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",

    "ARMT-16-64-4s-CL-QA-Inst-Synth-2k-4k-big-from-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",
    "ARMT-16-64-4s-CL-QA-Inst-Synth-1k-4k-big-from-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only",

    #"Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT",
    "Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",
    
    "Llama-3.2-8B-IT-Q1-only",
    "Llama-3.2-8B-IT",
    #"Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    #"Llama-3.2-1B-IT-Q1-only",
    #"Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-16-64-2s-CL-SmolMix05-Lora-CompLoss-Llama-3.1-8B-IT,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-2s-CL-QA-Inst-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-4s-CL-QA-Inst-Synth-2k-4k-big-from-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,ARMT-16-64-4s-CL-QA-Inst-Synth-1k-4k-big-from-Synth1k-2k-big-from-Synth-Lora-CompLoss-Llama-3.2-8B-IT-Q1-only,Llama-3.2-8B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only,Llama-3.2-8B-IT-Q1-only,Llama-3.2-8B-IT
qa1_0k,99.00,86.00,64.00,59.00,60.00,79.00,65.00,99.00
qa1_1k,96.00,60.00,52.00,54.00,53.00,44.00,24.00,94.00
qa1_2k,59.00,10.00,47.00,47.00,47.00,53.00,19.00,92.00
qa1_4k,38.00,18.00,19.00,23.00,30.00,51.00,12.00,91.00
qa1_8k,20.00,2.00,12.00,14.00,16.00,42.00,5.00,93.00
qa1_16k,6.00,0.00,6.00,8.00,6.00,26.00,3.00,82.00
qa2_0k,50.00,46.00,35.00,34.00,32.00,36.00,11.00,50.00
qa2_1k,49.00,37.00,18.00,15.00,16.00,26.00,22.00,49.00
qa2_2k,28.00,2.00,16.00,9.00,13.00,23.00,13.00,46.00
qa2_4k,17.00,1.00,3.00,4.00,4.00,15.00,10.00,50.00


### Optimized ARMT

In [26]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../optimized_armt/grouped_batching/test_res'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/executor_armt-1b-it-v2",
    "unsloth/Llama-3.2-1B-Instruct/armt-1b-it-v2",
]
disp_names = [
    "Optimized-ARMT-1b-IT",
    "Original-ARMT-1b-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1']#, 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['2k', '4k', '8k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Optimized-ARMT-1b-IT,Original-ARMT-1b-IT
qa1_2k,3.00,1.00
qa1_4k,3.00,0.00
qa1_8k,1.00,0.00


### Eval 1b model, FT on BL

In [85]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v3_all_lin_64_128_sink_SEGM_16x512_8192_64_1e-05_10000/checkpoint-7900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v3_all_lin_64_128_sink_SEGM_8x512_4096_64_1e-05_10000/checkpoint-8500/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v3_all_lin_64_128_sink_SEGM_8x512_4096_64_1e-05_10000/checkpoint-8500/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v3_all_lin_64_128_sink_SEGM_8x512_4096_64_1e-05_10000/checkpoint-8500/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v6_wm_1gpu_sw_q15_all_lin_64_128_sink_SEGM_8x512_4096_64_1e-04_10000/checkpoint-2200/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-600/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-500/pytorch_model.bin",
]
mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-600/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_8x512_4096_64_1e-04_10000/checkpoint-1200/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-500/pytorch_model.bin",   
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-2000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-2300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1800/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1600/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-2300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-3200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-4600/pytorch_model.bin",


    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-2000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-2900/pytorch_model.bin",

]
"""
mod_names = [
    "openai-community/gpt2/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/openai-community/gpt2/gpt2_v3_no_lora_all_lin_64_128_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-3100/pytorch_model.bin",
    "openai-community/gpt2/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/openai-community/gpt2/gpt2_v3_no_lora_all_lin_64_128_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-2800/pytorch_model.bin",
    "openai-community/gpt2/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/openai-community/gpt2/gpt2_v3_no_lora_all_lin_64_128_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-3900/pytorch_model.bin",
    "openai-community/gpt2/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/openai-community/gpt2/gpt2_v3_no_lora_all_lin_64_128_sink_SEGM_8x512_4096_64_1e-04_10000/checkpoint-2300/pytorch_model.bin",

]
"""
disp_names = [
    "Llama-3.2-1B-IT-CT",

    "ARMT-SW-64-128-all-lin-16s-512-CL-QA1-5-Llama-3.2-1B-IT-CT",
    "ARMT-SW-64-128-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT-CT",

    "ARMT-SW-64-128-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT",

    "ARMT-SW-64-128-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT-legacy",

    #"v6-left-pad-ARMT-SW-64-128-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT-CT",

    "v7-left-pad-ARMT-no-sw-16-64-all-lin-2s-512-CL-QA1-5-Llama-3.2-1B-IT-CT",
    "v7-left-pad-ARMT-no-sw-16-64-all-lin-3s-512-CL-QA1-5-Llama-3.2-1B-IT-CT",
    "v7-left-pad-ARMT-no-sw-16-64-all-lin-5s-512-CL-QA1-5-Llama-3.2-1B-IT-CT",
]
disp_names = [
    "ARMT-16-64-all-lin-2s-512-CL-QA1-5-Llama-3.2-1B-IT-no_tag",
    "ARMT-16-64-all-lin-3s-512-CL-QA1-5-Llama-3.2-1B-IT-no_tag",
    "ARMT-16-64-all-lin-5s-512-CL-QA1-5-Llama-3.2-1B-IT-no_tag (prelim cpt)",
    "ARMT-16-64-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT-no_tag (prelim cpt)",

    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-500)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-700)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-1000)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-1300)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-1700)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-2000)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-2300)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (final cpt-1800)",

    
    
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-300)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-500)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-700)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-1000)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-1400)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-1600)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-1900)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-2300)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-3200)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-4600)",
    

    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-200)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-700)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-1200)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-1400)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-2000)",
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (final cpt-2900)",
    
]
"""
disp_names = [
    "ARMT-2s-512-CL-QA1-GPT-2-legacy",
    "ARMT-3s-512-CL-QA1-GPT-2-legacy",
    "ARMT-5s-512-CL-QA1-GPT-2-legacy",
    "ARMT-8s-512-CL-QA1-GPT-2-legacy (prelim cpt)",
]
"""
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1']#, 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals_bl_ft/./unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-4600/pytorch_model.bin/qa1_8k_instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode.csv
No such file: ./../../rmt_ift/babilong_evals_bl_ft/./unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-4600/pytorch_model.bin/qa1_16k_instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode.csv


,ARMT-16-64-all-lin-2s-512-CL-QA1-5-Llama-3.2-1B-IT-no_tag,ARMT-16-64-all-lin-3s-512-CL-QA1-5-Llama-3.2-1B-IT-no_tag,ARMT-16-64-all-lin-5s-512-CL-QA1-5-Llama-3.2-1B-IT-no_tag (prelim cpt),ARMT-16-64-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT-no_tag (prelim cpt),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-500),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-700),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-1000),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-1300),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-1700),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-2000),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (prelim cpt-2300),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (final cpt-1800),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-300),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-500),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-700),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-1000),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-1400),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-1600),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-1900),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-2300),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-3200),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-4600),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-200),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-700),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-1200),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-1400),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-2000),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (final cpt-2900)
qa1_0k,2.00,37.00,3.00,11.00,22.00,19.00,4.00,4.00,4.00,1.00,1.00,4.00,75.00,92.00,99.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,20.00,14.00,14.00,23.00,24.00,38.00
qa1_1k,0.00,0.00,0.00,1.00,31.00,34.00,6.00,7.00,7.00,22.00,22.00,7.00,37.00,50.00,50.00,62.00,51.00,62.00,68.00,70.00,78.00,78.00,25.00,29.00,28.00,38.00,37.00,44.00
qa1_2k,0.00,0.00,0.00,0.00,16.00,16.00,0.00,0.00,0.00,5.00,5.00,0.00,28.00,30.00,40.00,53.00,48.00,54.00,63.00,74.00,77.00,78.00,18.00,11.00,10.00,17.00,18.00,25.00
qa1_4k,0.00,0.00,0.00,0.00,13.00,10.00,2.00,2.00,2.00,2.00,2.00,2.00,22.00,24.00,30.00,47.00,31.00,35.00,55.00,69.00,73.00,65.06,13.00,12.00,10.00,15.00,15.00,17.00
qa1_8k,0.00,0.00,0.00,0.00,13.00,7.00,1.00,1.00,1.00,1.00,1.00,1.00,24.00,23.00,20.00,44.00,25.00,28.00,25.00,31.00,28.00,nan,10.00,7.00,5.00,12.00,11.00,13.00
qa1_16k,0.00,0.00,0.00,0.00,8.00,11.00,1.00,1.00,1.00,1.00,1.00,1.00,20.00,19.00,15.00,37.00,14.00,19.00,12.00,12.00,19.00,nan,5.00,7.00,5.00,10.00,11.00,10.00


In [56]:
a = np.array([0, 0, 0, 0, 0,0,0,0,0,0,0,0,0,0])
tl = 2
tt = 1
ts = 3
a[-tl-tt:-tt] = 1
a

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0])

In [79]:
def mask_non_completion_multi(input, response_template, end_response_token_ids, tokenizer, ignore_index=-100):
    # adapted from huggingface DataCollatorForCompletionOnlyLM https://github.com/huggingface/trl/blob/main/trl/trainer/utils.py#L87
    if isinstance(response_template, str):
        # The user provides a string, must tokenize
        response_token_ids = tokenizer.encode(response_template, add_special_tokens=False)
    else:
        # The user already provides the token ids
        response_token_ids = response_template
    response_token_ids_start_idx = None
    response_token_ids_start_ids = []
    output = np.array(input.copy())
    end_response_token_start_ids = np.where(output == end_response_token_ids)[0]
    for idx in np.where(output == response_token_ids[0])[0]:
        # `response_token_ids` is `'### Response:\n'`, here we are just making sure that the token IDs match
        if (
            response_token_ids
            == output[idx : idx + len(response_token_ids)].tolist()
        ):
            response_token_ids_start_idx = idx
            response_token_ids_start_ids.append(response_token_ids_start_idx)
            #print(idx)

    #end_response_token_start_ids = np.where(output == end_response_token_ids)[0]
    #print(end_response_token_start_ids)
    if response_token_ids_start_idx is None:
        warnings.warn(
            f"Could not find response key `{response_template}` in the "
            f'following instance: {tokenizer.decode(output)} '
            f"This instance will be ignored in loss calculation. "
            f"Note, if this happens often, consider increasing the `max_seq_length`."
        )
        output[:] = ignore_index
    else:
        if len(response_token_ids_start_ids) == 1:
            # case of one reply from assistant
            response_token_ids_end_idx = response_token_ids_start_idx# + len(response_token_ids)
            output[:response_token_ids_end_idx] = ignore_index
        else:
            # case of multiple step conversation
            response_token_ids_start_ids = [el for el in response_token_ids_start_ids]
            response_token_ids_end_ids = []
            for el in response_token_ids_start_ids:
                buf_end_toks = end_response_token_start_ids - el
                closest_end_idx = end_response_token_start_ids[np.where(buf_end_toks > 0)[0][np.argmin((buf_end_toks)[np.where(buf_end_toks > 0)])]]
                #print(buf_end_toks)
                response_token_ids_end_ids.append(closest_end_idx)
            #print(response_token_ids_start_ids)
            #print(response_token_ids_end_ids)
            #response_token_ids_start_ids = [el + len(response_token_ids) for el in response_token_ids_start_ids]
            mask = np.zeros_like(output)
            for response_idx in range(len(response_token_ids_start_ids)):
                start_idx = response_token_ids_start_ids[response_idx]
                try:
                    end_idx = response_token_ids_end_ids[response_idx]
                except:
                    end_idx = -1
                mask[start_idx:end_idx] = 1
            output[np.where(mask == 0)[0]] = ignore_index

        # Make pytorch loss function ignore all tokens up through the end of the response key
        #output[:response_token_ids_end_idx] = ignore_index
    return output.tolist()

In [80]:
response_template = "<|start_header_id|>assistant<|end_header_id|>"
response_template_end_idx = 128009
tokenizer = AutoTokenizer.from_pretrained("unsloth/Llama-3.2-1B-Instruct")
messages = [
    {"role": "user", "content": f"Context:\n\nQuestion:\n\n"},
    {"role": "assistant", "content": f"Answer:\n\n"},
]
text = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=False
)

In [81]:
out = mask_non_completion_multi(text, response_template, response_template_end_idx, tokenizer)

In [82]:
tokenizer(response_template, add_special_tokens=False)

{'input_ids': [128006, 78191, 128007], 'attention_mask': [1, 1, 1]}

In [83]:
text

[128000,
 128006,
 9125,
 128007,
 271,
 38766,
 1303,
 33025,
 2696,
 25,
 6790,
 220,
 2366,
 18,
 198,
 15724,
 2696,
 25,
 220,
 1544,
 5033,
 220,
 2366,
 20,
 271,
 128009,
 128006,
 882,
 128007,
 271,
 2014,
 1473,
 14924,
 25,
 128009,
 128006,
 78191,
 128007,
 271,
 16533,
 25,
 128009]

In [84]:
out

[-100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 -100,
 128006,
 78191,
 128007,
 271,
 16533,
 25,
 128009]

### Eval 1b model, FT on BL - masks ablation

In [148]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-1800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-300/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-4600/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-600/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_8x512_4096_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_12x512_6144_64_1e-04_10000/checkpoint-200/pytorch_model.bin",


    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-2900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-2300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-2600/pytorch_model.bin",


    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_shift_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-2600/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_shift_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_shift_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-600/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_shift_mask_no_sw_1gpu_sw_q1_all_lin_16_32_sink_SEGM_8x512_4096_64_1e-04_10000/checkpoint-400/pytorch_model.bin",

]
disp_names = [
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (final cpt-1800)",
    "ARMT-16-64-all-lin-3s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (final cpt-300)",
    "ARMT-16-64-all-lin-5s-512-CL-QA1-Llama-3.2-1B-IT-no_tag (final cpt-300)",

    
    
    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-4600)",
    #"ARMT-16-64-all-lin-3s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-900)",
    "ARMT-16-64-all-lin-3s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-700)",
    #"ARMT-16-64-all-lin-5s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-600)",
    "ARMT-16-64-all-lin-5s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-200)",
    "ARMT-16-64-all-lin-8s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-200)",
    "ARMT-16-64-all-lin-12s-512-CL-QA1-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-200)",
    

    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (final cpt-2900)",
    "ARMT-16-64-all-lin-3s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (final cpt-100)",
    #"ARMT-16-64-all-lin-5s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (prelim cpt-2300)",
    "ARMT-16-64-all-lin-5s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (final cpt-2600)",


    "ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-shift-no_tag (final cpt-2600)",
    "ARMT-16-64-all-lin-3s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-shift-no_tag (final cpt-500)",
    "ARMT-16-64-all-lin-5s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-shift-no_tag (final cpt-600)",
    "ARMT-16-64-all-lin-8s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-shift-no_tag (final cpt-400)",
    
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1']#, 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
# results with assistant mask
#full_df = full_df[full_df.columns[3:8]]
# res for all mask with fixed target shift mask
#full_df = full_df[list(full_df.columns[:7]) + list(full_df.columns[11:])]
# compare target mask w and w/o shift
full_df = full_df[list(full_df.columns[8:])]


s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (final cpt-2900),ARMT-16-64-all-lin-3s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (final cpt-100),ARMT-16-64-all-lin-5s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-no_tag (final cpt-2600),ARMT-16-64-all-lin-2s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-shift-no_tag (final cpt-2600),ARMT-16-64-all-lin-3s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-shift-no_tag (final cpt-500),ARMT-16-64-all-lin-5s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-shift-no_tag (final cpt-600),ARMT-16-64-all-lin-8s-512-CL-QA1-Llama-3.2-1B-IT-targ-mask-shift-no_tag (final cpt-400)
qa1_0k,38.00,38.00,48.00,99.00,100.00,100.00,100.00
qa1_1k,44.00,40.00,18.00,34.00,41.00,47.00,53.00
qa1_2k,25.00,19.00,5.00,38.00,59.00,64.00,73.00
qa1_4k,17.00,3.00,2.00,27.00,60.00,69.00,82.00
qa1_8k,13.00,4.00,1.00,10.00,12.00,14.00,82.00
qa1_16k,10.00,1.00,1.00,6.00,8.00,5.00,76.00


### Eval 1b model, FT on BL - QA1-5 + bigger lora

In [186]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_all_lin_64_128_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-4200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_all_lin_64_128_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-8000/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_all_lin_64_128_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_all_lin_64_128_sink_SEGM_8x512_4096_64_1e-04_10000/checkpoint-7200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_all_lin_64_128_sink_SEGM_12x512_6144_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_all_lin_64_128_sink_SEGM_16x512_8192_64_1e-04_10000/checkpoint-200/pytorch_model.bin",


    # prolonged validation
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_all_lin_64_128_sink_SEGM_12x512_6144_test_32_64_1e-04_10000/checkpoint-1500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_all_lin_64_128_sink_SEGM_12x512_6144_test_32_64_1e-04_10000/checkpoint-2500/pytorch_model.bin",

]
disp_names = [
    "ARMT-64-128-all-lin-2s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-4200)",
    "ARMT-64-128-all-lin-3s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-8000)",

    "ARMT-64-128-all-lin-5s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-200)",
    "ARMT-64-128-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-7200)",
    "ARMT-64-128-all-lin-12s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-100)",
    "ARMT-64-128-all-lin-16s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-200)",

    # prolonged validation
    "ARMT-64-128-all-lin-12s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag+val_16k (prelim cpt-1500)",
    "ARMT-64-128-all-lin-12s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag+val_16k (prelim cpt-2500)",
   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
full_df = full_df[full_df.columns[[4,6,7]]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-64-128-all-lin-12s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-100),ARMT-64-128-all-lin-12s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag+val_16k (prelim cpt-1500),ARMT-64-128-all-lin-12s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag+val_16k (prelim cpt-2500)
qa1_0k,100.00,100.00,100.00
qa1_1k,99.00,99.00,99.00
qa1_2k,100.00,100.00,100.00
qa1_4k,100.00,100.00,100.00
qa1_8k,100.00,100.00,100.00
qa1_16k,100.00,100.00,100.00
qa1_32k,71.00,98.00,99.00
qa1_64k,36.00,86.00,97.00
qa2_0k,100.00,100.00,100.00
qa2_1k,100.00,99.00,100.00


### Eval 1b model, FT on BL - QA1-5 + smaller lora + 1024 segment

In [183]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_base_16_32_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-8400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_base_16_32_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-8000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_base_16_32_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_base_16_32_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-8400/pytorch_model.bin",
    
]
disp_names = [
    "ARMT-16-32-base-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-8400)",
    "ARMT-16-32-base-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-8000)",
    "ARMT-16-32-base-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-200)",
    "ARMT-16-32-base-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-8400)",

    
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
full_df = full_df[full_df.columns[:2]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-16-32-base-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-8400),ARMT-16-32-base-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-8000)
qa1_0k,100.00,100.00
qa1_1k,100.00,100.00
qa1_2k,94.00,98.00
qa1_4k,85.00,100.00
qa1_8k,65.00,100.00
qa1_16k,49.00,91.00
qa1_32k,15.00,4.00
qa1_64k,5.00,0.00
qa2_0k,96.00,98.00
qa2_1k,96.00,100.00


### Eval 1b model, FT on BL - QA1-5 + smaller lora + 512 segment

In [169]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_base_16_32_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-8000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_base_16_32_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-5500/pytorch_model.bin",
    
]
disp_names = [
    "ARMT-16-32-all-lin-2s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-8000)",
    "ARMT-16-32-all-lin-3s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-5500)",

    
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-16-32-all-lin-2s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-8000),ARMT-16-32-all-lin-3s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (prelim cpt-5500)
qa1_0k,100.00,100.00
qa1_1k,97.00,99.00
qa1_2k,97.00,99.00
qa1_4k,76.00,100.00
qa1_8k,29.00,47.00
qa1_16k,10.00,5.00
qa1_32k,4.00,4.00
qa1_64k,5.00,2.00
qa2_0k,99.00,99.00
qa2_1k,84.00,93.00


### Eval 1b model, FT on BL - QA1-5 + no lora + 512 segment

In [168]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_no_lora_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-9800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_no_lora_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-7700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_no_lora_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-400/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_no_lora_sink_SEGM_8x512_4096_64_1e-04_10000/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_no_lora_sink_SEGM_12x512_6144_64_1e-04_10000/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_sw_q15_no_lora_sink_SEGM_16x512_8192_64_1e-04_10000/checkpoint-6900/pytorch_model.bin",
    
]
disp_names = [
    "ARMT-no-lora-2s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-9800)",
    "ARMT-no-lora-3s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-7700)",
    "ARMT-no-lora-5s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-400)",

    "ARMT-no-lora-8s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-400)",
    "ARMT-no-lora-12s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-400)",
    "ARMT-no-lora-16s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-6900)",

    
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-no-lora-2s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-9800),ARMT-no-lora-3s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-7700),ARMT-no-lora-5s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-400),ARMT-no-lora-8s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-400),ARMT-no-lora-12s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-400),ARMT-no-lora-16s-512-CL-QA1-5-Llama-3.2-1B-IT-assit-mask-no_tag (final cpt-6900)
qa1_0k,100.00,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,99.00,98.00,96.00
qa1_2k,100.00,100.00,100.00,100.00,100.00,96.00
qa1_4k,99.00,99.00,100.00,100.00,99.00,96.00
qa1_8k,75.00,96.00,100.00,99.00,100.00,99.00
qa1_16k,37.00,73.00,86.00,100.00,100.00,100.00
qa1_32k,5.00,34.00,55.00,90.00,92.00,98.00
qa1_64k,3.00,6.00,11.00,52.00,83.00,96.00
qa2_0k,98.00,98.00,98.00,98.00,94.00,98.00
qa2_1k,99.00,99.00,98.00,97.00,99.00,97.00


### Eval 1b model, FT on BL - QA1-5 + target mask

In [167]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_shift_no_sw_1gpu_sw_q15_all_lin_64_128_sink_SEGM_2x512_1024_64_1e-04_10000/checkpoint-6900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_shift_no_sw_1gpu_sw_q15_all_lin_64_128_sink_SEGM_3x512_1536_64_1e-04_10000/checkpoint-300/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_shift_no_sw_1gpu_sw_q15_all_lin_64_128_sink_SEGM_5x512_2560_64_1e-04_10000/checkpoint-6200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_targ_mask_shift_no_sw_1gpu_sw_q15_all_lin_64_128_sink_SEGM_8x512_4096_64_1e-04_10000/checkpoint-300/pytorch_model.bin",
    
]
disp_names = [
    "ARMT-64-128-all-lin-2s-512-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6900)",
    "ARMT-64-128-all-lin-3s-512-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-300)",

    "ARMT-64-128-all-lin-5s-512-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6200)",
    "ARMT-64-128-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (prelim cpt-300)",
   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-64-128-all-lin-2s-512-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6900),ARMT-64-128-all-lin-3s-512-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-300),ARMT-64-128-all-lin-5s-512-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6200),ARMT-64-128-all-lin-8s-512-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (prelim cpt-300)
qa1_0k,85.00,82.00,89.00,89.00
qa1_1k,70.00,66.00,74.00,82.00
qa1_2k,49.00,45.00,75.00,74.00
qa1_4k,20.00,24.00,77.00,73.00
qa1_8k,21.00,10.00,67.00,58.00
qa1_16k,36.00,18.00,49.00,38.00
qa1_32k,12.00,15.00,8.00,35.00
qa1_64k,9.00,13.00,4.00,7.00
qa2_0k,89.00,84.00,74.00,80.00
qa2_1k,67.00,60.00,77.00,79.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, base lora 64-128

In [208]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_base_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-9200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_base_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-5300/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_base_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-2600/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_base_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-9300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_base_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-4400/pytorch_model.bin",

]
disp_names = [
    "ARMT-64-128-base-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-9200)",
    "ARMT-64-128-base-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-3500)",

    "ARMT-64-128-base-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-3600)",
    "ARMT-64-128-base-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-9300)",
    "ARMT-64-128-base-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-4400)",
   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-64-128-base-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-9200),ARMT-64-128-base-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-3500),ARMT-64-128-base-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-3600),ARMT-64-128-base-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-9300),ARMT-64-128-base-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-4400)
qa1_0k,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,100.00,100.00
qa1_2k,95.00,100.00,100.00,100.00,100.00
qa1_4k,89.00,100.00,100.00,100.00,99.00
qa1_8k,66.00,99.00,100.00,99.00,91.00
qa1_16k,44.00,95.00,91.00,94.00,77.00
qa1_32k,25.00,59.00,53.00,77.00,71.00
qa1_64k,4.00,1.00,15.00,64.00,66.00
qa2_0k,99.00,100.00,99.00,100.00,100.00
qa2_1k,99.00,100.00,100.00,100.00,100.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, all-lin lora 64-128

In [76]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-7900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-6100/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

]
disp_names = [
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-7900)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6100)",

    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-5500)",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
    "ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-7900),ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6100),ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-5500),ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100),ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)
qa1_0k,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,100.00,100.00
qa1_2k,100.00,100.00,100.00,100.00,100.00
qa1_4k,98.00,100.00,100.00,100.00,100.00
qa1_8k,53.00,100.00,100.00,100.00,100.00
qa1_16k,28.00,96.00,98.00,98.00,98.00
qa1_32k,15.00,81.00,97.00,97.00,97.00
qa1_64k,2.00,72.00,94.00,94.00,94.00
qa2_0k,100.00,99.00,100.00,100.00,100.00
qa2_1k,100.00,100.00,100.00,100.00,100.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, all-lin lora 64-128 ablation

In [67]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

]
disp_names = [
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-mlp",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-self_attn",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-q_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-v_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-o_proj",

    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-gate_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-up_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-down_proj",
    #"ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag_no-" in disp_name:
        print(disp_name.split("-")[-1])
        prompt_name = f'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode_lora_ablate_{disp_name.split("-")[-1]}'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
full_df.loc["summ"] = full_df.mean(axis=0)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


mlp
self_attn
q_proj
k_proj
v_proj
o_proj
gate_proj
up_proj
down_proj


,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100),ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-mlp,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-self_attn,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-q_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-v_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-o_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-gate_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-up_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-down_proj
qa1_0k,100.00,96.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,98.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_2k,100.00,30.00,95.00,99.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_4k,100.00,35.00,99.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_8k,100.00,20.00,92.00,97.00,100.00,100.00,100.00,100.00,100.00,99.00
qa1_16k,98.00,10.00,91.00,97.00,98.00,98.00,99.00,98.00,98.00,95.00
qa1_32k,97.00,5.00,59.00,92.00,97.00,96.00,95.00,93.00,93.00,85.00
qa1_64k,94.00,4.00,1.00,74.00,93.00,74.00,80.00,77.00,79.00,70.00
qa2_0k,100.00,70.00,100.00,100.00,100.00,100.00,99.00,100.00,100.00,100.00
qa2_1k,100.00,68.00,98.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, sequential all-lin lora 64-128 ablation

In [66]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

]
disp_names = [
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj;v_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj;v_proj;o_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj;v_proj;o_proj;gate_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj;v_proj;o_proj;gate_proj;q_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj;v_proj;o_proj;gate_proj;q_proj;up_proj",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj;v_proj;o_proj;gate_proj;q_proj;up_proj;down_proj",

   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag_no-" in disp_name:
        print(disp_name.split("-")[-1])
        prompt_name = f'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode_lora_ablate_{disp_name.split("-")[-1]}'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
cols = full_df.columns
cols = [el.replace(";", "-") for el in cols]
full_df.columns = cols



full_df.loc["avg"] = full_df.mean(axis=0)
print(full_df.mean(axis=0))

weights = [116457488, 113836048, 111214608, 107020304, 96534544, 92340240, 81854480, 71368720]
full_df.loc["params, %"] = np.round(np.array(weights) / max(weights) * 100, 2)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


k_proj
k_proj;v_proj
k_proj;v_proj;o_proj
k_proj;v_proj;o_proj;gate_proj
k_proj;v_proj;o_proj;gate_proj;q_proj
k_proj;v_proj;o_proj;gate_proj;q_proj;up_proj
k_proj;v_proj;o_proj;gate_proj;q_proj;up_proj;down_proj
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)                                               97.975
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj                                                     97.550
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj                                              96.200
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj-o_proj                                       86.300
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj-o_proj-gate_proj                             79.450
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-

,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100),ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj-o_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj-o_proj-gate_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj-o_proj-gate_proj-q_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj-o_proj-gate_proj-q_proj-up_proj,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-k_proj-v_proj-o_proj-gate_proj-q_proj-up_proj-down_proj
qa1_0k,100.00,100.00,100.00,100.00,100.00,98.00,52.00,8.00
qa1_1k,100.00,100.00,100.00,100.00,100.00,99.00,60.00,1.00
qa1_2k,100.00,100.00,100.00,100.00,98.00,92.00,30.00,14.00
qa1_4k,100.00,100.00,100.00,100.00,98.00,83.00,33.00,19.00
qa1_8k,100.00,100.00,100.00,99.00,96.00,69.00,17.00,21.00
qa1_16k,98.00,98.00,98.00,99.00,83.00,59.00,27.00,20.00
qa1_32k,97.00,97.00,96.00,76.00,63.00,36.00,5.00,16.00
qa1_64k,94.00,93.00,73.00,10.00,8.00,0.00,0.00,1.00
qa2_0k,100.00,100.00,100.00,99.00,96.00,97.00,38.00,0.00
qa2_1k,100.00,100.00,100.00,100.00,96.00,88.00,16.00,4.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, associative layers ablation

In [40]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    
]
disp_names = [
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-0;1;2;3",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-4;5;6;7",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-8;9;10;11",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-12;13;14;15",
    
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-0",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-1",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-2",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-3",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-4",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-5",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-6",

    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-7",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-8",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-9",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-10",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-11",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-12",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-13",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-14",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-15",
   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag_no-" in disp_name:
        print(disp_name.split("-")[-1])
        prompt_name = f'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode_armt_ablate_{disp_name.split("-")[-1]}'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
cols = full_df.columns
cols = [el.replace(";", "-") for el in cols]
full_df.columns = cols



full_df.loc["avg"] = full_df.mean(axis=0)
print(full_df.mean(axis=0))

#weights = [116457488, 113836048, 111214608, 107020304, 96534544, 92340240, 81854480, 71368720]
#full_df.loc["params, %"] = np.round(np.array(weights) / max(weights) * 100, 2)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:5]]
full_df = full_df[[full_df.columns[0]] + list(full_df.columns[5:])]

s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


0;1;2;3
4;5;6;7
8;9;10;11
12;13;14;15
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)    97.975
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-0-1-2-3         97.125
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-4-5-6-7         97.950
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-8-9-10-11       95.650
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-12-13-14-15     56.025
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-0               88.875
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-1               98.000
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-2               98.000
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-3               98.125
ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama

,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100),ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-0,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-1,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-2,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-3,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-4,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-5,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-6,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-7,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-8,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-9,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-10,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-11,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-12,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-13,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-14,ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag_no-15
qa1_0k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_2k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,82.00,91.00
qa1_4k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,99.00,100.00,77.00,89.00
qa1_8k,100.00,98.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,99.00,100.00,73.00,85.00
qa1_16k,98.00,97.00,98.00,98.00,98.00,98.00,98.00,98.00,98.00,98.00,98.00,98.00,98.00,97.00,98.00,71.00,84.00
qa1_32k,97.00,81.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,97.00,91.00,97.00,68.00,82.00
qa1_64k,94.00,47.00,95.00,94.00,95.00,94.00,94.00,94.00,94.00,95.00,93.00,91.00,95.00,76.00,94.00,66.00,78.00
qa2_0k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa2_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, all-lin lora 64-128 VS train ablations

In [94]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-7900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-6100/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-6000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-4400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-3900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_64_128_sink_SEGM_16x1024_16384_64_1e-04_10000/checkpoint-200/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_assoc_0_10_14_15_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-9500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_assoc_0_10_14_15_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-5000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_assoc_0_10_14_15_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_assoc_0_10_14_15_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-6800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_q_up_down_assoc_0_10_14_15_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-300/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_0_10_14_15_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-9200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_0_10_14_15_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-9500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_0_10_14_15_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_0_10_14_15_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-6100/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_0_10_14_15_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-300/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_14_15_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-8500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_14_15_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-4900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_14_15_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_14_15_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-4600/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_14_15_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_assoc_14_15_64_128_sink_SEGM_16x1024_16384_64_1e-04_10000/checkpoint-500/pytorch_model.bin",

]
disp_names = [
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-7900) [112M]",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-6100) [112M]",
    # ~112M pars
    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-5500) [112M]",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-100) [112M]",
    "ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100) [112M]",
    # 96M pars
    "ARMT-64-128-q-up-down-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-6000) [96M]",
    "ARMT-64-128-q-up-down-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-4400) [96M]",
    "ARMT-64-128-q-up-down-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-200) [96M]",
    "ARMT-64-128-q-up-down-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-3900) [96M]",
    "ARMT-64-128-q-up-down-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-400) [96M]",
    "ARMT-64-128-q-up-down-16s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-200) [96M]",
    # 43M pars
    "ARMT-64-128-q-up-down-assoc-0-10-14-15-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-9500) [43M]",
    "ARMT-64-128-q-up-down-assoc-0-10-14-15-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-5000) [43M]",
    "ARMT-64-128-q-up-down-assoc-0-10-14-15-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-200) [43M]",
    "ARMT-64-128-q-up-down-assoc-0-10-14-15-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-6800) [43M]",
    "ARMT-64-128-q-up-down-assoc-0-10-14-15-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-300) [43M]",
    # 63M
    "ARMT-64-128-all-lin-assoc-0-10-14-15-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-9200) [63M]",
    "ARMT-64-128-all-lin-assoc-0-10-14-15-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-9500) [63M]",
    "ARMT-64-128-all-lin-assoc-0-10-14-15-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-200) [63M]",
    "ARMT-64-128-all-lin-assoc-0-10-14-15-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-6100) [63M]",
    "ARMT-64-128-all-lin-assoc-0-10-14-15-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-300) [63M]",
    # 54M
    "ARMT-64-128-all-lin-assoc-14-15-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-8500) [54M]",
    "ARMT-64-128-all-lin-assoc-14-15-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-4900) [54M]",
    "ARMT-64-128-all-lin-assoc-14-15-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-200) [54M]",
    "ARMT-64-128-all-lin-assoc-14-15-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-4600) [54M]",
    "ARMT-64-128-all-lin-assoc-14-15-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-400) [54M]",
    "ARMT-64-128-all-lin-assoc-14-15-16s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-500) [54M]",


   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:8]]
#full_df = full_df[full_df.columns[[0,1,2,11,12,13,14,15,16]]]
full_df = full_df[full_df.columns[[0,1,2,3,4,21,22,23,24,25]]]


full_df.loc["avg"] = full_df.mean(axis=0)
#print(full_df.mean(axis=0))
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-7900) [112M],ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-6100) [112M],ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-5500) [112M],ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-100) [112M],ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100) [112M],ARMT-64-128-all-lin-assoc-14-15-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-8500) [54M],ARMT-64-128-all-lin-assoc-14-15-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-4900) [54M],ARMT-64-128-all-lin-assoc-14-15-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-200) [54M],ARMT-64-128-all-lin-assoc-14-15-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-4600) [54M],ARMT-64-128-all-lin-assoc-14-15-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-400) [54M]
qa1_0k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_2k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_4k,98.00,100.00,100.00,100.00,100.00,98.00,100.00,100.00,100.00,100.00
qa1_8k,53.00,100.00,100.00,100.00,100.00,92.00,100.00,100.00,100.00,100.00
qa1_16k,28.00,96.00,98.00,98.00,98.00,86.00,96.00,97.00,98.00,98.00
qa1_32k,15.00,81.00,97.00,97.00,97.00,65.00,88.00,91.00,96.00,98.00
qa1_64k,2.00,72.00,94.00,94.00,94.00,55.00,75.00,75.00,85.00,99.00
qa2_0k,100.00,99.00,100.00,100.00,100.00,98.00,100.00,100.00,100.00,96.00
qa2_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,98.00


In [12]:
lora_ablate = "mlp;q_proj"
lora_ablate = lora_ablate.split(";")
name = "lora.v_proj"
any([el in name for el in lora_ablate])

False

In [13]:
";".join(lora_ablate)

'mlp;q_proj'

### Eval 1b model, FT on BL - QA1-5, 1024 segm, all-lin lora 64-128 vs Full FT w/o ARMT

In [13]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-7900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-6100/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-100/pytorch_model.bin",


    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x8192_8192_64_1e-05_10000/checkpoint-4000",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x2048_2048_64_1e-05_10000/checkpoint-4500",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x1024_1024_64_1e-05_10000/checkpoint-4400",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_no_lora_sink_SEGM_1x8192_8192_64_1e-05_10000/checkpoint-4000",

    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x2048_2048_64_1e-05_10000/checkpoint-7200",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x1024_1024_64_1e-05_10000/checkpoint-7200",


    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-3900/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-7200/pytorch_model.bin",
    
]
disp_names = [
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-7900)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6100)",

    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-5500)",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
    #"ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",


    "Llama-3.2-1b-IT-64-128-all-lin-8192-QA1-5-IT-target-mask-no_tag (final cpt-4000)",
    "Llama-3.2-1b-IT-64-128-all-lin-2048-QA1-5-IT-target-mask-no_tag (final cpt-4500)",
    "Llama-3.2-1b-IT-64-128-all-lin-1024-QA1-5-IT-target-mask-no_tag (final cpt-4400)",
    "Llama-3.2-1b-IT-no-lora-8192-QA1-5-IT-target-mask-no_tag (final cpt-4000)",

    "Gemma-3-1b-IT-64-128-all-lin-2048-QA1-5-IT-target-mask-no_tag (final cpt-7200)",
    "Gemma-3-1b-IT-64-128-all-lin-1024-QA1-5-IT-target-mask-no_tag (final cpt-7200)",

    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag (final cpt-3900)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag (final cpt-7200)",

]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
full_df = full_df[full_df.columns[[0,1,2,3,5,6,8,9]]]
full_df.loc["avg"] = full_df.mean(axis=0)
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-7900),ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6100),ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-5500),ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100),Llama-3.2-1b-IT-64-128-all-lin-2048-QA1-5-IT-target-mask-no_tag (final cpt-4500),Llama-3.2-1b-IT-64-128-all-lin-1024-QA1-5-IT-target-mask-no_tag (final cpt-4400),Gemma-3-1b-IT-64-128-all-lin-2048-QA1-5-IT-target-mask-no_tag (final cpt-7200),Gemma-3-1b-IT-64-128-all-lin-1024-QA1-5-IT-target-mask-no_tag (final cpt-7200)
qa1_0k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,93.00
qa1_2k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_4k,98.00,100.00,100.00,100.00,100.00,100.00,100.00,99.00
qa1_8k,53.00,100.00,100.00,100.00,99.00,99.00,97.00,98.00
qa1_16k,28.00,96.00,98.00,98.00,96.00,95.00,95.00,95.00
qa1_32k,15.00,81.00,97.00,97.00,94.00,97.00,72.00,72.00
qa1_64k,2.00,72.00,94.00,94.00,87.00,88.00,0.00,0.00
qa2_0k,100.00,99.00,100.00,100.00,100.00,100.00,100.00,88.00
qa2_1k,100.00,100.00,100.00,100.00,100.00,99.00,99.00,46.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, Gemma

In [9]:
x=1

In [12]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-7900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-6100/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-100/pytorch_model.bin",


    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x8192_8192_64_1e-05_10000/checkpoint-4000",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x2048_2048_64_1e-05_10000/checkpoint-4500",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x1024_1024_64_1e-05_10000/checkpoint-4400",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_no_lora_sink_SEGM_1x8192_8192_64_1e-05_10000/checkpoint-4000",

    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x2048_2048_64_1e-05_10000/checkpoint-7200",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x1024_1024_64_1e-05_10000/checkpoint-7200",


    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-3900/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-7200/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-7000/pytorch_model.bin",

    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-4600/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-4600/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-8400/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-300/pytorch_model.bin",


    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/two_gpu_v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x4096_4096_64_1e-05_10000/checkpoint-5200",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_4x1024_4096_64_1e-04_10000/checkpoint-4400/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/two_gpu_v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x5120_5120_64_1e-05_10000/checkpoint-2200",

]
disp_names = [
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-7900)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6100)",

    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-5500)",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
    #"ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",


    "Llama-3.2-1b-IT-64-128-all-lin-8192-QA1-5-IT-target-mask-no_tag (final cpt-4000)",
    "Llama-3.2-1b-IT-64-128-all-lin-2048-QA1-5-IT-target-mask-no_tag (final cpt-4500)",
    "Llama-3.2-1b-IT-64-128-all-lin-1024-QA1-5-IT-target-mask-no_tag (final cpt-4400)",
    "Llama-3.2-1b-IT-no-lora-8192-QA1-5-IT-target-mask-no_tag (final cpt-4000)",

    "Gemma-3-1b-IT-64-128-all-lin-2048-QA1-5-IT-target-mask-no_tag (final cpt-7200)",
    "Gemma-3-1b-IT-64-128-all-lin-1024-QA1-5-IT-target-mask-no_tag (final cpt-7200)",

    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag (final cpt-3900)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag (final cpt-7200)",
    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag (final cpt-7000)",

    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-4600)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-4600)",
    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-8400)",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-300)",

    "Gemma-3-1b-IT-64-128-all-lin-4096-QA1-5-IT-target-mask-no_tag",
    "ARMT-64-128-all-lin-4s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (non-final cpt-4400)",
    "Gemma-3-1b-IT-64-128-all-lin-5120-QA1-5-IT-target-mask-no_tag",

]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
#full_df = full_df[full_df.columns[[8,9,10,11,12]]]
full_df = full_df[full_df.columns[[8,9,10,11,12,13,14,15,16,  17,18, 19]]]
#full_df = full_df[full_df.columns[[16]]]

#full_df = full_df[full_df.columns[[8,10,13]]]
full_df.loc["avg"] = full_df.mean(axis=0)
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Gemma-3-1b-IT-64-128-all-lin-2048-QA1-5-IT-target-mask-no_tag (final cpt-7200),Gemma-3-1b-IT-64-128-all-lin-1024-QA1-5-IT-target-mask-no_tag (final cpt-7200),ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag (final cpt-3900),ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag (final cpt-7200),ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag (final cpt-7000),ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-4600),ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-4600),ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-8400),ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-300),Gemma-3-1b-IT-64-128-all-lin-4096-QA1-5-IT-target-mask-no_tag,ARMT-64-128-all-lin-4s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (non-final cpt-4400),Gemma-3-1b-IT-64-128-all-lin-5120-QA1-5-IT-target-mask-no_tag
qa1_0k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_2k,100.00,100.00,100.00,100.00,100.00,97.00,99.00,100.00,100.00,100.00,100.00,100.00
qa1_4k,100.00,99.00,63.00,79.00,100.00,69.00,85.00,100.00,100.00,100.00,100.00,100.00
qa1_8k,97.00,98.00,35.00,52.00,98.00,43.00,61.00,98.00,99.00,92.31,99.00,99.00
qa1_16k,95.00,95.00,15.00,41.00,100.00,28.00,44.00,100.00,100.00,98.00,99.00,96.00
qa1_32k,72.00,72.00,11.00,37.00,100.00,27.00,17.00,100.00,100.00,95.00,95.00,94.00
qa1_64k,0.00,0.00,19.00,48.00,99.00,4.00,13.00,99.00,99.00,7.00,47.00,3.00
qa2_0k,100.00,100.00,100.00,100.00,100.00,100.00,99.00,100.00,100.00,99.00,100.00,93.00
qa2_1k,99.00,100.00,100.00,100.00,100.00,100.00,99.00,100.00,99.00,100.00,99.00,99.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, Gemma - for paper

In [27]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #"google/gemma-3-1b-it",
    "google/gemma-3-1b-it",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/two_gpu_v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x5120_5120_64_1e-05_10000/checkpoint-2200",

    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-4600/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-4600/pytorch_model.bin",
    "google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-8400/pytorch_model.bin",
    #"google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-300/pytorch_model.bin",


    #"google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/two_gpu_v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x4096_4096_64_1e-05_10000/checkpoint-5200",
    #"google/gemma-3-1b-it/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/google/gemma-3-1b-it/left_shift_v8_assist_mask_no_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_4x1024_4096_64_1e-04_10000/checkpoint-4400/pytorch_model.bin",
    
]
disp_names = [
    #"Gemma-3-1b-Instruct-CT",
    "Gemma-3-1b-Instruct-IT",
    "Gemma-3-1b-IT-64-128-all-lin-5120-QA1-5-IT-target-mask-no_tag",
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-4600)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-4600)",
    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-8400)",
    #"ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (final cpt-300)",

    #"Gemma-3-1b-IT-64-128-all-lin-4096-QA1-5-IT-target-mask-no_tag",
    #"ARMT-64-128-all-lin-4s-1024-CL-QA1-5-Gemma-3-1B-IT-target-mask-no_tag-eos_fix (non-final cpt-4400)",
    

]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "IT" in disp_name and "no_tag" not in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_no_tags_mode'
    else:
        if "-CT" in disp_name:
            prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
            all_names = [
                [f'./{mod_name}']]
        elif "-legacy" in disp_name:
            prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
            all_names = [
                [f'./{mod_name}']]
        elif "-no_tag" in disp_name:
            prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
            all_names = [
                [f'./{mod_name}']]
        elif "-E" in disp_name:
            prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
            all_names = [
                [f'./{mod_name}']]
        elif "Q1-Ans" in disp_name:
            prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
            all_names = [
                [f'./{mod_name}']]
        elif "Q1-only" in disp_name:
            prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
            all_names = [
                [f'./{mod_name}']]
        elif "Q1" in disp_name:
            prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
            all_names = [
                [f'./{mod_name}']]
        elif mod_name == "meta-llama/Llama-3.2-1B":
            prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
            all_names = [
                [f'./{mod_name}']]
        else:
            prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa4', 'qa5']#['qa1', 'qa2', 'qa3']#, 'qa4', 'qa5']
        task_names = ['QA4', 'QA5']#['QA1', 'QA2', 'QA3']#, 'QA4', 'QA5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                task_name = task_names[j]
                overall_results[disp_names[dat_idx]][task_name + ", " + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
#full_df = full_df[full_df.columns[[8,9,10,11,12]]]
#full_df = full_df[full_df.columns[[8,9,10,11,12,13,14,15,16,  17,18, 19]]]
#full_df = full_df[full_df.columns[[16]]]

#full_df = full_df[full_df.columns[[8,10,13]]]
new_cols = ["Gemma-3-1B-IT", "Gemma-3-1B-IT,\n BABILong, 5k", "ARMT,\n BABILong, 2k", "ARMT,\n BABILong, 3k", "ARMT,\n BABILong, 5k"]
full_df.columns = new_cols
full_df.loc["avg"] = full_df.mean(axis=0)
s = full_df.style
lines = ['QA1, ', 'QA2, ', 'QA3, ', 'QA4, ', 'QA5, ']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=0).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Gemma-3-1B-IT,"Gemma-3-1B-IT, BABILong, 5k","ARMT, BABILong, 2k","ARMT, BABILong, 3k","ARMT, BABILong, 5k"
"QA4, 0k",30,95,100,100,100
"QA4, 1k",12,100,100,100,100
"QA4, 2k",11,100,92,93,90
"QA4, 4k",7,99,67,77,83
"QA4, 8k",6,97,44,66,82
"QA4, 16k",14,81,39,62,79
"QA4, 32k",1,65,42,36,79
"QA4, 64k",0,5,6,10,82
"QA5, 0k",70,97,99,99,99
"QA5, 1k",39,100,99,99,99


In [28]:
#!pip install tabulate

In [29]:
#print(full_df.to_markdown(floatfmt=".0f"))

In [30]:
latex_table = (
    full_df.T.style
    #.hide(axis='index')
    .format(precision=0).background_gradient(axis=None, cmap=cmap)
    .to_latex(
        hrules=True,
        convert_css=True,
        column_format="l" + "c" * full_df.shape[1],
    )
)
print(latex_table.replace("_", "-")
      .replace("In-Domain (0k-8k)", "\\begin{tabular}[c]{@{}l@{}}{In-Domain}\\\\ {(0k-8k)}\\end{tabular}")
      .replace("OOD (8k-65k)", "\\begin{tabular}[c]{@{}l@{}}{OOD}\\\\ {(8k-65k)}\\end{tabular}")
      .replace("Long-OOD (32k-65k)", "\\begin{tabular}[c]{@{}l@{}}{Long-OOD}\\\\ {(32k-65k)}\\end{tabular}")
      .replace("Full (0k-65k)", "\\begin{tabular}[c]{@{}l@{}}{Full}\\\\ {(0k-65k)}\\end{tabular}"))

\begin{tabular}{lccccc}
\toprule
 & QA4, 0k & QA4, 1k & QA4, 2k & QA4, 4k & QA4, 8k & QA4, 16k & QA4, 32k & QA4, 64k & QA5, 0k & QA5, 1k & QA5, 2k & QA5, 4k & QA5, 8k & QA5, 16k & QA5, 32k & QA5, 64k & avg \\
\midrule
Gemma-3-1B-IT & {\cellcolor[HTML]{FF9800}} \color[HTML]{000000} 30 & {\cellcolor[HTML]{FF3C00}} \color[HTML]{F1F1F1} 12 & {\cellcolor[HTML]{FF3800}} \color[HTML]{F1F1F1} 11 & {\cellcolor[HTML]{FF2200}} \color[HTML]{F1F1F1} 7 & {\cellcolor[HTML]{FF1E00}} \color[HTML]{F1F1F1} 6 & {\cellcolor[HTML]{FF4600}} \color[HTML]{F1F1F1} 14 & {\cellcolor[HTML]{FF0400}} \color[HTML]{F1F1F1} 1 & {\cellcolor[HTML]{FF0000}} \color[HTML]{F1F1F1} 0 & {\cellcolor[HTML]{98CC00}} \color[HTML]{000000} 70 & {\cellcolor[HTML]{FFC600}} \color[HTML]{000000} 39 & {\cellcolor[HTML]{EAF500}} \color[HTML]{000000} 54 & {\cellcolor[HTML]{FFCC00}} \color[HTML]{000000} 40 & {\cellcolor[HTML]{FF9E00}} \color[HTML]{000000} 31 & {\cellcolor[HTML]{FF6A00}} \color[HTML]{F1F1F1} 21 & {\cellcolor[HTML]{FF2200}} \

### Eval 1b model, FT on BL - QA1-5, 1024 segm, Gemma with ARMT pretrain

In [18]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "google/gemma-3-1b-it",
    "google/gemma-3-1b-it",
    "irodkin/ARMT-Gemma-3-1B-IT-8x1024",
    "irodkin/ARMT-Gemma-3-1B-IT-8x1024",
]
disp_names = [
    "Gemma-3-1b-Instruct-CT",
    "Gemma-3-1b-Instruct-IT",
    "ARMT-pretrain-Gemma-3-1b-Instruct-CT",
    "ARMT-pretrain-Gemma-3-1b-Instruct-IT",
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "IT" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_no_tags_mode'
    else:
        if "-CT" in disp_name:
            prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
            all_names = [
                [f'./{mod_name}']]
        elif "-legacy" in disp_name:
            prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
            all_names = [
                [f'./{mod_name}']]
        elif "-no_tag" in disp_name:
            prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
            all_names = [
                [f'./{mod_name}']]
        elif "-E" in disp_name:
            prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
            all_names = [
                [f'./{mod_name}']]
        elif "Q1-Ans" in disp_name:
            prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
            all_names = [
                [f'./{mod_name}']]
        elif "Q1-only" in disp_name:
            prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
            all_names = [
                [f'./{mod_name}']]
        elif "Q1" in disp_name:
            prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
            all_names = [
                [f'./{mod_name}']]
        elif mod_name == "meta-llama/Llama-3.2-1B":
            prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
            all_names = [
                [f'./{mod_name}']]
        else:
            prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
#full_df = full_df[full_df.columns[[8,9,10,11,12]]]
#full_df = full_df[full_df.columns[[8,9,10,11,12,13,14,15,16]]]
#full_df = full_df[full_df.columns[[16]]]

#full_df = full_df[full_df.columns[[8,10,13]]]
full_df.loc["avg"] = full_df.mean(axis=0)
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Gemma-3-1b-Instruct-CT,Gemma-3-1b-Instruct-IT,ARMT-pretrain-Gemma-3-1b-Instruct-CT,ARMT-pretrain-Gemma-3-1b-Instruct-IT
qa1_0k,42.00,83.00,38.00,68.00
qa1_1k,2.00,60.00,6.00,50.00
qa1_2k,1.00,60.00,3.00,1.00
qa1_4k,0.00,40.00,0.00,1.00
qa1_8k,0.00,8.00,0.00,0.00
qa1_16k,0.00,4.00,1.00,0.00
qa1_32k,0.00,0.00,1.00,0.00
qa1_64k,0.00,0.00,0.00,0.00
qa2_0k,1.00,48.00,34.00,35.00
qa2_1k,2.00,47.00,11.00,12.00


### Eval 1b model, FT on BL - QA1-5, 1024 segm, ARMT-Llama-SW

In [61]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-7900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-6100/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-9000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-6800/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_sw_1gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-6800/pytorch_model.bin",
    

    
]
disp_names = [
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-7900)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6100)",

    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-5500)",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",
    #"ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100)",

    "ARMT-SW-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-9800)",
    "ARMT-SW-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6800)",

    "ARMT-SW-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-300)",
    "ARMT-SW-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6800)",

]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:2]]
#full_df = full_df[full_df.columns[[0,1,2,3,5,6,8,9]]]
full_df.loc["avg"] = full_df.mean(axis=0)
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-7900),ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6100),ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-5500),ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100),ARMT-SW-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-9800),ARMT-SW-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6800),ARMT-SW-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-300),ARMT-SW-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-6800)
qa1_0k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_2k,100.00,100.00,100.00,100.00,93.00,98.00,98.00,98.00
qa1_4k,98.00,100.00,100.00,100.00,75.00,99.00,99.00,99.00
qa1_8k,53.00,100.00,100.00,100.00,27.00,90.00,99.00,100.00
qa1_16k,28.00,96.00,98.00,98.00,18.00,75.00,85.00,98.00
qa1_32k,15.00,81.00,97.00,97.00,3.00,64.00,60.00,89.00
qa1_64k,2.00,72.00,94.00,94.00,6.00,51.00,36.00,79.00
qa2_0k,100.00,99.00,100.00,100.00,97.00,99.00,100.00,100.00
qa2_1k,100.00,100.00,100.00,100.00,97.00,99.00,100.00,99.00


In [107]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [108]:
name = "google/gemma-3-1b-it"
model = AutoModelForCausalLM.from_pretrained(name)
tokenizer = AutoTokenizer.from_pretrained(name)

In [109]:
messages = [
        {"role": "user", "content": f"Text"},
        {"role": "assistant", "content": f"Text"},
    ]
template = tokenizer.apply_chat_template(messages, tokenize=False)
template_parts = template.split("Text")
template_ids = [tokenizer(el, add_special_tokens=False)["input_ids"] for el in template_parts]

In [110]:
template_parts

['<bos><start_of_turn>user\n',
 '<end_of_turn>\n<start_of_turn>model\n',
 '<end_of_turn>\n']

In [111]:
template_ids

[[2, 105, 2364, 107], [106, 107, 105, 4368, 107], [106, 107]]

In [114]:
messages = [
        {"role": "user", "content": f"Text"},
    ]
template = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
template_parts = template.split("Text")
template_ids = [tokenizer(el, add_special_tokens=False)["input_ids"] for el in template_parts]

In [115]:
template_ids

[[2, 105, 2364, 107], [106, 107, 105, 4368, 107]]

In [72]:
model.generation_config.eos_token_id

[1, 106]

In [28]:
name = "unsloth/Llama-3.2-1B-Instruct"
model = AutoModelForCausalLM.from_pretrained(name)
tokenizer = AutoTokenizer.from_pretrained(name)

In [29]:
messages = [
        {"role": "user", "content": f"Text"},
        {"role": "assistant", "content": f"Text"},
    ]
template = tokenizer.apply_chat_template(messages, tokenize=False)
template_parts = template.split("Text")
template_ids = [tokenizer(el, add_special_tokens=False)["input_ids"] for el in template_parts]

In [30]:
template_parts

['<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nCutting Knowledge Date: December 2023\nToday Date: 15 Oct 2025\n\n<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n',
 '<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n',
 '<|eot_id|>']

In [32]:
tokenizer.name_or_path.lower()

'unsloth/llama-3.2-1b-instruct'

In [27]:
model

Gemma3ForCausalLM(
  (model): Gemma3TextModel(
    (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
    (layers): ModuleList(
      (0-25): 26 x Gemma3DecoderLayer(
        (self_attn): Gemma3Attention(
          (q_proj): Linear(in_features=1152, out_features=1024, bias=False)
          (k_proj): Linear(in_features=1152, out_features=256, bias=False)
          (v_proj): Linear(in_features=1152, out_features=256, bias=False)
          (o_proj): Linear(in_features=1024, out_features=1152, bias=False)
          (q_norm): Gemma3RMSNorm((256,), eps=1e-06)
          (k_norm): Gemma3RMSNorm((256,), eps=1e-06)
        )
        (mlp): Gemma3MLP(
          (gate_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (up_proj): Linear(in_features=1152, out_features=6912, bias=False)
          (down_proj): Linear(in_features=6912, out_features=1152, bias=False)
          (act_fn): PytorchGELUTanh()
        )
        (input_layernorm): Gemma3RMSNorm((11

In [39]:
ds1 = datasets.load_from_disk("../../data/mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k/")

In [40]:
ds1

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 328257
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 5131
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 34690
    })
})

In [45]:
ds2 = datasets.load_from_disk("../../data/mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_16k/")

Loading dataset from disk:   0%|          | 0/19 [00:00<?, ?it/s]

In [47]:
ds1["validation"] = ds2["validation"]

In [48]:
ds1

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 328257
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 5481
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 34690
    })
})

In [49]:
ds1.save_to_disk("../../data/mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k_val_16k/")

Saving the dataset (0/10 shards):   0%|          | 0/328257 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/5481 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/34690 [00:00<?, ? examples/s]

In [46]:
ds2

DatasetDict({
    train: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 341371
    })
    validation: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 5481
    })
    test: Dataset({
        features: ['repo_name', 'fname', 'context', 'question', 'length', 'answer', 'full_len'],
        num_rows: 35080
    })
})

In [82]:
del model, tokenizer

### Eval 1b model, FT on BL-MT-v2-mix-50%

In [117]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-7900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-6100/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_2k/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-4800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_4k/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-7400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k_val_16k/unsloth/Llama-3.2-1B-Instruct/long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-1400/pytorch_model.bin",

]
disp_names = [
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-7900) [112M]",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-6100) [112M]",
    # ~112M pars
    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-5500) [112M]",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-100) [112M]",
    #"ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100) [112M]",
    # bl-mt-v2-mix
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-4800) [112M]",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-500) [112M]",
    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-7400) [112M]",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-300) [112M]",
    "ARMT-64-128-all-lin-8s-1024-16s-valid-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-1400) [112M]",
    


   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:8]]
#full_df = full_df[full_df.columns[[0,1,2,11,12,13,14,15,16]]]
#full_df = full_df[full_df.columns[[0,1,2,3,4,21,22,23,24,25]]]


full_df.loc["avg"] = full_df.mean(axis=0)
#print(full_df.mean(axis=0))
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-7900) [112M],ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-6100) [112M],ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-5500) [112M],ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-100) [112M],ARMT-64-128-all-lin-2s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-4800) [112M],ARMT-64-128-all-lin-3s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-500) [112M],ARMT-64-128-all-lin-5s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-7400) [112M],ARMT-64-128-all-lin-8s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-300) [112M],ARMT-64-128-all-lin-8s-1024-16s-valid-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-1400) [112M]
qa1_0k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_1k,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00,100.00
qa1_2k,100.00,100.00,100.00,100.00,85.00,83.00,97.00,97.00,100.00
qa1_4k,98.00,100.00,100.00,100.00,69.00,72.00,94.00,93.00,99.00
qa1_8k,53.00,100.00,100.00,100.00,49.00,60.00,95.00,95.00,100.00
qa1_16k,28.00,96.00,98.00,98.00,39.00,47.00,20.00,30.00,93.00
qa1_32k,15.00,81.00,97.00,97.00,5.00,13.00,1.00,0.00,3.00
qa1_64k,2.00,72.00,94.00,94.00,4.00,3.00,0.00,0.00,0.00
qa2_0k,100.00,99.00,100.00,100.00,98.00,98.00,98.00,98.00,84.00
qa2_1k,100.00,100.00,100.00,100.00,98.00,99.00,100.00,100.00,89.00


### Eval 1b model, FT on BL-MT-v2-mix-50% - fixed ckpt and bigger lora

In [ ]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_ft'
results_folder_v7_2 = '../../rmt_ift/DEBUG_babilong_evals_bl_ft'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-7900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-6100/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/finetune/babilong_multitask/armt_sw/pg19/babilong/unsloth/Llama-3.2-1B-Instruct/v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_12x1024_12288_64_1e-04_10000/checkpoint-100/pytorch_model.bin",


    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k/unsloth/Llama-3.2-1B-Instruct/v8_no_armt_assist_mask_no_sw_1gpu_8192_q15_all_lin_64_128_sink_SEGM_1x8192_8192_64_1e-05_10000/checkpoint-9300",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_2k/unsloth/Llama-3.2-1B-Instruct/ckpt_fix_long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-8800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_4k/unsloth/Llama-3.2-1B-Instruct/ckpt_fix_long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k/unsloth/Llama-3.2-1B-Instruct/ckpt_fix_long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-400/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k_val_16k/unsloth/Llama-3.2-1B-Instruct/ckpt_fix_long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_64_128_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-500/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_2k/unsloth/Llama-3.2-1B-Instruct/ckpt_fix_long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_128_256_sink_SEGM_2x1024_2048_64_1e-04_10000/checkpoint-8800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_4k/unsloth/Llama-3.2-1B-Instruct/ckpt_fix_long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_128_256_sink_SEGM_3x1024_3072_64_1e-04_10000/checkpoint-200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k/unsloth/Llama-3.2-1B-Instruct/ckpt_fix_long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_128_256_sink_SEGM_5x1024_5120_64_1e-04_10000/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/finetune/bl_mix_mt_v2/armt_sw/pg19/babilong+mt_prepared_v2/manytypes_flat_stacked_no_empty_sv_0k_8k_val_16k/unsloth/Llama-3.2-1B-Instruct/ckpt_fix_long_val_v8_assist_mask_no_sw_4gpu_1024_q15_all_lin_128_256_sink_SEGM_8x1024_8192_64_1e-04_10000/checkpoint-100/pytorch_model.bin",

    
]
disp_names = [
    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-7900) [112M]",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-6100) [112M]",
    # ~112M pars
    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-5500) [112M]",
    "ARMT-64-128-all-lin-8s-1024-CL-QA1-5-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-100) [112M]",
    #"ARMT-64-128-all-lin-12s-1024-CL-QA1-5-Llama-3.2-1B-IT-target-mask-no_tag (final cpt-100) [112M]",
    # bl-mt-v2-mix
    "Llama-64-128-all-lin-8s-1024-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-9300)",

    "ARMT-64-128-all-lin-2s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-8800)",
    "ARMT-64-128-all-lin-3s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-200)",
    "ARMT-64-128-all-lin-5s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-400)",
    "ARMT-64-128-all-lin-8s-1024-16s-valid-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-500)",

    "ARMT-128-256-all-lin-2s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-8800) [112M]",
    "ARMT-128-256-all-lin-3s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-200) [112M]",
    "ARMT-128-256-all-lin-5s-1024-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-300) [112M]",
    "ARMT-128-256-all-lin-8s-1024-16s-valid-CL-QA1-5+MT-v2-50%-Llama-3.2-1B-IT-assist-mask-no_tag (final cpt-100) [112M]",
    


   
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:8]]
#full_df = full_df[full_df.columns[[0,1,2,11,12,13,14,15,16]]]
#full_df = full_df[full_df.columns[[0,1,2,3,4,21,22,23,24,25]]]


full_df.loc["avg"] = full_df.mean(axis=0)
#print(full_df.mean(axis=0))
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "64k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


### Eval SFT ARMT SmolLM 135M

In [9]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_bl_from_inst'
results_folder_v7_2 = '../../rmt_ift/babilong_evals_bl_from_inst'

#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "HuggingFaceTB/SmolLM2-135M-Instruct/runs/inst_pretrain_armt_sw/armt_1k_8s_smollm/smol_smoltalk/HuggingFaceTB/SmolLM2-135M/mask_fix_assist_mask_no_sw_no_lora_SEGM_8x1024_8064_64_3e-04_1000/checkpoint-1000/pytorch_model.bin",
    "HuggingFaceTB/SmolLM2-135M-Instruct/runs/inst_pretrain_armt_sw/armt_1k_8s_smollm/smol_smoltalk/HuggingFaceTB/SmolLM2-135M/mask_fix_assist_mask_no_sw_no_lora_SEGM_8x1024_8064_64_3e-04_1000/checkpoint-1000/pytorch_model.bin",
    "HuggingFaceTB/SmolLM2-135M",
    "HuggingFaceTB/SmolLM2-135M",
]
disp_names = [
    "SmolLM2-135M-Instruct-CT",
    "SmolLM2-135M-Instruct-IT",
    "ARMT-8s-1024-SmolLM2-135M-Instruct-CT",
    "ARMT-8s-1024-SmolLM2-135M-Instruct-IT",
    "SmolLM2-135M",
    "SmolLM2-135M-All_no_ct",
  
]
all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-legacy" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_legacy_bl_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-no_tag" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_no_tags_mode'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-All_no_ct" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or mod_name == "HuggingFaceTB/SmolLM2-135M":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        #tasks = ['qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "v7_no_sw_wm_1gpu_sw_q15_all_lin_16_32_sink_SEGM_2x512" in mod_name:
                    fname = f'./{results_folder_v7_2}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
#full_df = full_df[full_df.columns[3:8]]
#full_df = full_df[full_df.columns[:8]]
#full_df = full_df[full_df.columns[[0,1,2,11,12,13,14,15,16]]]
#full_df = full_df[full_df.columns[[0,1,2,3,4,21,22,23,24,25]]]


full_df.loc["avg"] = full_df.mean(axis=0)
#print(full_df.mean(axis=0))
s = full_df.style
lines = ['qa1_', 'qa2_', 'qa3_', 'qa4_', 'qa5_']
lines = [el + "16k" for el in lines]
for l0 in lines:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,SmolLM2-135M-Instruct-CT,SmolLM2-135M-Instruct-IT,ARMT-8s-1024-SmolLM2-135M-Instruct-CT,ARMT-8s-1024-SmolLM2-135M-Instruct-IT,SmolLM2-135M,SmolLM2-135M-All_no_ct
qa1_0k,39.00,24.00,34.00,43.00,51.00,41.00
qa1_1k,18.00,34.00,18.00,30.00,25.00,25.00
qa1_2k,24.00,30.00,11.00,13.00,28.00,28.00
qa1_4k,17.00,15.00,4.00,6.00,17.00,20.00
qa1_8k,11.00,12.00,2.00,2.00,16.00,17.00
qa1_16k,0.00,0.00,0.00,0.00,0.00,0.00
qa2_0k,30.00,30.00,26.00,33.00,30.00,21.00
qa2_1k,26.00,24.00,24.00,26.00,29.00,17.00
qa2_2k,27.00,26.00,17.00,20.00,28.00,18.00
qa2_4k,23.00,25.00,10.00,12.00,18.00,16.00


### Loss & lora ablations for model with attn on prev input

In [19]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',

    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-8500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only_sw/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2200/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/checkpoint-6500-prelim/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",


    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"Dolly-Llama-3.2-1B",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    
    
    #"Smoltalk-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",

    "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",

    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1",

    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1",
    
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",

    "Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    "Llama-3.2-1B-IT-Q1-only",
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "I-Q1-only" in disp_name:
        prompt_name = 'instruction_yes_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1,Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only,Llama-3.2-1B-IT-Q1-only,Llama-3.2-1B-IT
qa1_0k,58.20,13.40,12.80,23.80,4.40,64.10,27.00,67.40,59.40,68.40,57.80,45.90,9.80,72.90
qa1_1k,71.90,10.60,35.30,41.40,2.70,2.80,0.70,1.20,3.50,2.30,1.70,51.60,0.00,66.80
qa1_2k,70.10,29.50,33.30,38.40,26.30,13.50,8.70,0.70,28.30,15.30,10.40,53.20,0.60,60.90
qa1_4k,62.40,16.90,7.90,18.00,13.70,6.60,6.20,0.50,16.10,7.80,5.10,38.40,0.60,58.30
qa1_8k,52.00,5.60,0.90,6.00,6.60,2.30,2.10,0.70,7.40,3.80,2.00,18.70,0.30,55.60
qa1_16k,47.20,0.90,0.00,2.60,3.40,1.70,1.90,0.20,3.60,1.60,0.50,31.70,0.40,45.60
qa2_0k,33.83,3.00,5.61,37.04,26.63,27.23,27.93,14.51,35.74,33.43,32.63,30.53,28.83,45.15
qa2_1k,32.23,0.80,7.41,16.12,3.20,2.20,1.50,1.60,1.70,2.10,1.70,10.81,9.51,35.34
qa2_2k,29.13,1.80,11.81,3.70,10.01,5.81,10.71,1.90,12.91,5.41,5.21,8.31,5.21,27.33
qa2_4k,26.23,2.40,4.00,2.70,5.11,3.10,5.91,0.70,6.81,3.80,4.90,10.21,0.30,20.42


### Various ablations - armt-only, fixes on smoltalk

In [506]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',

    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    # "meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    # "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",

    # "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-8500/pytorch_model.bin",
    # "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only_sw/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/checkpoint-6500-prelim/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-7500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-7500/pytorch_model.bin",


    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_32/checkpoint-9500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_3968_32/checkpoint-7500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_7936_32/checkpoint-7500/pytorch_model.bin",

    
    "meta-llama/Llama-3.2-1B/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix_4k_res/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-2000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_group_batch/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64/checkpoint-9500/pytorch_model.bin",
    
    
    # "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    
    # "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    "Smoltalk-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    
    
    #"Smoltalk-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    # "ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    # "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",

    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",

    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1",

    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1",
    
    # "ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only-CompLoss-Llama-3.2-1B-IT-Q1-only",
    # "ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT_prelim",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-8s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",

    "ARMT-32-32-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT",
    "ARMT-32-32-4s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT",
    "ARMT-32-32-8s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT",

    
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B",


    "ARMT-16-64-4s-CL-SmoltalkMix05_4k_res-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-GrBatch-Llama-3.2-1B-IT_prelim",
    "ARMT-32-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    
    # "Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    # "Llama-3.2-1B-IT-Q1-only",
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "I-Q1-only" in disp_name:
        prompt_name = 'instruction_yes_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Smoltalk-Llama-3.2-1B,ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT_prelim,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-8s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-32-32-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT,ARMT-32-32-4s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT,ARMT-32-32-8s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B,ARMT-16-64-4s-CL-SmoltalkMix05_4k_res-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-GrBatch-Llama-3.2-1B-IT_prelim,ARMT-32-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,Llama-3.2-1B-IT
qa1_0k,58.20,67.00,73.80,73.20,73.80,74.40,72.10,78.30,76.00,76.20,66.80,78.00,76.10,75.50,72.90
qa1_1k,71.90,63.40,63.50,66.00,66.30,66.10,64.30,65.70,60.90,53.10,64.80,63.50,67.10,66.40,66.80
qa1_2k,70.10,56.10,39.20,47.70,46.50,45.50,43.50,51.70,49.90,48.70,29.30,33.40,46.60,50.30,60.90
qa1_4k,62.40,30.80,26.60,11.00,11.10,30.80,27.20,23.50,31.00,33.70,13.80,19.70,16.80,14.70,58.30
qa1_8k,52.00,12.60,0.40,5.10,5.50,17.90,16.30,7.90,16.10,19.60,6.60,10.10,4.30,6.90,55.60
qa1_16k,47.20,1.80,0.00,2.80,1.90,5.90,10.10,3.30,8.70,9.30,4.10,3.70,1.30,0.20,45.60
qa2_0k,33.83,39.64,42.94,51.75,51.65,52.15,50.45,51.85,51.55,51.55,40.84,53.75,51.65,53.45,45.15
qa2_1k,32.23,27.83,32.33,32.03,31.43,32.03,32.13,27.83,28.63,22.32,31.13,31.43,33.13,33.83,35.34
qa2_2k,29.13,23.32,23.12,25.93,24.92,19.72,22.92,20.62,24.02,23.92,23.02,16.42,25.33,23.52,27.33
qa2_4k,26.23,11.91,12.71,7.81,5.91,14.71,16.32,12.41,15.32,16.12,11.61,11.51,13.71,8.31,20.42


### Smoltalk training ablations

In [504]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    "rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',

    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    # "meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    # "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",

    # "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-8500/pytorch_model.bin",
    # "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only_sw/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2200/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/checkpoint-6500-prelim/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-7500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-7500/pytorch_model.bin",

    
    "meta-llama/Llama-3.2-1B/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix_4k_res/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-2000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_group_batch/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-5500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_group_batch/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64/checkpoint-9500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_wscl/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_1x1024_1008_64/checkpoint-9500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_wscl/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_32/checkpoint-10000/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_full_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_64mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1920_32/checkpoint-10000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_64mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1920_64/checkpoint-10000/pytorch_model.bin",


    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_32/checkpoint-9500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_3968_32/checkpoint-7500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_7936_32/checkpoint-7500/pytorch_model.bin",

    # "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    
    # "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    "Smoltalk-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    
    
    #"Smoltalk-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    # "ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    # "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",

    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",

    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1",

    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1",
    
    # "ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only-CompLoss-Llama-3.2-1B-IT-Q1-only",
    # "ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT_prelim",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B",


    #"ARMT-16-64-4s-CL-SmoltalkMix05_4k_res-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-GrBatch-Llama-3.2-1B-IT_prelim",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-GrBatch-Llama-3.2-1B-IT",
    "ARMT-32-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-1s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-from-1s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-32-32-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-FullLoss-Llama-3.2-1B-IT",
    "ARMT-64-32-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",
    "ARMT-64-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",


    "ARMT-32-32-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT",
    "ARMT-32-32-4s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT",
    "ARMT-32-32-8s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT",
    # "Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    # "Llama-3.2-1B-IT-Q1-only",
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "I-Q1-only" in disp_name:
        prompt_name = 'instruction_yes_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_7936_32/checkpoint-7500/pytorch_model.bin/qa5_8k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv
No such file: ./../../rmt_ift/babilong_evals/./unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_7936_32/checkpoint-7500/pytorch_model.bin/qa5_16k_instruction_yes_examples_yes_post_prompt_yes_chat_template_yes.csv


,Llama-3.2-1B-E,Smoltalk-Llama-3.2-1B,ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT_prelim,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-GrBatch-Llama-3.2-1B-IT_prelim,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-GrBatch-Llama-3.2-1B-IT,ARMT-32-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-1s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-from-1s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-32-32-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-FullLoss-Llama-3.2-1B-IT,ARMT-64-32-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-64-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT,ARMT-32-32-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT,ARMT-32-32-4s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT,ARMT-32-32-8s-CL-SmoltalkMix05-noLora-mask-CompLoss-gr-batch-Llama-3.2-1B-IT,Llama-3.2-1B-IT
qa1_0k,58.20,67.00,73.80,73.20,73.80,66.80,76.10,77.10,75.50,74.50,74.10,75.10,73.00,75.30,75.90,78.30,76.00,76.20,72.90
qa1_1k,71.90,63.40,63.50,66.00,66.30,64.80,67.10,68.30,66.40,64.10,63.50,65.80,66.40,66.90,66.50,65.70,60.90,53.10,66.80
qa1_2k,70.10,56.10,39.20,47.70,46.50,29.30,46.60,49.80,50.30,51.20,36.40,51.80,49.50,47.50,47.20,51.70,49.90,48.70,60.90
qa1_4k,62.40,30.80,26.60,11.00,11.10,13.80,16.80,18.70,14.70,33.10,18.70,21.30,3.60,6.80,10.30,23.50,31.00,33.70,58.30
qa1_8k,52.00,12.60,0.40,5.10,5.50,6.60,4.30,6.50,6.90,16.90,1.10,4.40,0.10,2.40,3.10,7.90,16.10,19.60,55.60
qa1_16k,47.20,1.80,0.00,2.80,1.90,4.10,1.30,2.60,0.20,8.20,0.30,1.00,0.00,1.30,0.60,3.30,8.70,9.30,45.60
qa2_0k,33.83,39.64,42.94,51.75,51.65,40.84,51.65,51.85,53.45,52.45,51.25,53.75,42.24,52.35,52.55,51.85,51.55,51.55,45.15
qa2_1k,32.23,27.83,32.33,32.03,31.43,31.13,33.13,33.03,33.83,29.43,31.63,31.93,24.62,30.33,30.23,27.83,28.63,22.32,35.34
qa2_2k,29.13,23.32,23.12,25.93,24.92,23.02,25.33,23.92,23.52,11.91,19.42,22.62,23.12,17.12,18.92,20.62,24.02,23.92,27.33
qa2_4k,26.23,11.91,12.71,7.81,5.91,11.61,13.71,12.31,8.31,4.70,11.31,12.31,5.71,3.90,5.81,12.41,15.32,16.12,20.42


### Loss ablations for model with qa inst dataset

In [487]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',

    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-3900/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-3900/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-8500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only_sw/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2200/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/checkpoint-6500-prelim/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",


    "runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"Dolly-Llama-3.2-1B",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    
    
    #"Smoltalk-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    "ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",

    "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",

    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    "ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1",

    "ARMT-16-64-2s-CL-QaInst-1k-2k-big-noLora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    "ARMT-16-64-2s-CL-QaInst-1k-2k-big-noLora-CompLoss-Llama-3.2-1B-IT-I-Q1-only",

    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1",
    
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",

    "Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    "Llama-3.2-1B-IT-Q1-only",
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "I-Q1-only" in disp_name:
        prompt_name = 'instruction_yes_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only,ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only,ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1,ARMT-16-64-2s-CL-QaInst-1k-2k-big-noLora-CompLoss-Llama-3.2-1B-IT-Q1-only,ARMT-16-64-2s-CL-QaInst-1k-2k-big-noLora-CompLoss-Llama-3.2-1B-IT-I-Q1-only,Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only,Llama-3.2-1B-IT-Q1-only,Llama-3.2-1B-IT
qa1_0k,58.20,13.40,12.80,23.80,4.40,64.10,27.00,67.40,59.40,68.40,57.80,8.50,41.00,45.90,9.80,72.90
qa1_1k,71.90,10.60,35.30,41.40,2.70,2.80,0.70,1.20,3.50,2.30,1.70,19.10,4.20,51.60,0.00,66.80
qa1_2k,70.10,29.50,33.30,38.40,26.30,13.50,8.70,0.70,28.30,15.30,10.40,20.80,41.20,53.20,0.60,60.90
qa1_4k,62.40,16.90,7.90,18.00,13.70,6.60,6.20,0.50,16.10,7.80,5.10,2.80,8.50,38.40,0.60,58.30
qa1_8k,52.00,5.60,0.90,6.00,6.60,2.30,2.10,0.70,7.40,3.80,2.00,0.50,0.80,18.70,0.30,55.60
qa1_16k,47.20,0.90,0.00,2.60,3.40,1.70,1.90,0.20,3.60,1.60,0.50,0.00,0.00,31.70,0.40,45.60
qa2_0k,33.83,3.00,5.61,37.04,26.63,27.23,27.93,14.51,35.74,33.43,32.63,21.72,1.70,30.53,28.83,45.15
qa2_1k,32.23,0.80,7.41,16.12,3.20,2.20,1.50,1.60,1.70,2.10,1.70,9.51,0.20,10.81,9.51,35.34
qa2_2k,29.13,1.80,11.81,3.70,10.01,5.81,10.71,1.90,12.91,5.41,5.21,12.41,10.01,8.31,5.21,27.33
qa2_4k,26.23,2.40,4.00,2.70,5.11,3.10,5.91,0.70,6.81,3.80,4.90,1.30,1.60,10.21,0.30,20.42


### Loss ablations for model with qa inst dataset, for pres

In [523]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    #"meta-llama/Llama-3.2-1B",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',

    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_no_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-2800/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-4500/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2800/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-1700/pytorch_model.bin",

    
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_no_lora_prev_input/armt_1k_2s_llama_base/synth_reduced_1k_1792_1m_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-9700/pytorch_model.bin",

    "unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-3900/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_cot/armt_1k_2s_llama_base/synth_reduced_qa_inst_1k_1920_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-3900/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64_1e-05_10000/checkpoint-8500/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/loss_abaltion_armt_only_sw/armt_1k_2s_llama_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_4x512_1984_64_1e-05_10000/checkpoint-2200/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_comp_multi_mask_no_pack/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/checkpoint-6500-prelim/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/loss_abaltion_lora/armt_1k_2s_llama_base/synth_reduced_1k_1792_4mod_big/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984_64_1e-05_10000_32/checkpoint-3900/pytorch_model.bin",


    #"runs/loss_abaltion_lora/llama_2048_base/synth_reduced_1k_1792_big/unsloth/Llama-3.2-1B-Instruct/SEGM_1x2048_2048_1e-05_10000/checkpoint-9800",
    
    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    #"Llama-3.2-1B-E",
    #"Dolly-Llama-3.2-1B",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    
    
    #"Smoltalk-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-Ans",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-Q1-only",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-CompLoss-Llama-3.2-1B-Q1-only",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Lora-CompLoss-Llama-3.2-1B-IT-Q1-only",

    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-Lora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",

    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-I-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-1m-noLora_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1",

    "ARMT-16-64-2s-CL-QaInst-1k-2k-big-noLora-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-QaInst-1k-2k-big-noLora-CompLoss-Llama-3.2-1B-IT-I-Q1-only",

    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B",
    #"ARMT-32-64-2s-CL-DebugSynth1k-2k-Div-Lora-CompLoss-Llama-3.2-1B-Q1",
    
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-DebugSynth1k-2k-noLora_ARMT-only_prev_inp-CompLoss-Llama-3.2-1B-IT-Q1-only",
    #"ARMT-16-64-2s-CL-SmoltalkMix05-noLora-mask-CompLoss-Llama-3.2-1B-IT",

    #"Llama-3.2-1B-DebugSynth1k-2k-Lora-CompLoss-IT-Q1-only",

    "Llama-3.2-1B-IT-Q1-only",
    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "I-Q1-only" in disp_name:
        prompt_name = 'instruction_yes_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,ARMT-16-64-2s-CL-QaInst-1k-2k-big-noLora-CompLoss-Llama-3.2-1B-IT-Q1-only,Llama-3.2-1B-IT-Q1-only,Llama-3.2-1B-IT
qa1_0k,8.50,9.80,72.90
qa1_1k,19.10,0.00,66.80
qa1_2k,20.80,0.60,60.90
qa1_4k,2.80,0.60,58.30
qa1_8k,0.50,0.30,55.60
qa1_16k,0.00,0.40,45.60
qa2_0k,21.72,28.83,45.15
qa2_1k,9.51,9.51,35.34
qa2_2k,12.41,5.21,27.33
qa2_4k,1.30,0.30,20.42


### IFT Comparison

In [470]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    "rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x8128_8128/checkpoint-1600",
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",
    "Flan-Llama-3.2-1B",
    
    "Smoltalk-Llama-3.2-1B",
    "Smoltalk-8k-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,Flan-Llama-3.2-1B,Smoltalk-Llama-3.2-1B,Smoltalk-8k-Llama-3.2-1B,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,57.00,67.00,64.60,72.90
qa1_1k,71.90,65.00,47.40,63.40,58.40,66.80
qa1_2k,70.10,65.10,45.70,56.10,40.20,60.90
qa1_4k,62.40,57.60,43.30,30.80,18.20,58.30
qa1_8k,52.00,44.30,39.70,12.60,11.20,55.60
qa1_16k,47.20,37.90,34.80,1.80,9.40,45.60
qa2_0k,33.83,44.14,35.94,39.64,40.04,45.15
qa2_1k,32.23,33.13,14.01,27.83,28.43,35.34
qa2_2k,29.13,30.83,18.72,23.32,19.42,27.33
qa2_4k,26.23,27.13,14.61,11.91,12.21,20.42


### Question first results, answer variations

In [30]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals_quest_first'
results_folder_regular = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"meta-llama/Llama-3.2-1B",
    #"meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",

    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"rmt_ift_comparison/full_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-91000",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_mem_mean_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1022/checkpoint-600/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/rmt_ift_from_2s_LM_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_992/checkpoint-900/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-100/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step19500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step31500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_step49500.bin",
    #"meta-llama/Llama-3.2-1B/data/pretrained_models/ARMT-llama3.2-1B/armt_llama3.2-1B_ctx4096_step19500.bin",

    #"rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_31500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_49500_64_16/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer_from_scratch/LM_pg19_19500_64_16/dolly_long_05_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/short_synth/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-200/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-27000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-12000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_5x1024_5040_64/checkpoint-6000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/sharegpt/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-1000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_4x1024_4032_64/checkpoint-1000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_mix/meta-llama/Llama-3.2-1B/SEGM_8x1024_8064_64/checkpoint-950/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_LM_hf_trainer/LM_pg19_19500_64_16/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64-inter/checkpoint-22000-inter/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",

    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-100000/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer/flan/meta-llama/Llama-3.2-1B/SEGM_3x1024_3024_64/checkpoint-94000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",

    
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-Regular-E",
    #"Llama-3.2-1B",
    #"Llama-3.2-1B-E",
    "Llama-3.2-1B-Zero",
    "Llama-3.2-1B-Zero-Q2",
    "Llama-3.2-1B-All",
    "Llama-3.2-1B-E",

    "Llama-3.2-1B-E-Ans",
    "Llama-3.2-1B-All-Ans",
    "Llama-3.2-1B-Regular-E-Ans",
    #"DollyLoraOld-Llama-3.2-1B",
    #"Dolly-Llama-3.2-1B",
    #"Flan-Llama-3.2-1B",
    #"RMT-1k-16mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-Dolly-Llama-3.2-1B",
    #"RMT-1k-1mem-mean-Dolly-Llama-3.2-1B",
    #"RMT-1k-16mem-LM-2s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Llama-3.2-1B-E",
    #"ARMT-1k-16mem-64-4s-LM-19.5k-Llama-3.2-1B-E",

    #"Dolly-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-LM-19.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-31.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-49.5k-Dolly-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-DollyMix05-8s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT",
    #"ARMT-1k-16mem-64-2s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-5s-CL-SGPT-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-CL-SGPT-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-LM-19.5k-Flan-22k-2s-Llama-3.2-1B-IT-Q2",
    #"ARMT-1k-16mem-64-2s-ShortSynth-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-2s-Flan-Llama-3.2-1B-IT-Q2",
    #"ARMT-1k-16mem-64-3s-Flan-Llama-3.2-1B-IT-Q2",
    #"ARMT-1k-16mem-64-2s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-4s-CL-DollyMix-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-8s-DollyMix-Llama-3.2-1B",

    #"ARMT-1k-16mem-64-2s-CL-FLAN-Llama-3.2-1B",
    #"ARMT-1k-16mem-64-3s-CL-FLAN-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
    "Llama-3.2-1B-IT-Q2",
    "Llama-3.2-1B-IT-Ans",
    "Llama-3.2-1B-IT-Regular-Ans",
    "Llama-3.2-1B-IT-Regular",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "IT-Regular-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "IT-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Regular-E-Ans" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Regular-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Regular" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E-Ans" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-All-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-All" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-Zero-Q2" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_quest_first_2_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-IT-Q2" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_quest_first_2_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-Zero" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or "data/pretrained_models" in mod_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "Regular" in disp_name and not "Regular-E-Ans" in disp_name and not "IT-Regular-Ans" in disp_name:
                    fname = f'./{results_folder_regular}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                #elif "mix" in model_name or "mergekit" in model_name:
                #    fname = f'./{new_results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-Regular-E,Llama-3.2-1B-Zero,Llama-3.2-1B-Zero-Q2,Llama-3.2-1B-All,Llama-3.2-1B-E,Llama-3.2-1B-E-Ans,Llama-3.2-1B-All-Ans,Llama-3.2-1B-Regular-E-Ans,Llama-3.2-1B-IT,Llama-3.2-1B-IT-Q2,Llama-3.2-1B-IT-Ans,Llama-3.2-1B-IT-Regular-Ans,Llama-3.2-1B-IT-Regular
qa1_0k,58.20,0.00,21.10,0.00,0.00,58.90,58.50,65.70,60.70,60.80,62.70,71.10,72.90
qa1_1k,71.90,0.00,7.50,0.00,0.00,52.20,59.30,72.80,49.90,59.60,52.00,66.80,66.80
qa1_2k,70.10,0.10,3.70,0.00,0.00,51.50,52.80,70.50,40.30,51.70,36.00,61.10,60.90
qa1_4k,62.40,0.00,7.70,0.00,0.00,36.60,24.10,63.10,41.20,46.10,39.70,60.50,58.30
qa1_8k,52.00,0.00,4.50,0.00,0.00,20.50,17.50,53.90,31.80,40.20,26.40,54.80,55.60
qa1_16k,47.20,0.00,3.40,0.00,0.00,14.00,5.70,45.30,20.80,31.10,17.10,45.20,45.60
qa2_0k,33.83,0.00,9.81,0.00,0.00,39.14,34.83,38.54,38.44,41.94,39.54,44.74,45.15
qa2_1k,32.23,0.00,6.21,0.00,0.00,30.73,30.93,32.43,31.13,33.63,29.93,33.43,35.34
qa2_2k,29.13,0.00,3.90,0.00,0.00,24.42,28.03,30.33,20.12,29.73,21.62,27.03,27.33
qa2_4k,26.23,0.00,5.01,0.00,0.00,15.12,21.72,26.83,14.11,21.52,9.51,21.52,20.42


### Babilong-pretrained results

In [44]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "meta-llama/Llama-3.2-1B",
    #"llama_non_it_exps/meta-llama/Llama-3.2-1B/databricks-dolly-15k_ift_1024_seed_42/checkpoint-2000",
    "rmt_ift_comparison/ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-700",
    
    #"meta-llama/Llama-3.2-1B/rmt_ift_comparison/armt_mem_fix_2s_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_1984/checkpoint-500/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_full_loss_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/TEST_LORA_armt_all_s_ift_from_base_hf_trainer/dolly/meta-llama/Llama-3.2-1B/SEGM_2x1024_2016_64/checkpoint-1000/pytorch_model.bin",

    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-50/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/mmix_lora_armt_all_s_ift_from_base_hf_trainer/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-50/pytorch_model.bin",
    
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-400/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/lora_armt_all_s_ift_from_base_hf_trainer/comploss/dolly/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-300/pytorch_model.bin",
    #"rmt_ift_comparison/ift_from_base_hf_trainer/smoltalk/meta-llama/Llama-3.2-1B/SEGM_1x1024_1024/checkpoint-10000",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_2x1024_2016_64/checkpoint-10000/pytorch_model.bin",
    #"unsloth/Llama-3.2-1B-Instruct/runs/armt_ift/smol_lora_comp/smoltalk_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64/checkpoint-9000/pytorch_model.bin",
    "unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",
    "unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",
    "unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",
    "unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",
    "unsloth/Llama-3.2-1B-Instruct/data/pretrained_models/RMT-Llama-3.2-1B-Instruct-4x1024-mem16-lora-babilong-qa1-5/model.safetensors",

    #'unsloth/Llama-3.2-1B-Instruct/runs/packing_ablation/armt_long_synth/2k_synth/unsloth/Llama-3.2-1B-Instruct/SEGM_4x1024_4032_64_1e-05_3000/checkpoint-600/pytorch_model.bin',
    
    #"meta-llama/Llama-3.2-1B/runs/armt_ift/armt_all_s_ift_from_base_hf_trainer_from_scratch/dolly_long_05_mix/unsloth/Llama-3.2-1B-Instruct/SEGM_8x1024_8064_64/checkpoint-150/pytorch_model.bin",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "Llama-3.2-1B-E",
    #"DollyLoraOld-Llama-3.2-1B",
    "Dolly-Llama-3.2-1B",

    #"ARMT-16-64-2s-Dolly-CT-Llama-3.2-1B",
    #"ARMT-16-64-8s-DollyMix05-Llama-3.2-1B-IT",
    #"ARMT-16-64-8s-CL-DollyMix05-Lora-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B",
    #"ARMT-16-64-8s-CL-DollyMix05-FullLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    
    "ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT",

    #"ARMT-16-64-2s-CL-Dolly-Lora-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-Dolly-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-2s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-SmoltalkMix05-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT",
    "ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT-Zero",
    "ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT-Q-Ans",
    "ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT-Ans",
    "ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT-Zero-Ans",
    #"ARMT-1k-16mem-64-8s-DollyMix05-Llama-3.2-1B-IT",
    #"ARMT-16-64-4s-CL-LongSynth2k-Lora-CompLoss-Llama-3.2-1B-IT",
    #"Smoltalk-Llama-3.2-1B",

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "IT-Q-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-Zero-Ans" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "IT-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-Zero" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k']#, '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


,Llama-3.2-1B-E,Dolly-Llama-3.2-1B,ARMT-16-64-8s-CL-DollyMix05-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT,ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT-Zero,ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT-Q-Ans,ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT-Ans,ARMT-16-64-4s-CL-BabiLong-QA1-5-Lora-CompLoss-Llama-3.2-1B-IT-Zero-Ans,Llama-3.2-1B-IT
qa1_0k,58.20,66.10,72.90,0.00,42.10,0.00,0.00,87.40,72.90
qa1_1k,71.90,65.00,63.50,0.00,11.50,0.00,0.00,16.80,66.80
qa1_2k,70.10,65.10,49.80,0.40,17.60,0.70,0.80,16.00,60.90
qa1_4k,62.40,57.60,35.50,0.30,18.00,0.40,0.50,17.70,58.30
qa1_8k,52.00,44.30,19.40,2.30,17.80,8.10,11.40,17.60,55.60
qa1_16k,47.20,37.90,10.40,16.40,18.20,13.40,13.50,18.20,45.60
qa2_0k,33.83,44.14,46.95,0.10,21.92,0.60,0.20,60.46,45.15
qa2_1k,32.23,33.13,35.94,0.50,10.11,1.70,0.90,21.22,35.34
qa2_2k,29.13,30.83,18.72,0.20,14.01,0.80,0.60,13.31,27.33
qa2_4k,26.23,27.13,12.41,0.60,14.41,1.50,1.60,13.81,20.42


### Smol models & olmo

In [15]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../../../rvanova/armt-instruct-tune/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    
    #"HuggingFaceTB/SmolLM2-135M/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/HuggingFaceTB/SmolLM2-135M/SEGM_2x1024_1984_32/checkpoint-2500/pytorch_model.bin",

    "HuggingFaceTB/SmolLM2-1.7B",
    #"HuggingFaceTB/SmolLM2-1.7B",
    "HuggingFaceTB/SmolLM2-1.7B-Instruct",
    "HuggingFaceTB/SmolLM2-135M",
    "HuggingFaceTB/SmolLM2-135M-Instruct",
    "HuggingFaceTB/SmolLM2-360M",
    "HuggingFaceTB/SmolLM2-360M-Instruct",

    "allenai/allenai/OLMo-2-1124-7B",
    "allenai/allenai/OLMo-2-1124-7B-SFT",
    "allenai/allenai/OLMo-2-1124-7B-DPO",
    "allenai/allenai/OLMo-2-1124-7B-Instruct",
    #"unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    
    #"ARMT-2s-smol-SmolLM2-135M",
    "SmolLM2-1.7B",
    #"SmolLM2-1.7B-E",
    "SmolLM2-1.7B-IT",
    "SmolLM2-135M",
    "SmolLM2-135M-IT",
    "SmolLM2-360M",
    "SmolLM2-360M-IT",
    "OLMo-2-1124-7B",
    "OLMo-2-1124-7B-SFT",
    "OLMo-2-1124-7B-DPO",
    "OLMo-2-1124-7B-IT",
    

    #"Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or disp_name == "SmolLM2-135M" or disp_name == "SmolLM2-1.7B" or disp_name == "OLMo-2-1124-7B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k']#, '16k', '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
#for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
for l0 in ['qa1_8k', 'qa2_8k', 'qa3_8k', 'qa4_8k', 'qa5_8k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../../../rvanova/armt-instruct-tune/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa3_0k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../../../rvanova/armt-instruct-tune/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa3_1k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../../../rvanova/armt-instruct-tune/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa3_2k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../../../rvanova/armt-instruct-tune/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa3_4k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../../../rvanova/armt-instruct-tune/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa3_8k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../../../rvanova/armt-instruct-tune/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B-Instruct/qa1_2k_instruction_yes_examples_

,SmolLM2-1.7B,SmolLM2-1.7B-IT,SmolLM2-135M,SmolLM2-135M-IT,SmolLM2-360M,SmolLM2-360M-IT,OLMo-2-1124-7B,OLMo-2-1124-7B-SFT,OLMo-2-1124-7B-DPO,OLMo-2-1124-7B-IT
qa1_0k,74.00,83.00,46.50,26.90,nan,59.00,80.00,97.00,98.00,98.00
qa1_1k,50.00,68.60,39.40,37.70,nan,52.60,72.00,89.00,86.00,86.00
qa1_2k,58.50,nan,nan,nan,nan,nan,51.00,78.00,80.00,80.00
qa1_4k,41.60,nan,nan,nan,nan,nan,39.00,63.00,58.00,59.00
qa1_8k,27.30,nan,nan,nan,nan,nan,0.00,0.00,0.00,0.00
qa2_0k,43.14,48.05,33.93,34.43,nan,41.34,11.00,58.00,59.00,59.00
qa2_1k,17.72,31.13,27.83,27.43,nan,29.53,12.00,57.00,57.00,57.00
qa2_2k,22.42,nan,nan,nan,nan,nan,18.00,42.00,45.00,44.00
qa2_4k,16.82,nan,nan,nan,nan,nan,9.00,32.00,34.00,36.00
qa2_8k,13.71,nan,nan,nan,nan,nan,0.00,0.00,0.00,0.00


### Smol models

In [517]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
results_folder = '../../rmt_ift/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'

mod_names = [
    "HuggingFaceTB/SmolLM2-135M",
    "HuggingFaceTB/SmolLM2-135M",
    "HuggingFaceTB/SmolLM2-135M/runs/armt_ift/smol_comp_multi_mask_no_pack_32mem_fix_segm/smoltalk_long_05_mix/HuggingFaceTB/SmolLM2-135M/SEGM_2x1024_1984_32/checkpoint-2500/pytorch_model.bin",

    "HuggingFaceTB/SmolLM2-1.7B",
    "HuggingFaceTB/SmolLM2-1.7B",
    
    "unsloth/Llama-3.2-1B-Instruct",
]
disp_names = [
    "SmolLM2-135M",
    "SmolLM2-135M-E",
    "ARMT-2s-smol-SmolLM2-135M",
    "SmolLM2-1.7B",
    "SmolLM2-1.7B-E",
    

    "Llama-3.2-1B-IT",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    disp_name = disp_names[dat_idx]
    if "-CT" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "-E" in disp_name:
        prompt_name = 'instruction_no_examples_yes_post_prompt_no_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-Ans" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes_answer_fill_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1-only" in disp_name:
        prompt_name = 'instruction_no_examples_no_post_prompt_no_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif "Q1" in disp_name:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes_quest_first_yes'
        all_names = [
            [f'./{mod_name}']]
    elif mod_name == "meta-llama/Llama-3.2-1B" or disp_name == "SmolLM2-135M"or disp_name == "SmolLM2-1.7B":
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
    all_names = [
        [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[disp_names[dat_idx]] = {}
        tasks = ["qa2"]# ['qa1', 'qa2', 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        tasks = ['qa1', 'qa2', 'qa3', 'qa4', 'qa5']
        lengths = ['0k', '1k', '2k', '4k', '8k']#, '16k', '32k']#, '32k', '64k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                if "lm_no_st" in model_name:
                    fname = f'./{results_folder_rmt}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                elif "mix" in model_name or "mergekit" in model_name:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                else:
                    fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
                if ctx_length == "32k":
                    df.dropna(inplace=True)
                    #print(len(df))
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                #if model_name == "Qwen/Qwen2.5-0.5B-Instruct/runs/lm/flan/Qwen/Qwen2.5-0.5B-Instruct/linear_adamw_wd1e-03_4086-128-1x4096_mem5_bs32_regular_bptt-4_lora_freeze_from_cpt_0-1/run_2/model_best/model.safetensors":
                #    overall_results["RMT4k-Qwen2.5-0.5B-Instruct"][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                #else:
                overall_results[disp_names[dat_idx]][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
full_df = pd.concat(all_dfs, axis=1)
from matplotlib.colors import LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list('ryg', ["red", "yellow", "green"], N=256)
# old cmap
#full_df.style.format(precision=2).background_gradient(axis=0, cmap='YlOrRd')
s = full_df.style
#for l0 in ['qa1_16k', 'qa2_16k', 'qa3_16k', 'qa4_16k', 'qa5_16k']:
for l0 in ['qa1_8k', 'qa2_8k', 'qa3_8k', 'qa4_8k', 'qa5_8k']:
    s.set_table_styles({l0: [{'selector': '', 'props': 'border-bottom: 3px solid black;'}]}, overwrite=False, axis=1)
fres = s.format(precision=2).background_gradient(axis=None, cmap=cmap)
#fres = full_df.style.format(precision=2).background_gradient(axis=0, cmap=cmap)
from pandas import option_context

with option_context('display.max_colwidth', 20):
    display(fres)


No such file: ./../../rmt_ift/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa1_2k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../rmt_ift/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa1_4k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../rmt_ift/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa1_8k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../rmt_ift/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa2_2k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../rmt_ift/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa2_4k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../rmt_ift/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa2_8k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../rmt_ift/babilong_evals/./HuggingFaceTB/SmolLM2-1.7B/qa3_2k_instruction_yes_examples_yes_post_pro

,SmolLM2-135M,SmolLM2-135M-E,ARMT-2s-smol-SmolLM2-135M,SmolLM2-1.7B,SmolLM2-1.7B-E,Llama-3.2-1B-IT
qa1_0k,51.00,51.00,6.00,81.00,72.00,72.90
qa1_1k,25.00,31.00,5.00,53.00,72.00,66.80
qa1_2k,28.00,29.00,10.00,nan,nan,60.90
qa1_4k,17.00,17.00,1.00,nan,nan,58.30
qa1_8k,16.00,18.00,1.00,nan,nan,55.60
qa2_0k,30.00,34.00,4.00,40.00,36.00,45.15
qa2_1k,29.00,28.00,14.00,11.00,28.00,35.34
qa2_2k,28.00,24.00,10.00,nan,nan,27.33
qa2_4k,18.00,18.00,2.00,nan,nan,20.42
qa2_8k,10.00,11.00,0.00,nan,nan,19.02


### Get model's loss values

In [92]:
def extract_loss_values(path, loss_type):
    files = os.listdir(path)
    log_file = [el for el in files if el[-4:] == ".log"]
    fpath = os.path.join(path, log_file[0])
    with open(fpath, "r") as f:
        text = f.read()
    loss_key = f"Validation on {loss_type} loss: "
    loss_value = text.split(loss_key)[-1].split("\n")[0]
    return loss_value

In [93]:
pathes = [
    "../../runs/loss_evals/dolly/meta-llama/Llama-3.2-1B/llama_32_1b_base",
    "../../runs/loss_evals/dolly/unsloth/Llama-3.2-1B-Instruct/llama_32_1b_it",
    "../../runs/loss_evals/dolly/meta-llama/Llama-3.2-1B/lm_rmt_2x1024_mem16_llama_32_1b_base",
    "../../runs/loss_evals/dolly/meta-llama/Llama-3.2-1B/lm_rmt_dolly_ift_2x1024_mem16_llama_32_1b_base",
]
names = [
    "Llama-3.2-1B",
    "Llama-3.2-1B-IT",
    "LM+RMT-2s1k-Llama-3.2-1B",
    "LM+RMT-2s1k+DollyIFT-Llama-3.2-1B",
]
overall_res = {}
keys = ["train", "valid", "test"]
for loss_key in keys:
    overall_res[loss_key] = {}
    for idx in range(len(pathes)):
        buf_path = pathes[idx]
        buf_name = names[idx]
        overall_res[loss_key][buf_name] = extract_loss_values(buf_path, loss_key)

In [94]:
pd.DataFrame.from_dict(overall_res)

,train,valid,test
Llama-3.2-1B,2.9373,2.8825,2.8266
Llama-3.2-1B-IT,2.9095,2.8659,2.8346
LM+RMT-2s1k-Llama-3.2-1B,2.9838,2.9508,3.2406
LM+RMT-2s1k+DollyIFT-Llama-3.2-1B,1.4782,1.7550,2.1753


In [111]:
pathes = [
    "../../runs/loss_evals/sharegpt/meta-llama/Llama-3.2-1B/llama_32_1b_base",
    "../../runs/loss_evals/sharegpt/unsloth/Llama-3.2-1B-Instruct/llama_32_1b_it",
    "../../runs/loss_evals/sharegpt/meta-llama/Llama-3.2-1B/lm_rmt_2x1024_mem16_llama_32_1b_base",
    "../../runs/loss_evals/sharegpt/meta-llama/Llama-3.2-1B/lm_rmt_dolly_ift_2x1024_mem16_llama_32_1b_base",
]
names = [
    "Llama-3.2-1B",
    "Llama-3.2-1B-IT",
    "LM+RMT-2s1k-Llama-3.2-1B",
    "LM+RMT-2s1k+SGPT-IFT-Llama-3.2-1B",
]
overall_res = {}
keys = ["train", "valid", "test"]
for loss_key in keys:
    overall_res[loss_key] = {}
    for idx in range(len(pathes)):
        buf_path = pathes[idx]
        buf_name = names[idx]
        overall_res[loss_key][buf_name] = extract_loss_values(buf_path, loss_key)
pd.DataFrame.from_dict(overall_res)

,train,valid,test
Llama-3.2-1B,2.1761,2.1842,2.2153
Llama-3.2-1B-IT,2.1616,2.1714,2.1725
LM+RMT-2s1k-Llama-3.2-1B,2.4640,2.4719,2.7451
LM+RMT-2s1k+SGPT-IFT-Llama-3.2-1B,1.5477,1.5664,1.7863


In [91]:
overall_res

{'train': {'Llama-3.2-1B': '2.9373',
  'Llama-3.2-1B-IT': '2.9095',
  'LM+RMT-2s1k-Llama-3.2-1B': '2024-10-25 15:52:55,959 - root - INFO - preparing dataset for dolly',
  'LM+RMT-2s1k+DollyIFT-Llama-3.2-1B': '1.4782'},
 'valid': {'Llama-3.2-1B': '2.8825',
  'Llama-3.2-1B-IT': '2.8659',
  'LM+RMT-2s1k-Llama-3.2-1B': '2024-10-25 15:52:55,959 - root - INFO - preparing dataset for dolly',
  'LM+RMT-2s1k+DollyIFT-Llama-3.2-1B': '1.7550'},
 'test': {'Llama-3.2-1B': '2.8266',
  'Llama-3.2-1B-IT': '2.8346',
  'LM+RMT-2s1k-Llama-3.2-1B': '2024-10-25 15:52:55,959 - root - INFO - preparing dataset for dolly',
  'LM+RMT-2s1k+DollyIFT-Llama-3.2-1B': '2.1753'}}

In [75]:
test_path = "../../runs/loss_evals/dolly/meta-llama/Llama-3.2-1B/llama_32_1b_base"
tl = "Validation on valid loss: "

In [76]:
extract_loss_values(test_path, tl)

2024-10-25 15:23:44,971 - root - INFO - preparing dataset for dolly
2024-10-25 15:23:52,159 - root - INFO - preparing validation data from babilong
2024-10-25 15:23:52,354 - root - INFO - Using model class: <class 'transformers.models.auto.modeling_auto.AutoModelForCausalLM'>
2024-10-25 15:23:52,355 - root - INFO - Loading pretrained model: meta-llama/Llama-3.2-1B
2024-10-25 15:23:53,028 - root - INFO - Using optimizer class: <class 'torch.optim.adamw.AdamW'>
2024-10-25 15:23:54,382 - root - INFO - setting up trainer with accelerator state:
Distributed environment: MULTI_GPU  Backend: nccl
Num processes: 1
Process index: 0
Local process index: 0
Device: cuda:0

Mixed precision type: bf16

2024-10-25 15:23:54,806 - root - INFO - Running validation on train set:
2024-10-25 15:23:54,807 - root - INFO - start validation at step 0
2024-10-25 15:24:43,154 - root - INFO - Validation on train loss: 2.9373
2024-10-25 15:24:43,156 - root - INFO - Validation on train perplexity: 19.4360
2024-10-2

In [98]:
import datasets

In [99]:
sgpt = datasets.load_from_disk("../../data/sharegpt_raw_cleaned/hf_dataset_sharegpt")

In [105]:
lens = [len(el[0]["content"].split(" ")) + len(el[1]["content"].split(" ")) for el in sgpt["train"]["messages"][:10000]]

In [107]:
np.mean(lens), np.max(lens), np.percentile(lens, 90)

(240.4926, 2905, 457.10000000000036)

### Eval base vs LongAlign models on Babilong to find out if it works

In [215]:
# extract results into table
results_folder = '../../babilong/babilong_evals'
#model_name = './recurrent-memory-transformer/test_exps/gpt_2_ift_flan_1024_seed_42/checkpoint-5000'
#model_name = './gpt2_old'


mod_names = [
    #"Qwen/Qwen2.5-0.5B",
    #"Qwen/Qwen2.5-1.5B",
    #"Qwen/Qwen2.5-3B",
    "THUDM_no_chat/LongAlign-7B-64k-base",
    "THUDM_no_chat/LongAlign-7B-64k",
    "THUDM/LongAlign-6B-64k-base",
    "THUDM/LongAlign-6B-64k",
]

all_dfs = []
for dat_idx, mod_name in enumerate(mod_names):
    #gkuzmin/rmt_it/babilong/babilong_evals/test_exps/gpt_2_lora_ift_flan_1024_seed_42/checkpoint-1000
    overall_results = {}
    #dname = ift_dataset_path.split("-")[0]
    if "instruct" in mod_name.lower():
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_yes'
        all_names = [
            [f'./{mod_name}']]
    else:
        prompt_name = 'instruction_yes_examples_yes_post_prompt_yes_chat_template_no'
        all_names = [
            [f'./{mod_name}']]
    names = all_names[0]
    for model_name in names:
        overall_results[model_name] = {}
        tasks = ['qa1', 'qa2']#, 'qa4', 'qa5']#, 'qa3', 'qa4', 'qa5']#, 'qa6', 'qa7', 'qa8', 'qa9', 'qa10']
        lengths = ['0k', '1k', '2k', '4k', '8k', '16k', '32k']
        accuracy = np.zeros((len(tasks), len(lengths)))
        for j, task in enumerate(tasks):
            #overall_results[model_name][task] = {}
            for i, ctx_length in enumerate(lengths):
                fname = f'./{results_folder}/{model_name}/{task}_{ctx_length}_{prompt_name}.csv'
                if not os.path.isfile(fname):
                    print(f'No such file: {fname}')
                    continue
        
                df = pd.read_csv(fname)
        
                if df['output'].dtype != object:
                    df['output'] = df['output'].astype(str)
                df['output'] = df['output'].fillna('')
        
        
                df['correct'] = df.apply(lambda row: compare_answers(row['target'], row['output'],
                                                                     row['question'], TASK_LABELS[task]
                                                                     ), axis=1)
                score = df['correct'].sum()
                overall_results[model_name][task + "_" + ctx_length] = float(np.round(100 * score / len(df) if len(df) > 0 else 0, 2))
                accuracy[j, i] = 100 * score / len(df) if len(df) > 0 else 0
        #print(accuracy)
    overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}
    mod_df = pd.DataFrame.from_dict(overall_results)
    all_dfs.append(mod_df)
pd.concat(all_dfs, axis=1)

No such file: ./../../babilong/babilong_evals/./THUDM/LongAlign-6B-64k/qa1_2k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../babilong/babilong_evals/./THUDM/LongAlign-6B-64k/qa1_4k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../babilong/babilong_evals/./THUDM/LongAlign-6B-64k/qa1_8k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../babilong/babilong_evals/./THUDM/LongAlign-6B-64k/qa1_16k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../babilong/babilong_evals/./THUDM/LongAlign-6B-64k/qa1_32k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../babilong/babilong_evals/./THUDM/LongAlign-6B-64k/qa2_0k_instruction_yes_examples_yes_post_prompt_yes_chat_template_no.csv
No such file: ./../../babilong/babilong_evals/./THUDM/LongAlign-6B-64k/qa2_1k_instruction_yes_examples_yes_post_prompt_yes_chat_templa

,LongAlign-7B-64k-base,LongAlign-7B-64k,LongAlign-6B-64k-base,LongAlign-6B-64k
qa1_0k,18.10,79.90,68.90,78.7
qa1_1k,31.50,1.40,73.00,79.5
qa1_2k,18.30,4.00,62.10,NaN
qa1_4k,24.10,13.20,58.20,NaN
qa1_8k,34.70,20.10,50.40,NaN
qa1_16k,39.80,35.50,45.70,NaN
qa1_32k,37.90,18.60,45.70,NaN
qa2_0k,25.93,2.20,21.62,NaN
qa2_1k,22.12,0.50,27.33,NaN
qa2_2k,16.52,3.10,26.13,NaN


In [107]:
# flatten dict
#df = pd.json_normalize(overall_results[list(overall_results.keys())[0]], sep='_')
#new_res = {}
overall_results = {key.split("/")[-1]:value for key,value in overall_results.items()}

In [108]:
pd.DataFrame.from_dict(overall_results)

,Qwen2.5-0.5B,checkpoint-100,checkpoint-300,checkpoint-500,checkpoint-1000,checkpoint-2000,checkpoint-6000,checkpoint-10000
qa1_0k,2.60,0.0,0.00,0.00,0.00,0.00,0.00,0.40
qa1_1k,45.80,0.0,6.10,0.80,16.70,1.50,4.60,4.40
qa2_0k,0.00,0.0,0.00,0.00,1.10,1.60,1.70,1.60
qa2_1k,14.81,0.1,4.00,0.10,5.71,4.40,3.20,5.81
qa4_0k,18.72,0.0,0.00,0.00,1.00,0.50,0.40,0.50
qa4_1k,29.63,0.0,2.30,0.80,12.51,6.71,2.80,5.31
qa5_0k,2.10,0.0,0.90,0.10,13.71,2.90,6.41,15.52
qa5_1k,43.73,0.5,11.33,2.61,27.68,8.83,13.14,23.67


In [56]:
overall_results

{'gpt2': {'qa1_0k': 15.7, 'qa1_1k': 23.3, 'qa2_0k': 21.12, 'qa2_1k': 19.72},
 'checkpoint-1000': {'qa1_0k': 38.1,
  'qa1_1k': 31.6,
  'qa2_0k': 24.72,
  'qa2_1k': 18.42},
 'checkpoint-2000': {'qa1_0k': 27.1,
  'qa1_1k': 31.7,
  'qa2_0k': 12.41,
  'qa2_1k': 18.32},
 'checkpoint-3000': {'qa1_0k': 39.1,
  'qa1_1k': 37.5,
  'qa2_0k': 20.42,
  'qa2_1k': 13.61},
 'checkpoint-4000': {'qa1_0k': 41.8,
  'qa1_1k': 25.5,
  'qa2_0k': 16.72,
  'qa2_1k': 6.11},
 'checkpoint-5000': {'qa1_0k': 41.4,
  'qa1_1k': 28.5,
  'qa2_0k': 21.22,
  'qa2_1k': 9.01},
 'checkpoint-6000': {'qa1_0k': 33.7,
  'qa1_1k': 23.0,
  'qa2_0k': 23.72,
  'qa2_1k': 15.72},
 'checkpoint-7000': {'qa1_0k': 31.8,
  'qa1_1k': 21.9,
  'qa2_0k': 24.22,
  'qa2_1k': 14.81},
 'checkpoint-8000': {'qa1_0k': 20.6,
  'qa1_1k': 15.6,
  'qa2_0k': 10.91,
  'qa2_1k': 4.0},
 'checkpoint-9000': {'qa1_0k': 30.9,
  'qa1_1k': 18.6,
  'qa2_0k': 24.72,
  'qa2_1k': 7.21},
 'checkpoint-10000': {'qa1_0k': 31.0,
  'qa1_1k': 15.6,
  'qa2_0k': 31.83,
  'qa2_

We have more examples with evaluation of various models in ./notebooks and ./scripts folders.

E.g. how to run models that are hosted on HuggingFace with default prompts `scripts/run_model_on_babilong.py`.

```bash
RESULTS_FOLDER="./babilong_evals"
DATASET_NAME="booydar/babilong-1k-samples"
MODEL_NAME="mistralai/Mistral-7B-Instruct-v0.2"
TASKS=("qa1" "qa2" "qa3" "qa4" "qa5")
LENGTHS=("0k" "1k" "2k" "4k" "8k" "16k" "32k")
USE_CHAT_TEMPLATE=true
USE_INSTRUCTION=true
USE_EXAMPLES=true
USE_POST_PROMPT=true
API_URL=""

echo running $MODEL_NAME on "${TASKS[@]}" with "${LENGTHS[@]}"

python scripts/run_model_on_babilong.py \
    --results_folder "$RESULTS_FOLDER" \
    --dataset_name "$DATASET_NAME" \
    --model_name "$MODEL_NAME" \
    --tasks "${TASKS[@]}" \
    --lengths "${LENGTHS[@]}" \
    $( [ "$USE_CHAT_TEMPLATE" == true ] && echo "--use_chat_template" ) \
    $( [ "$USE_INSTRUCTION" == true ] && echo "--use_instruction" ) \
    $( [ "$USE_EXAMPLES" == true ] && echo "--use_examples" ) \
    $( [ "$USE_POST_PROMPT" == true ] && echo "--use_post_prompt" ) \
    --api_url "$API_URL"
```

It is also possible to evaluate models that are running through API (e.g., llamacpp):
```bash
RESULTS_FOLDER="./babilong_evals"
DATASET_NAME="booydar/babilong"
MODEL_NAME="CohereForAI/c4ai-command-r-v01"
TASKS=("qa1" "qa2" "qa3" "qa4" "qa5")
LENGTHS=("64k" "128k")
USE_CHAT_TEMPLATE=true
USE_INSTRUCTION=true
USE_EXAMPLES=true
USE_POST_PROMPT=true
API_URL="http://localhost:8081/completion"

echo running $MODEL_NAME on "${TASKS[@]}" with "${LENGTHS[@]}"

python scripts/run_model_on_babilong.py \
    --results_folder "$RESULTS_FOLDER" \
    --dataset_name "$DATASET_NAME" \
    --model_name "$MODEL_NAME" \
    --tasks "${TASKS[@]}" \
    --lengths "${LENGTHS[@]}" \
    $( [ "$USE_CHAT_TEMPLATE" == true ] && echo "--use_chat_template" ) \
    $( [ "$USE_INSTRUCTION" == true ] && echo "--use_instruction" ) \
    $( [ "$USE_EXAMPLES" == true ] && echo "--use_examples" ) \
    $( [ "$USE_POST_PROMPT" == true ] && echo "--use_post_prompt" ) \
    --api_url "$API_URL"
```